ptb-xl preprocessing

In [ ]:
from __future__ import annotations

import ast
import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import wfdb
from tqdm import tqdm

warnings.filterwarnings("ignore")

ROOT_DIR = Path("PTBXL_ROOT")
OUTPUT_DIR = Path("OUTPUT_DIR")

USE_SAMPLING_RATE = 500
TARGET_LENGTH = 5000
TARGET_LEADS = 12

TRAIN_FOLDS = set(range(1, 9))
VAL_FOLDS = {9}
TEST_FOLDS = {10}

CLASS_NAMES = ["NORM", "MI", "STTC", "CD", "HYP"]
CLASS_TO_INT = {name: idx for idx, name in enumerate(CLASS_NAMES)}
INT_TO_CLASS = {idx: name for name, idx in CLASS_TO_INT.items()}

TIE_BREAK_PRIORITY = ["MI", "STTC", "CD", "HYP", "NORM"]

EPS = 1e-8
DTYPE = np.float32

DROP_RECORDS_WITHOUT_DIAGNOSTIC_SUPERCLASS = True


def ensure_required_files() -> None:
    required = [
        ROOT_DIR / "ptbxl_database.csv",
        ROOT_DIR / "scp_statements.csv",
        ROOT_DIR / "records500",
    ]

    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required PTB-XL files/folders:\n" + "\n".join(missing)
        )


def parse_scp_codes(value: Any) -> Dict[str, float]:
    if isinstance(value, dict):
        return {str(k): float(v) for k, v in value.items()}

    if pd.isna(value):
        return {}

    try:
        parsed = ast.literal_eval(str(value))
        if not isinstance(parsed, dict):
            return {}
        return {str(k): float(v) for k, v in parsed.items()}
    except Exception:
        return {}


def split_from_fold(fold: int) -> str:
    fold = int(fold)

    if fold in TRAIN_FOLDS:
        return "train"
    if fold in VAL_FOLDS:
        return "val"
    if fold in TEST_FOLDS:
        return "test"

    raise ValueError(f"Unexpected strat_fold value: {fold}")


def load_official_metadata() -> Tuple[pd.DataFrame, pd.DataFrame]:
    db = pd.read_csv(ROOT_DIR / "ptbxl_database.csv", index_col="ecg_id")
    db.index = db.index.astype(int)
    db["ecg_id"] = db.index.astype(int)
    db["scp_codes_parsed"] = db["scp_codes"].apply(parse_scp_codes)

    scp = pd.read_csv(ROOT_DIR / "scp_statements.csv", index_col=0)
    scp.index = scp.index.astype(str)
    scp_diag = scp[scp["diagnostic"] == 1].copy()

    if "diagnostic_class" not in scp_diag.columns:
        raise KeyError("scp_statements.csv does not contain 'diagnostic_class'.")

    return db, scp_diag


def aggregate_superclasses(
    scp_codes: Dict[str, float],
    scp_diag: pd.DataFrame,
) -> Tuple[List[str], Dict[str, float], np.ndarray, int, str]:
    class_scores = {name: -np.inf for name in CLASS_NAMES}

    for scp_code, confidence in scp_codes.items():
        if scp_code not in scp_diag.index:
            continue

        superclass = scp_diag.loc[scp_code, "diagnostic_class"]

        if isinstance(superclass, pd.Series):
            superclass = superclass.iloc[0]

        superclass = str(superclass)

        if superclass in class_scores:
            class_scores[superclass] = max(
                class_scores[superclass],
                float(confidence),
            )

    superclasses = [
        name for name in CLASS_NAMES if np.isfinite(class_scores[name])
    ]

    multilabel = np.zeros(len(CLASS_NAMES), dtype=np.int64)

    for name in superclasses:
        multilabel[CLASS_TO_INT[name]] = 1

    if not superclasses:
        return [], class_scores, multilabel, -1, "NO_DIAG"

    max_score = max(class_scores[name] for name in superclasses)
    tied = [name for name in superclasses if class_scores[name] == max_score]

    if len(tied) == 1:
        single_name = tied[0]
    else:
        single_name = next(name for name in TIE_BREAK_PRIORITY if name in tied)

    single_int = CLASS_TO_INT[single_name]

    return superclasses, class_scores, multilabel, single_int, single_name


def attach_labels_and_split(
    db: pd.DataFrame,
    scp_diag: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    dropped_no_diag = 0

    for _, row in tqdm(db.iterrows(), total=len(db), desc="Aggregating labels"):
        superclasses, class_scores, multilabel, single_int, single_name = (
            aggregate_superclasses(row["scp_codes_parsed"], scp_diag)
        )

        if DROP_RECORDS_WITHOUT_DIAGNOSTIC_SUPERCLASS and not superclasses:
            dropped_no_diag += 1
            continue

        filename_hr = str(row["filename_hr"]).replace("\\", "/")

        rows.append(
            {
                "ecg_id": int(row["ecg_id"]),
                "patient_id": int(row["patient_id"]),
                "age": row.get("age", np.nan),
                "sex": row.get("sex", np.nan),
                "height": row.get("height", np.nan),
                "weight": row.get("weight", np.nan),
                "strat_fold": int(row["strat_fold"]),
                "split": split_from_fold(int(row["strat_fold"])),
                "filename_hr": filename_hr,
                "record_path_no_ext": str(ROOT_DIR / filename_hr),
                "scp_codes": json.dumps(row["scp_codes_parsed"]),
                "diagnostic_superclasses": ";".join(superclasses),
                "label_name": single_name,
                "label_numeric": int(single_int),
                "multilabel_NORM": int(multilabel[CLASS_TO_INT["NORM"]]),
                "multilabel_MI": int(multilabel[CLASS_TO_INT["MI"]]),
                "multilabel_STTC": int(multilabel[CLASS_TO_INT["STTC"]]),
                "multilabel_CD": int(multilabel[CLASS_TO_INT["CD"]]),
                "multilabel_HYP": int(multilabel[CLASS_TO_INT["HYP"]]),
                "score_NORM": None
                if not np.isfinite(class_scores["NORM"])
                else float(class_scores["NORM"]),
                "score_MI": None
                if not np.isfinite(class_scores["MI"])
                else float(class_scores["MI"]),
                "score_STTC": None
                if not np.isfinite(class_scores["STTC"])
                else float(class_scores["STTC"]),
                "score_CD": None
                if not np.isfinite(class_scores["CD"])
                else float(class_scores["CD"]),
                "score_HYP": None
                if not np.isfinite(class_scores["HYP"])
                else float(class_scores["HYP"]),
            }
        )

    metadata = pd.DataFrame(rows)
    print(f"Dropped records with no diagnostic superclass: {dropped_no_diag}")

    return metadata


def verify_record_file_exists(record_path_no_ext: str) -> None:
    base = Path(record_path_no_ext)
    hea = base.with_suffix(".hea")
    dat = base.with_suffix(".dat")

    if not hea.exists() or not dat.exists():
        raise FileNotFoundError(
            f"Missing WFDB pair for {base}\nExpected:\n  {hea}\n  {dat}"
        )


def read_ptbxl_signal(record_path_no_ext: str) -> np.ndarray:
    verify_record_file_exists(record_path_no_ext)

    signal, meta = wfdb.rdsamp(record_path_no_ext)
    signal = np.asarray(signal, dtype=DTYPE)

    fs = float(meta.get("fs", USE_SAMPLING_RATE))

    if int(round(fs)) != USE_SAMPLING_RATE:
        raise ValueError(
            f"Unexpected sampling rate {fs} Hz for {record_path_no_ext}. "
            f"This script expects records500 at {USE_SAMPLING_RATE} Hz."
        )

    if signal.ndim != 2:
        raise ValueError(
            f"Unexpected signal ndim={signal.ndim} for {record_path_no_ext}"
        )

    if signal.shape[1] != TARGET_LEADS:
        raise ValueError(
            f"Expected {TARGET_LEADS} leads, got {signal.shape[1]} "
            f"for {record_path_no_ext}"
        )

    if signal.shape[0] > TARGET_LENGTH:
        start = (signal.shape[0] - TARGET_LENGTH) // 2
        signal = signal[start : start + TARGET_LENGTH, :]
    elif signal.shape[0] < TARGET_LENGTH:
        pad_len = TARGET_LENGTH - signal.shape[0]
        signal = np.pad(signal, ((0, pad_len), (0, 0)), mode="edge")

    if signal.shape != (TARGET_LENGTH, TARGET_LEADS):
        raise ValueError(
            f"Postprocessing shape mismatch for {record_path_no_ext}: "
            f"{signal.shape}"
        )

    if np.isnan(signal).any() or np.isinf(signal).any():
        signal = np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)

    return signal.astype(DTYPE, copy=False)


def compute_train_global_per_lead_stats(
    metadata: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray]:
    train_df = metadata[metadata["split"] == "train"].reset_index(drop=True)

    if len(train_df) == 0:
        raise ValueError("No training records found.")

    total_count = 0
    lead_sum = np.zeros(TARGET_LEADS, dtype=np.float64)
    lead_sumsq = np.zeros(TARGET_LEADS, dtype=np.float64)

    for _, row in tqdm(
        train_df.iterrows(),
        total=len(train_df),
        desc="Computing train mean/std",
    ):
        signal = read_ptbxl_signal(row["record_path_no_ext"])
        lead_sum += signal.sum(axis=0, dtype=np.float64)
        lead_sumsq += np.square(signal, dtype=np.float64).sum(
            axis=0,
            dtype=np.float64,
        )
        total_count += signal.shape[0]

    mean = lead_sum / total_count
    var = (lead_sumsq / total_count) - np.square(mean)
    var = np.maximum(var, EPS)
    std = np.sqrt(var)

    return mean.astype(DTYPE), std.astype(DTYPE)


def normalize_signal(
    signal: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
) -> np.ndarray:
    return ((signal - mean[None, :]) / std[None, :]).astype(
        DTYPE,
        copy=False,
    )


def save_one_split(
    metadata: pd.DataFrame,
    split_name: str,
    train_mean: np.ndarray,
    train_std: np.ndarray,
) -> Dict[str, Any]:
    split_df = metadata[metadata["split"] == split_name].copy().reset_index(
        drop=True
    )

    n = len(split_df)

    if n == 0:
        raise ValueError(f"No records found for split='{split_name}'.")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    signal_path = OUTPUT_DIR / f"{split_name}_signals.npy"
    label_path = OUTPUT_DIR / f"{split_name}_labels.npy"
    multilabel_path = OUTPUT_DIR / f"{split_name}_multilabel_labels.npy"
    metadata_path = OUTPUT_DIR / f"{split_name}_metadata.csv"

    signals_mm = np.lib.format.open_memmap(
        signal_path,
        mode="w+",
        dtype=DTYPE,
        shape=(n, TARGET_LENGTH, TARGET_LEADS),
    )

    single_labels = split_df["label_numeric"].to_numpy(dtype=np.int64)

    multi_cols = [
        "multilabel_NORM",
        "multilabel_MI",
        "multilabel_STTC",
        "multilabel_CD",
        "multilabel_HYP",
    ]

    multilabels = split_df[multi_cols].to_numpy(dtype=np.int64)

    for i, (_, row) in enumerate(
        tqdm(split_df.iterrows(), total=n, desc=f"Saving {split_name}")
    ):
        signal = read_ptbxl_signal(row["record_path_no_ext"])
        signal = normalize_signal(signal, train_mean, train_std)
        signals_mm[i] = signal

    del signals_mm

    np.save(label_path, single_labels)
    np.save(multilabel_path, multilabels)

    split_df.insert(0, "record_idx", np.arange(n, dtype=np.int64))
    split_df.to_csv(metadata_path, index=False)

    return {
        "split": split_name,
        "n_records": int(n),
        "n_unique_patients": int(split_df["patient_id"].nunique()),
        "signals_file": str(signal_path),
        "labels_file": str(label_path),
        "multilabel_file": str(multilabel_path),
        "metadata_file": str(metadata_path),
        "single_label_distribution": {
            name: int((single_labels == CLASS_TO_INT[name]).sum())
            for name in CLASS_NAMES
        },
        "multilabel_positive_counts": {
            name: int(multilabels[:, CLASS_TO_INT[name]].sum())
            for name in CLASS_NAMES
        },
    }


def patient_overlap_check(metadata: pd.DataFrame) -> Dict[str, List[int]]:
    patients = {
        split: set(
            metadata.loc[
                metadata["split"] == split,
                "patient_id",
            ]
            .astype(int)
            .tolist()
        )
        for split in ["train", "val", "test"]
    }

    return {
        "train_val": sorted(list(patients["train"] & patients["val"])),
        "train_test": sorted(list(patients["train"] & patients["test"])),
        "val_test": sorted(list(patients["val"] & patients["test"])),
    }


def validate_saved_split(split_name: str) -> Dict[str, Any]:
    signals = np.load(OUTPUT_DIR / f"{split_name}_signals.npy", mmap_mode="r")
    labels = np.load(OUTPUT_DIR / f"{split_name}_labels.npy")
    multilabels = np.load(OUTPUT_DIR / f"{split_name}_multilabel_labels.npy")
    meta = pd.read_csv(OUTPUT_DIR / f"{split_name}_metadata.csv")

    issues = []

    if signals.shape[0] != len(labels):
        issues.append("signals/labels length mismatch")

    if signals.shape[0] != len(multilabels):
        issues.append("signals/multilabel length mismatch")

    if signals.shape[0] != len(meta):
        issues.append("signals/metadata length mismatch")

    if signals.shape[1:] != (TARGET_LENGTH, TARGET_LEADS):
        issues.append(f"signal shape mismatch: {signals.shape}")

    if labels.ndim != 1:
        issues.append(f"single-label array should be 1D, got {labels.shape}")

    if multilabels.shape[1] != len(CLASS_NAMES):
        issues.append(f"multilabel columns mismatch, got {multilabels.shape}")

    check_indices = np.unique(
        np.linspace(
            0,
            len(signals) - 1,
            num=min(10, len(signals)),
            dtype=int,
        )
    )

    for idx in check_indices:
        arr = np.asarray(signals[idx])

        if np.isnan(arr).any():
            issues.append(f"NaN found in signal index {idx}")
            break

        if np.isinf(arr).any():
            issues.append(f"Inf found in signal index {idx}")
            break

    return {
        "split": split_name,
        "signals_shape": tuple(int(x) for x in signals.shape),
        "labels_shape": tuple(int(x) for x in labels.shape),
        "multilabel_shape": tuple(int(x) for x in multilabels.shape),
        "metadata_rows": int(len(meta)),
        "unique_patients": int(meta["patient_id"].nunique()),
        "issues": issues,
        "ok": len(issues) == 0,
    }


def write_summary(
    metadata: pd.DataFrame,
    train_mean: np.ndarray,
    train_std: np.ndarray,
    split_stats: List[Dict[str, Any]],
    validation_stats: List[Dict[str, Any]],
) -> None:
    overlaps = patient_overlap_check(metadata)
    total_unique_patients = int(metadata["patient_id"].nunique())
    total_records = int(len(metadata))

    dataset_stats = {
        "dataset": "PTB-XL",
        "root_dir": str(ROOT_DIR),
        "output_dir": str(OUTPUT_DIR),
        "total_records_after_filter": total_records,
        "total_unique_patients_after_filter": total_unique_patients,
        "sampling_rate_hz": USE_SAMPLING_RATE,
        "target_length": TARGET_LENGTH,
        "target_leads": TARGET_LEADS,
        "split_policy": {
            "train": sorted(list(TRAIN_FOLDS)),
            "val": sorted(list(VAL_FOLDS)),
            "test": sorted(list(TEST_FOLDS)),
        },
        "normalization": "Train-set global per-lead z-score",
        "train_per_lead_mean": [float(x) for x in train_mean],
        "train_per_lead_std": [float(x) for x in train_std],
        "class_names": CLASS_NAMES,
        "single_label_policy": (
            "Dominant diagnostic superclass by maximum SCP confidence score; "
            "tie-break priority MI > STTC > CD > HYP > NORM."
        ),
        "single_label_distribution_all": {
            name: int((metadata["label_name"] == name).sum())
            for name in CLASS_NAMES
        },
        "multilabel_positive_counts_all": {
            name: int(metadata[f"multilabel_{name}"].sum())
            for name in CLASS_NAMES
        },
        "split_stats": split_stats,
        "patient_overlap": {key: len(value) for key, value in overlaps.items()},
        "patient_overlap_ids": overlaps,
        "validation_stats": validation_stats,
    }

    with open(OUTPUT_DIR / "dataset_stats.json", "w", encoding="utf-8") as file:
        json.dump(dataset_stats, file, indent=2)

    lines = [
        "=" * 90,
        "PTB-XL PREPROCESSING SUMMARY",
        "=" * 90,
        f"Root directory: {ROOT_DIR}",
        f"Output directory: {OUTPUT_DIR}",
        "",
        f"Records retained: {total_records}",
        f"Unique patients retained: {total_unique_patients}",
        f"Signal shape per ECG: ({TARGET_LENGTH}, {TARGET_LEADS})",
        f"Sampling rate: {USE_SAMPLING_RATE} Hz",
        "",
        "Split policy:",
        "  Train = strat_fold 1-8",
        "  Val   = strat_fold 9",
        "  Test  = strat_fold 10",
        "",
        "Patient overlap check:",
    ]

    for key, value in overlaps.items():
        lines.append(f"  {key}: {len(value)} overlapping patients")

    lines.append("")
    lines.append("Single-label distribution:")

    for name in CLASS_NAMES:
        count = int((metadata["label_name"] == name).sum())
        pct = 100.0 * count / max(total_records, 1)
        lines.append(f"  {name}: {count} ({pct:.2f}%)")

    lines.append("")
    lines.append("Multi-label positive counts:")

    for name in CLASS_NAMES:
        count = int(metadata[f"multilabel_{name}"].sum())
        pct = 100.0 * count / max(total_records, 1)
        lines.append(f"  {name}: {count} ({pct:.2f}% positive)")

    lines.append("")
    lines.append("Saved split details:")

    for item in split_stats:
        lines.append(
            f"  {item['split']}: records={item['n_records']}, "
            f"patients={item['n_unique_patients']}, "
            f"single_label_distribution={item['single_label_distribution']}"
        )

    lines.append("")
    lines.append("Validation:")

    for item in validation_stats:
        lines.append(
            f"  {item['split']}: ok={item['ok']}, "
            f"signals={item['signals_shape']}, "
            f"labels={item['labels_shape']}, "
            f"multilabel={item['multilabel_shape']}, "
            f"issues={item['issues']}"
        )

    lines.append("")
    lines.append("Files created per split:")
    lines.append("  - {split}_signals.npy")
    lines.append("  - {split}_labels.npy")
    lines.append("  - {split}_multilabel_labels.npy")
    lines.append("  - {split}_metadata.csv")
    lines.append("=" * 90)

    summary_text = "\n".join(lines)

    print("\n" + summary_text)

    with open(
        OUTPUT_DIR / "preprocessing_summary.txt",
        "w",
        encoding="utf-8",
    ) as file:
        file.write(summary_text)


def main() -> None:
    print("=" * 90)
    print("PTB-XL PREPROCESSING PIPELINE")
    print("=" * 90)
    print(f"ROOT_DIR   : {ROOT_DIR}")
    print(f"OUTPUT_DIR : {OUTPUT_DIR}")
    print("=" * 90)

    ensure_required_files()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    db, scp_diag = load_official_metadata()

    print(f"Loaded ptbxl_database.csv records: {len(db)}")
    print(f"Loaded diagnostic SCP statements: {len(scp_diag)}")

    metadata = attach_labels_and_split(db, scp_diag)
    metadata = metadata.sort_values("ecg_id").reset_index(drop=True)
    metadata.to_csv(
        OUTPUT_DIR / "all_metadata_preprocessed_index.csv",
        index=False,
    )

    print(f"Records retained after diagnostic filtering: {len(metadata)}")
    print(f"Unique patient_id count: {metadata['patient_id'].nunique()}")
    print("Split record counts:")
    print(metadata["split"].value_counts().to_string())

    overlaps = patient_overlap_check(metadata)

    if any(len(value) > 0 for value in overlaps.values()):
        raise RuntimeError(
            f"Patient leakage detected across splits: "
            f"{ {key: len(value) for key, value in overlaps.items()} }"
        )

    print("Patient overlap across train/val/test: none")

    train_mean, train_std = compute_train_global_per_lead_stats(metadata)

    np.save(OUTPUT_DIR / "train_global_per_lead_mean.npy", train_mean)
    np.save(OUTPUT_DIR / "train_global_per_lead_std.npy", train_std)

    print("Saved training normalization statistics.")

    split_stats = []

    for split_name in ["train", "val", "test"]:
        split_stats.append(
            save_one_split(metadata, split_name, train_mean, train_std)
        )

    validation_stats = [
        validate_saved_split(split_name)
        for split_name in ["train", "val", "test"]
    ]

    if not all(item["ok"] for item in validation_stats):
        print("WARNING: Some validation checks reported issues.")

        for item in validation_stats:
            if not item["ok"]:
                print(item)
    else:
        print("All saved split validation checks passed")

    write_summary(metadata, train_mean, train_std, split_stats, validation_stats)

    print("\nDONE. PTB-XL preprocessing completed.")
    print(f"Output folder: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

chapman preprocessing

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

ROOT_DIR = Path("CHAPMAN_ROOT")
RAW_ECG_DIR = ROOT_DIR / "ECGData"
DIAGNOSTICS_XLSX = ROOT_DIR / "Diagnostics.xlsx"
OUT_DIR = ROOT_DIR / "processed_chapman_4class"

TARGET_LENGTH = 5000
N_LEADS = 12
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAINVAL = 0.125
RANDOM_STATE = 42

DTYPE = np.float32
EPS = 1e-8
CENTER_CROP_IF_LONGER = False

GROUP_ORDER = ["SB", "AFIB", "GSVT", "SR"]
GROUP_TO_LABEL = {group: idx for idx, group in enumerate(GROUP_ORDER)}
LABEL_TO_GROUP = {idx: group for group, idx in GROUP_TO_LABEL.items()}

ORIGINAL_RHYTHM_TO_GROUP = {
    "SB": "SB",
    "AFIB": "AFIB",
    "AF": "AFIB",
    "SVT": "GSVT",
    "ST": "GSVT",
    "AT": "GSVT",
    "AVNRT": "GSVT",
    "AVRT": "GSVT",
    "SAAWR": "GSVT",
    "SR": "SR",
    "SA": "SR",
}


def ensure_required_inputs() -> None:
    required = [RAW_ECG_DIR, DIAGNOSTICS_XLSX]
    missing = [str(path) for path in required if not path.exists()]

    if missing:
        raise FileNotFoundError(
            "Missing required Chapman input(s):\n" + "\n".join(missing)
        )


def normalize_rhythm_string(value: Any) -> str:
    return str(value).strip().upper()


def load_and_prepare_diagnostics() -> pd.DataFrame:
    print("Loading Diagnostics.xlsx")

    diag = pd.read_excel(DIAGNOSTICS_XLSX)

    required_cols = {"FileName", "Rhythm"}
    missing_cols = required_cols - set(diag.columns)

    if missing_cols:
        raise KeyError(f"Diagnostics.xlsx missing required columns: {missing_cols}")

    diag = diag.copy()
    diag["FileName"] = diag["FileName"].astype(str).str.strip()
    diag["Rhythm_original"] = diag["Rhythm"].apply(normalize_rhythm_string)
    diag["FileName_csv"] = diag["FileName"] + ".csv"

    unknown_rhythms = sorted(
        set(diag["Rhythm_original"].unique())
        - set(ORIGINAL_RHYTHM_TO_GROUP.keys())
    )

    if unknown_rhythms:
        print(f"Unmapped rhythms dropped: {unknown_rhythms}")

    before_group_filter = len(diag)
    diag["Rhythm_group"] = diag["Rhythm_original"].map(ORIGINAL_RHYTHM_TO_GROUP)
    diag = diag[diag["Rhythm_group"].notna()].copy()
    dropped_unknown = before_group_filter - len(diag)

    diag["label"] = diag["Rhythm_group"].map(GROUP_TO_LABEL).astype(np.int64)

    available_files = {path.name for path in RAW_ECG_DIR.glob("*.csv")}
    before_file_filter = len(diag)
    diag = diag[diag["FileName_csv"].isin(available_files)].copy()
    dropped_missing_files = before_file_filter - len(diag)

    duplicate_files = int(diag["FileName_csv"].duplicated().sum())

    if duplicate_files > 0:
        print(f"Duplicate FileName_csv rows removed: {duplicate_files}")
        diag = diag.drop_duplicates(subset=["FileName_csv"], keep="first").copy()

    diag = diag.reset_index(drop=True)

    print(f"Records after rhythm filtering: {before_group_filter - dropped_unknown}")
    print(f"Dropped unmapped rhythms: {dropped_unknown}")
    print(f"Dropped missing ECG files: {dropped_missing_files}")
    print(f"Final valid samples: {len(diag)}")
    print("Class distribution:")
    print(diag["Rhythm_group"].value_counts().reindex(GROUP_ORDER, fill_value=0).to_string())

    return diag


def split_dataframe(diag: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    trainval_df, test_df = train_test_split(
        diag,
        test_size=TEST_SIZE,
        stratify=diag["label"],
        random_state=RANDOM_STATE,
    )

    train_df, val_df = train_test_split(
        trainval_df,
        test_size=VAL_SIZE_WITHIN_TRAINVAL,
        stratify=trainval_df["label"],
        random_state=RANDOM_STATE,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print("\nSplit sizes:")
    print(f"Train: {len(train_df)}")
    print(f"Val: {len(val_df)}")
    print(f"Test: {len(test_df)}")

    return train_df, val_df, test_df


def load_ecg_raw_csv(filepath: Path) -> np.ndarray:
    if not filepath.exists():
        raise FileNotFoundError(f"Missing ECG CSV file: {filepath}")

    df = pd.read_csv(filepath)

    missing_leads = [lead for lead in LEAD_NAMES if lead not in df.columns]

    if missing_leads:
        raise KeyError(f"Missing lead columns in {filepath.name}: {missing_leads}")

    signal = df[LEAD_NAMES].to_numpy(dtype=DTYPE, copy=True)

    if signal.ndim != 2 or signal.shape[1] != N_LEADS:
        raise ValueError(f"Unexpected signal shape for {filepath.name}: {signal.shape}")

    if np.isnan(signal).any() or np.isinf(signal).any():
        signal = np.nan_to_num(
            signal,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(DTYPE, copy=False)

    length = signal.shape[0]

    if length > TARGET_LENGTH:
        if CENTER_CROP_IF_LONGER:
            start = (length - TARGET_LENGTH) // 2
            signal = signal[start : start + TARGET_LENGTH, :]
        else:
            signal = signal[:TARGET_LENGTH, :]
    elif length < TARGET_LENGTH:
        if length == 0:
            raise ValueError(f"Empty signal file: {filepath.name}")

        pad_len = TARGET_LENGTH - length
        signal = np.pad(signal, ((0, pad_len), (0, 0)), mode="edge")

    if signal.shape != (TARGET_LENGTH, N_LEADS):
        raise ValueError(
            f"Signal shape mismatch after processing {filepath.name}: {signal.shape}"
        )

    return signal.astype(DTYPE, copy=False)


def compute_train_global_per_lead_stats(
    train_df: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray]:
    print("\nComputing train normalization statistics")

    lead_sum = np.zeros(N_LEADS, dtype=np.float64)
    lead_sumsq = np.zeros(N_LEADS, dtype=np.float64)
    total_timepoints = 0

    for _, row in tqdm(
        train_df.iterrows(),
        total=len(train_df),
        desc="Train stats",
    ):
        filepath = RAW_ECG_DIR / row["FileName_csv"]
        signal = load_ecg_raw_csv(filepath)

        lead_sum += signal.sum(axis=0, dtype=np.float64)
        lead_sumsq += np.square(signal, dtype=np.float64).sum(
            axis=0,
            dtype=np.float64,
        )
        total_timepoints += signal.shape[0]

    means = lead_sum / max(1, total_timepoints)
    variances = (lead_sumsq / max(1, total_timepoints)) - np.square(means)
    variances = np.maximum(variances, EPS)

    stds = np.sqrt(variances)
    stds[stds < 1e-6] = 1.0

    means = means.astype(DTYPE)
    stds = stds.astype(DTYPE)

    print("Lead means:", means)
    print("Lead stds:", stds)

    return means, stds


def standardize_signal(
    signal: np.ndarray,
    means: np.ndarray,
    stds: np.ndarray,
) -> np.ndarray:
    return ((signal - means[None, :]) / stds[None, :]).astype(
        DTYPE,
        copy=False,
    )


def save_split(
    df: pd.DataFrame,
    split_name: str,
    means: np.ndarray,
    stds: np.ndarray,
) -> Dict[str, Any]:
    print(f"\nSaving {split_name}")

    n = len(df)

    signals_path = OUT_DIR / f"{split_name}_signals.npy"
    labels_path = OUT_DIR / f"{split_name}_labels.npy"
    metadata_path = OUT_DIR / f"{split_name}_metadata.csv"

    signals_mm = np.lib.format.open_memmap(
        signals_path,
        mode="w+",
        dtype=DTYPE,
        shape=(n, TARGET_LENGTH, N_LEADS),
    )

    labels = np.zeros(n, dtype=np.int64)
    metadata_rows: List[Dict[str, Any]] = []

    for i, (_, row) in enumerate(
        tqdm(df.iterrows(), total=n, desc=f"Processing {split_name}")
    ):
        filepath = RAW_ECG_DIR / row["FileName_csv"]
        signal = load_ecg_raw_csv(filepath)
        signal = standardize_signal(signal, means, stds)

        signals_mm[i] = signal
        labels[i] = int(row["label"])

        metadata_rows.append(
            {
                "record_idx": int(i),
                "FileName": str(row["FileName"]),
                "FileName_csv": str(row["FileName_csv"]),
                "patient_id": str(row["FileName"]),
                "Rhythm_original": str(row["Rhythm_original"]),
                "Rhythm_group": str(row["Rhythm_group"]),
                "label": int(row["label"]),
                "split": split_name,
            }
        )

    del signals_mm

    np.save(labels_path, labels)

    metadata_df = pd.DataFrame(metadata_rows)
    metadata_df.to_csv(metadata_path, index=False)

    class_counts = {
        GROUP_ORDER[class_idx]: int((labels == class_idx).sum())
        for class_idx in range(len(GROUP_ORDER))
    }

    print(f"{split_name} signals: ({n}, {TARGET_LENGTH}, {N_LEADS})")
    print(f"{split_name} labels: {labels.shape}")
    print(f"{split_name} class counts: {class_counts}")

    return {
        "split": split_name,
        "n_samples": int(n),
        "signals_path": str(signals_path),
        "labels_path": str(labels_path),
        "metadata_path": str(metadata_path),
        "class_counts": class_counts,
    }


def check_split_disjointness(
    train_meta: pd.DataFrame,
    val_meta: pd.DataFrame,
    test_meta: pd.DataFrame,
) -> Dict[str, int]:
    train_ids = set(train_meta["patient_id"].astype(str).tolist())
    val_ids = set(val_meta["patient_id"].astype(str).tolist())
    test_ids = set(test_meta["patient_id"].astype(str).tolist())

    return {
        "train_val": len(train_ids & val_ids),
        "train_test": len(train_ids & test_ids),
        "val_test": len(val_ids & test_ids),
    }


def validate_saved_split(split_name: str) -> Dict[str, Any]:
    signals = np.load(OUT_DIR / f"{split_name}_signals.npy", mmap_mode="r")
    labels = np.load(OUT_DIR / f"{split_name}_labels.npy")
    metadata = pd.read_csv(OUT_DIR / f"{split_name}_metadata.csv")

    issues: List[str] = []

    if len(signals) != len(labels):
        issues.append("signals/labels count mismatch")

    if len(signals) != len(metadata):
        issues.append("signals/metadata count mismatch")

    if signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
        issues.append(f"signal shape mismatch: {signals.shape}")

    if labels.ndim != 1:
        issues.append(f"labels should be 1D, got {labels.shape}")

    seen_labels = sorted(np.unique(labels).tolist())

    if seen_labels != list(range(len(GROUP_ORDER))):
        issues.append(f"not all labels present; labels seen={seen_labels}")

    if len(signals) > 0:
        check_indices = np.unique(
            np.linspace(
                0,
                len(signals) - 1,
                num=min(10, len(signals)),
                dtype=int,
            )
        )

        for idx in check_indices:
            arr = np.asarray(signals[idx])

            if np.isnan(arr).any():
                issues.append(f"NaN found in signal index {idx}")
                break

            if np.isinf(arr).any():
                issues.append(f"Inf found in signal index {idx}")
                break

    return {
        "split": split_name,
        "signals_shape": tuple(int(x) for x in signals.shape),
        "labels_shape": tuple(int(x) for x in labels.shape),
        "metadata_rows": int(len(metadata)),
        "class_distribution": {
            LABEL_TO_GROUP[int(k)]: int(v)
            for k, v in zip(*np.unique(labels, return_counts=True))
        },
        "ok": len(issues) == 0,
        "issues": issues,
    }


def build_all_metadata_split_index() -> pd.DataFrame:
    parts = []

    for split_name in ["train", "val", "test"]:
        part = pd.read_csv(OUT_DIR / f"{split_name}_metadata.csv")
        parts.append(part)

    all_meta = pd.concat(parts, axis=0, ignore_index=True)
    all_meta.to_csv(OUT_DIR / "all_metadata_split_index.csv", index=False)

    return all_meta


def save_class_mapping_json() -> None:
    payload = {
        "group_order": GROUP_ORDER,
        "group_to_label": GROUP_TO_LABEL,
        "label_to_group": {str(key): value for key, value in LABEL_TO_GROUP.items()},
        "original_rhythm_to_group": ORIGINAL_RHYTHM_TO_GROUP,
        "lead_order": LEAD_NAMES,
        "signal_length": TARGET_LENGTH,
        "n_leads": N_LEADS,
    }

    with open(
        OUT_DIR / "chapman_4class_class_mapping.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(payload, file, indent=2)


def save_stats_and_summary(
    diag: pd.DataFrame,
    means: np.ndarray,
    stds: np.ndarray,
    split_stats: List[Dict[str, Any]],
    validations: List[Dict[str, Any]],
    overlaps: Dict[str, int],
    all_meta: pd.DataFrame,
) -> None:
    original_rhythm_counts = diag["Rhythm_original"].value_counts().to_dict()
    grouped_counts = (
        diag["Rhythm_group"]
        .value_counts()
        .reindex(GROUP_ORDER, fill_value=0)
        .to_dict()
    )

    stats_payload = {
        "dataset": "Chapman-Shaoxing ECG dataset",
        "source_waveform_folder": str(RAW_ECG_DIR),
        "diagnostics_file": str(DIAGNOSTICS_XLSX),
        "output_dir": str(OUT_DIR),
        "task": "4-class grouped rhythm classification",
        "group_order": GROUP_ORDER,
        "group_to_label": GROUP_TO_LABEL,
        "original_rhythm_to_group": ORIGINAL_RHYTHM_TO_GROUP,
        "total_records_retained": int(len(diag)),
        "signal_shape_per_record": [TARGET_LENGTH, N_LEADS],
        "lead_order": LEAD_NAMES,
        "split_policy": {
            "train": "70%",
            "val": "10%",
            "test": "20%",
            "random_state": RANDOM_STATE,
        },
        "original_rhythm_counts": {
            str(key): int(value)
            for key, value in original_rhythm_counts.items()
        },
        "grouped_class_counts": {
            str(key): int(value)
            for key, value in grouped_counts.items()
        },
        "train_global_per_lead_mean": [float(x) for x in means],
        "train_global_per_lead_std": [float(x) for x in stds],
        "split_stats": split_stats,
        "split_overlap_patient_id_counts": overlaps,
        "validation": validations,
        "all_metadata_rows": int(len(all_meta)),
    }

    with open(
        OUT_DIR / "chapman_preprocessing_stats.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(stats_payload, file, indent=2)

    lines: List[str] = [
        "=" * 90,
        "CHAPMAN 4-CLASS PREPROCESSING SUMMARY",
        "=" * 90,
        f"Root directory: {ROOT_DIR}",
        f"Waveform folder: {RAW_ECG_DIR}",
        f"Output directory: {OUT_DIR}",
        "",
        "Task: 4-class grouped rhythm classification",
    ]

    for group in GROUP_ORDER:
        label = GROUP_TO_LABEL[group]
        source_rhythms = [
            key for key, value in ORIGINAL_RHYTHM_TO_GROUP.items()
            if value == group
        ]
        lines.append(f"Label {label}: {group} <- {', '.join(source_rhythms)}")

    lines.extend(
        [
            "",
            f"Records retained: {len(diag)}",
            f"Signal shape per ECG: ({TARGET_LENGTH}, {N_LEADS})",
            f"Lead order: {LEAD_NAMES}",
            "",
            "Grouped class counts:",
        ]
    )

    for group in GROUP_ORDER:
        lines.append(f"{group}: {int(grouped_counts[group])}")

    lines.append("")
    lines.append("Split statistics:")

    for item in split_stats:
        lines.append(
            f"{item['split']}: samples={item['n_samples']}, "
            f"class_counts={item['class_counts']}"
        )

    lines.append("")
    lines.append("Split overlap audit:")

    for key, value in overlaps.items():
        lines.append(f"{key}: {value}")

    lines.append("")
    lines.append("Validation checks:")

    for item in validations:
        lines.append(
            f"{item['split']}: ok={item['ok']}, "
            f"signals={item['signals_shape']}, "
            f"labels={item['labels_shape']}, "
            f"metadata_rows={item['metadata_rows']}, "
            f"issues={item['issues']}"
        )

    lines.append("")
    lines.append("Normalization: train-only global per-lead z-score")
    lines.append("=" * 90)

    summary_text = "\n".join(lines)

    print("\n" + summary_text)

    with open(
        OUT_DIR / "chapman_preprocessing_summary.txt",
        "w",
        encoding="utf-8",
    ) as file:
        file.write(summary_text)


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 90)
    print("CHAPMAN 4-CLASS PREPROCESSING")
    print("=" * 90)
    print(f"ROOT_DIR: {ROOT_DIR}")
    print(f"RAW_ECG_DIR: {RAW_ECG_DIR}")
    print(f"DIAGNOSTICS_XLSX: {DIAGNOSTICS_XLSX}")
    print(f"OUT_DIR: {OUT_DIR}")
    print("=" * 90)

    ensure_required_inputs()

    diag = load_and_prepare_diagnostics()
    train_df, val_df, test_df = split_dataframe(diag)

    means, stds = compute_train_global_per_lead_stats(train_df)

    np.save(OUT_DIR / "train_global_per_lead_mean.npy", means)
    np.save(OUT_DIR / "train_global_per_lead_std.npy", stds)

    split_stats = [
        save_split(train_df, "train", means, stds),
        save_split(val_df, "val", means, stds),
        save_split(test_df, "test", means, stds),
    ]

    validations = [
        validate_saved_split(split_name)
        for split_name in ["train", "val", "test"]
    ]

    if not all(item["ok"] for item in validations):
        print("Some validation checks reported issues")

        for item in validations:
            if not item["ok"]:
                print(item)
    else:
        print("All validation checks passed")

    train_meta = pd.read_csv(OUT_DIR / "train_metadata.csv")
    val_meta = pd.read_csv(OUT_DIR / "val_metadata.csv")
    test_meta = pd.read_csv(OUT_DIR / "test_metadata.csv")

    overlaps = check_split_disjointness(train_meta, val_meta, test_meta)

    if any(value > 0 for value in overlaps.values()):
        raise RuntimeError(f"Split overlap detected: {overlaps}")

    print(f"Split overlap audit passed: {overlaps}")

    all_meta = build_all_metadata_split_index()

    save_class_mapping_json()
    save_stats_and_summary(
        diag=diag,
        means=means,
        stds=stds,
        split_stats=split_stats,
        validations=validations,
        overlaps=overlaps,
        all_meta=all_meta,
    )

    print("\nDONE. Chapman preprocessing completed.")
    print(f"Output folder: {OUT_DIR}")


if __name__ == "__main__":
    main()

01_ptbxl_policy_training

In [ ]:
from __future__ import annotations

import csv
import json
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


DATA_DIR = Path("your_data_directory")


OUTPUT_DIR = DATA_DIR / "lead_masking_policy_ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXISTING_P060_MIXED_RESULTS_CSV = (
    DATA_DIR / "supervised_lead_masked_controls" / "masked_control_reduced_lead_results.csv"
)


CLASS_NAMES = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES = len(CLASS_NAMES)

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

BATCH_SIZE = 128
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

INPUT_CHANNELS = 12
BASE_FILTERS = 64
RESNET_BLOCKS = [2, 2, 2, 2]
EMBED_DIM = 512
DROPOUT = 0.10

EPOCHS = 70
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
MIN_LR_RATIO = 0.02
GRAD_CLIP_NORM = 5.0
EARLY_STOPPING_PATIENCE = 15

LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

CLINICAL_LEAD_SETS: Tuple[Tuple[int, ...], ...] = (
    tuple(range(12)),            # Full 12-lead
    (0, 1, 2, 3, 4, 5),          # 6 limb leads
    (6, 7, 8, 9, 10, 11),        # 6 precordial leads
    (0, 1, 2),                   # I, II, III
    (1,),                        # Lead II only
    (10,),                       # V5 only
)

MIN_RANDOM_KEPT_LEADS = 1
MAX_RANDOM_KEPT_LEADS = 12

ABLATION_VARIANTS = [
    {
        "name": "p030_mixed",
        "display_name": "p=0.30 Mixed",
        "lead_mask_prob": 0.30,
        "mask_policy": "mixed",
        "clinical_subset_prob": 0.50,
    },
    {
        "name": "p090_mixed",
        "display_name": "p=0.90 Mixed",
        "lead_mask_prob": 0.90,
        "mask_policy": "mixed",
        "clinical_subset_prob": 0.50,
    },
    {
        "name": "p060_clinical_only",
        "display_name": "p=0.60 Clinical Only",
        "lead_mask_prob": 0.60,
        "mask_policy": "clinical_only",
        "clinical_subset_prob": 1.00,
    },
    {
        "name": "p060_random_only",
        "display_name": "p=0.60 Random Only",
        "lead_mask_prob": 0.60,
        "mask_policy": "random_only",
        "clinical_subset_prob": 0.00,
    },
]

LEAD_CONFIGS = {
    "12_lead_full": {
        "display_name": "12-lead full",
        "lead_indices": list(range(12)),
    },
    "6_limb": {
        "display_name": "6 limb leads",
        "lead_indices": [0, 1, 2, 3, 4, 5],
    },
    "6_precordial": {
        "display_name": "6 precordial leads",
        "lead_indices": [6, 7, 8, 9, 10, 11],
    },
    "3_limb_I_II_III": {
        "display_name": "3 limb leads (I, II, III)",
        "lead_indices": [0, 1, 2],
    },
    "lead_II_only": {
        "display_name": "Lead II only",
        "lead_indices": [1],
    },
    "V5_only": {
        "display_name": "V5 only",
        "lead_indices": [10],
    },
}


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)


class LeadMaskAugment:
    """
    Lead masking augmentation for supervised training.

    Input:
      x: numpy array [T, 12]

    Output:
      masked x: numpy array [T, 12]
      meta: dictionary for audit logging
    """

    def __init__(
        self,
        lead_mask_prob: float,
        mask_policy: str,
        clinical_subset_prob: float = 0.50,
        min_random_kept_leads: int = MIN_RANDOM_KEPT_LEADS,
        max_random_kept_leads: int = MAX_RANDOM_KEPT_LEADS,
    ):
        self.lead_mask_prob = float(lead_mask_prob)
        self.mask_policy = str(mask_policy)
        self.clinical_subset_prob = float(clinical_subset_prob)
        self.min_random_kept_leads = int(min_random_kept_leads)
        self.max_random_kept_leads = int(max_random_kept_leads)

        if self.mask_policy not in {"mixed", "clinical_only", "random_only"}:
            raise ValueError(f"Unsupported mask_policy: {self.mask_policy}")

    def __call__(self, x: np.ndarray) -> Tuple[np.ndarray, Dict[str, Any]]:
        x = np.asarray(x, dtype=np.float32).copy()
        if x.ndim != 2 or x.shape[1] != 12:
            raise ValueError(f"Expected signal shape (T, 12), got {x.shape}")

        default_meta = {
            "lead_mask_applied": False,
            "mask_type": "none",
            "kept_leads": list(range(12)),
            "kept_lead_count": 12,
        }

        if random.random() >= self.lead_mask_prob:
            return x, default_meta

        if self.mask_policy == "clinical_only":
            kept = tuple(sorted(random.choice(CLINICAL_LEAD_SETS)))
            mask_type = "clinical_subset"
        elif self.mask_policy == "random_only":
            k = random.randint(self.min_random_kept_leads, self.max_random_kept_leads)
            kept = tuple(sorted(random.sample(range(12), k=k)))
            mask_type = "random_subset"
        else:  # mixed
            if random.random() < self.clinical_subset_prob:
                kept = tuple(sorted(random.choice(CLINICAL_LEAD_SETS)))
                mask_type = "clinical_subset"
            else:
                k = random.randint(self.min_random_kept_leads, self.max_random_kept_leads)
                kept = tuple(sorted(random.sample(range(12), k=k)))
                mask_type = "random_subset"

        dropped = [i for i in range(12) if i not in kept]
        if dropped:
            x[:, dropped] = 0.0

        meta = {
            "lead_mask_applied": True,
            "mask_type": mask_type,
            "kept_leads": list(map(int, kept)),
            "kept_lead_count": int(len(kept)),
        }
        return x, meta


class PTBXLAblationTrainDataset(Dataset):
    def __init__(
        self,
        signals_file: Path,
        labels_file: Path,
        metadata_file: Path,
        augmenter: LeadMaskAugment,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(signals_file, mmap_mode=mmap_mode)
        self.labels = np.load(labels_file).astype(np.int64)
        self.metadata = pd.read_csv(metadata_file)
        self.augmenter = augmenter
        self._validate()

    def _validate(self) -> None:
        if len(self.signals) != len(self.labels):
            raise ValueError(f"Signals/labels mismatch: {len(self.signals)} vs {len(self.labels)}")
        if len(self.signals) != len(self.metadata):
            raise ValueError(f"Signals/metadata mismatch: {len(self.signals)} vs {len(self.metadata)}")
        if self.signals.ndim != 3 or self.signals.shape[1:] != (5000, 12):
            raise ValueError(f"Expected train signals shape (N, 5000, 12), got {self.signals.shape}")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32).copy()
        y = int(self.labels[idx])
        x, aug_meta = self.augmenter(x)
        x = torch.from_numpy(x.T.copy()).float()  # [12, 5000]
        row = self.metadata.iloc[idx]
        return {
            "signal": x,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(row["record_idx"]) if "record_idx" in row else int(idx),
            "ecg_id": int(row["ecg_id"]) if "ecg_id" in row else -1,
            "patient_id": int(row["patient_id"]) if "patient_id" in row else -1,
            "aug_meta": aug_meta,
        }


class PTBXLFixedLeadEvalDataset(Dataset):
    def __init__(
        self,
        signals_file: Path,
        labels_file: Path,
        metadata_file: Path,
        lead_indices_to_keep: Optional[List[int]] = None,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(signals_file, mmap_mode=mmap_mode)
        self.labels = np.load(labels_file).astype(np.int64)
        self.metadata = pd.read_csv(metadata_file)
        self.lead_indices_to_keep = list(range(12)) if lead_indices_to_keep is None else sorted(list(map(int, lead_indices_to_keep)))
        self.lead_indices_to_zero = [i for i in range(12) if i not in self.lead_indices_to_keep]
        self._validate()

    def _validate(self) -> None:
        if len(self.signals) != len(self.labels):
            raise ValueError(f"Signals/labels mismatch: {len(self.signals)} vs {len(self.labels)}")
        if len(self.signals) != len(self.metadata):
            raise ValueError(f"Signals/metadata mismatch: {len(self.signals)} vs {len(self.metadata)}")
        if self.signals.ndim != 3 or self.signals.shape[1:] != (5000, 12):
            raise ValueError(f"Expected eval signals shape (N, 5000, 12), got {self.signals.shape}")
        if not self.lead_indices_to_keep:
            raise ValueError("At least one lead must be retained.")
        if min(self.lead_indices_to_keep) < 0 or max(self.lead_indices_to_keep) > 11:
            raise ValueError(f"Invalid lead indices: {self.lead_indices_to_keep}")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32).copy()
        y = int(self.labels[idx])
        if self.lead_indices_to_zero:
            x[:, self.lead_indices_to_zero] = 0.0
        x = torch.from_numpy(x.T.copy()).float()
        row = self.metadata.iloc[idx]
        return {
            "signal": x,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(row["record_idx"]) if "record_idx" in row else int(idx),
            "ecg_id": int(row["ecg_id"]) if "ecg_id" in row else -1,
            "patient_id": int(row["patient_id"]) if "patient_id" in row else -1,
        }


def train_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "signal": torch.stack([item["signal"] for item in batch], dim=0),
        "label": torch.stack([item["label"] for item in batch], dim=0),
        "record_idx": torch.tensor([item["record_idx"] for item in batch], dtype=torch.long),
        "ecg_id": torch.tensor([item["ecg_id"] for item in batch], dtype=torch.long),
        "patient_id": torch.tensor([item["patient_id"] for item in batch], dtype=torch.long),
        "aug_meta": [item["aug_meta"] for item in batch],
    }


def eval_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "signal": torch.stack([item["signal"] for item in batch], dim=0),
        "label": torch.stack([item["label"] for item in batch], dim=0),
        "record_idx": torch.tensor([item["record_idx"] for item in batch], dtype=torch.long),
        "ecg_id": torch.tensor([item["ecg_id"] for item in batch], dtype=torch.long),
        "patient_id": torch.tensor([item["patient_id"] for item in batch], dtype=torch.long),
    }


def make_train_and_val_loaders(variant: Dict[str, Any]) -> Tuple[PTBXLAblationTrainDataset, DataLoader, DataLoader]:
    augmenter = LeadMaskAugment(
        lead_mask_prob=variant["lead_mask_prob"],
        mask_policy=variant["mask_policy"],
        clinical_subset_prob=variant["clinical_subset_prob"],
    )

    train_ds = PTBXLAblationTrainDataset(
        signals_file=DATA_DIR / "train_signals.npy",
        labels_file=DATA_DIR / "train_labels.npy",
        metadata_file=DATA_DIR / "train_metadata.csv",
        augmenter=augmenter,
        mmap_mode="r",
    )
    val_ds = PTBXLFixedLeadEvalDataset(
        signals_file=DATA_DIR / "val_signals.npy",
        labels_file=DATA_DIR / "val_labels.npy",
        metadata_file=DATA_DIR / "val_metadata.csv",
        lead_indices_to_keep=list(range(12)),
        mmap_mode="r",
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=train_collate_fn,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=eval_collate_fn,
    )
    return train_ds, train_loader, val_loader


def make_test_loader(lead_indices_to_keep: List[int]) -> DataLoader:
    test_ds = PTBXLFixedLeadEvalDataset(
        signals_file=DATA_DIR / "test_signals.npy",
        labels_file=DATA_DIR / "test_labels.npy",
        metadata_file=DATA_DIR / "test_metadata.csv",
        lead_indices_to_keep=lead_indices_to_keep,
        mmap_mode="r",
    )
    return DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=eval_collate_fn,
    )


class BasicBlock1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, kernel_size: int = 7, dropout: float = 0.0):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(identity)
        out = out + identity
        out = self.relu(out)
        return out


class ResNet1DEncoder(nn.Module):
    def __init__(self, input_channels: int = INPUT_CHANNELS, base_filters: int = BASE_FILTERS, block_counts: List[int] = RESNET_BLOCKS, embedding_dim: int = EMBED_DIM, dropout: float = DROPOUT):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(input_channels, base_filters, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        channel_plan = [base_filters, base_filters * 2, base_filters * 4, base_filters * 8]
        in_channels = base_filters
        stages = []
        for stage_idx, (out_channels, n_blocks) in enumerate(zip(channel_plan, block_counts)):
            stride = 1 if stage_idx == 0 else 2
            stage, in_channels = self._make_stage(in_channels, out_channels, n_blocks, stride, dropout)
            stages.append(stage)
        self.backbone = nn.Sequential(*stages)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )
        self.embedding_dim = embedding_dim
        self._init_weights()

    @staticmethod
    def _make_stage(in_channels: int, out_channels: int, n_blocks: int, first_stride: int, dropout: float) -> Tuple[nn.Sequential, int]:
        blocks = [BasicBlock1D(in_channels, out_channels, stride=first_stride, kernel_size=7, dropout=dropout)]
        for _ in range(1, n_blocks):
            blocks.append(BasicBlock1D(out_channels, out_channels, stride=1, kernel_size=7, dropout=dropout))
        return nn.Sequential(*blocks), out_channels

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.backbone(x)
        x = self.global_pool(x)
        x = self.embedding_head(x)
        return x


class ECGClassifier(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.encoder = ResNet1DEncoder()
        self.classifier = nn.Linear(self.encoder.embedding_dim, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.encoder(x)
        return self.classifier(emb)


def metrics_from_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
    metrics: Dict[str, Any] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0,
    )
    metrics["per_class_precision"] = {CLASS_NAMES[i]: float(precision[i]) for i in range(NUM_CLASSES)}
    metrics["per_class_recall"] = {CLASS_NAMES[i]: float(recall[i]) for i in range(NUM_CLASSES)}
    metrics["per_class_f1"] = {CLASS_NAMES[i]: float(f1[i]) for i in range(NUM_CLASSES)}
    metrics["per_class_support"] = {CLASS_NAMES[i]: int(support[i]) for i in range(NUM_CLASSES)}
    metrics["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    return metrics


def summarize_aug_batch(aug_meta_list: List[Dict[str, Any]]) -> Dict[str, int]:
    summary = {
        "masked": 0,
        "unmasked": 0,
        "clinical_subset": 0,
        "random_subset": 0,
    }
    for meta in aug_meta_list:
        if meta["lead_mask_applied"]:
            summary["masked"] += 1
            if meta["mask_type"] in summary:
                summary[meta["mask_type"]] += 1
        else:
            summary["unmasked"] += 1
    return summary


def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, scaler: GradScaler) -> Dict[str, Any]:
    model.train()
    total_loss = 0.0
    total_items = 0
    all_true: List[int] = []
    all_pred: List[int] = []
    aug_totals = {"masked": 0, "unmasked": 0, "clinical_subset": 0, "random_subset": 0}

    for batch in tqdm(loader, desc="Training", leave=False):
        x = batch["signal"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        aug_summary = summarize_aug_batch(batch["aug_meta"])
        for k, v in aug_summary.items():
            aug_totals[k] += int(v)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss: {loss.item()}")

        scaler.scale(loss).backward()
        if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits, dim=1)
        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        total_items += bs
        all_true.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())

    y_true = np.asarray(all_true, dtype=np.int64)
    y_pred = np.asarray(all_pred, dtype=np.int64)
    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_items)
    metrics["aug_totals"] = aug_totals
    return metrics


def evaluate_model(model: nn.Module, loader: DataLoader, criterion: Optional[nn.Module] = None) -> Tuple[Dict[str, Any], pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_items = 0
    all_true: List[int] = []
    all_pred: List[int] = []
    all_prob: List[np.ndarray] = []
    all_record_idx: List[int] = []
    all_ecg_id: List[int] = []
    all_patient_id: List[int] = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            x = batch["signal"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)
            with autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y) if criterion is not None else None
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            bs = y.size(0)
            if loss is not None:
                total_loss += float(loss.item()) * bs
            total_items += bs
            all_true.extend(y.detach().cpu().numpy().tolist())
            all_pred.extend(preds.detach().cpu().numpy().tolist())
            all_prob.extend(probs.detach().cpu().numpy())
            all_record_idx.extend(batch["record_idx"].cpu().numpy().tolist())
            all_ecg_id.extend(batch["ecg_id"].cpu().numpy().tolist())
            all_patient_id.extend(batch["patient_id"].cpu().numpy().tolist())

    y_true = np.asarray(all_true, dtype=np.int64)
    y_pred = np.asarray(all_pred, dtype=np.int64)
    probs_arr = np.asarray(all_prob, dtype=np.float32)
    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_items) if criterion is not None else float("nan")

    pred_df = pd.DataFrame({
        "record_idx": all_record_idx,
        "ecg_id": all_ecg_id,
        "patient_id": all_patient_id,
        "y_true": y_true,
        "y_true_name": [CLASS_NAMES[i] for i in y_true],
        "y_pred": y_pred,
        "y_pred_name": [CLASS_NAMES[i] for i in y_pred],
    })
    for i, cls in enumerate(CLASS_NAMES):
        pred_df[f"prob_{cls}"] = probs_arr[:, i]
    return metrics, pred_df


def set_warmup_cosine_lr(optimizer: torch.optim.Optimizer, epoch: int, total_epochs: int, base_lr: float, warmup_epochs: int, min_lr_ratio: float) -> float:
    if epoch < warmup_epochs:
        lr = base_lr * float(epoch + 1) / float(max(1, warmup_epochs))
    else:
        progress = (epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr = base_lr * (min_lr_ratio + (1.0 - min_lr_ratio) * cosine)
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr


def save_history_csv(history: List[Dict[str, Any]], path: Path) -> None:
    if not history:
        return
    fieldnames = list(history[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(history)


def save_checkpoint(path: Path, model: nn.Module, epoch: int, best_val_macro_f1: float, variant: Dict[str, Any]) -> None:
    torch.save({
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "encoder_state_dict": model.encoder.state_dict(),
        "classifier_state_dict": model.classifier.state_dict(),
        "best_val_macro_f1": float(best_val_macro_f1),
        "variant": variant,
    }, path)


def load_checkpoint(path: Path, model: nn.Module) -> Dict[str, Any]:
    ckpt = torch.load(path, map_location="cpu")
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    if missing or unexpected:
        raise RuntimeError(f"Checkpoint mismatch. Missing={missing}, Unexpected={unexpected}")
    return ckpt


def save_eval_artifacts(variant_name: str, lead_key: str, metrics: Dict[str, Any], pred_df: pd.DataFrame) -> None:
    pred_df.to_csv(OUTPUT_DIR / f"predictions_{variant_name}_{lead_key}.csv", index=False)
    np.save(OUTPUT_DIR / f"confusion_{variant_name}_{lead_key}.npy", metrics["confusion_matrix"])
    report = classification_report(
        pred_df["y_true"].to_numpy(),
        pred_df["y_pred"].to_numpy(),
        labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
    with open(OUTPUT_DIR / f"classification_report_{variant_name}_{lead_key}.txt", "w", encoding="utf-8") as f:
        f.write(report)


def train_ablation_variant(variant: Dict[str, Any]) -> Dict[str, Any]:
    name = variant["name"]
    display_name = variant["display_name"]

    print("=" * 155)
    print(f"TRAINING ABLATION VARIANT: {display_name}")
    print("=" * 155)
    print(f"Mask probability: {variant['lead_mask_prob']}")
    print(f"Mask policy     : {variant['mask_policy']}")
    print(f"Clinical prob   : {variant['clinical_subset_prob']}")
    print("Loss             : Plain CrossEntropyLoss")
    print("Sampler          : Normal shuffle")
    print("=" * 155)

    seed_everything(SEED)
    train_ds, train_loader, val_loader = make_train_and_val_loaders(variant)
    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(device="cuda", enabled=USE_AMP)

    best_val_macro_f1 = -1.0
    best_epoch = -1
    patience = 0
    history: List[Dict[str, Any]] = []
    start = time.time()
    best_path = OUTPUT_DIR / f"best_{name}.pt"

    for epoch in range(EPOCHS):
        lr = set_warmup_cosine_lr(optimizer, epoch, EPOCHS, BASE_LR, WARMUP_EPOCHS, MIN_LR_RATIO)
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_metrics, _ = evaluate_model(model, val_loader, criterion)

        aug_totals = train_metrics["aug_totals"]
        total_seen = aug_totals["masked"] + aug_totals["unmasked"]
        observed_mask_rate = aug_totals["masked"] / max(1, total_seen)

        row = {
            "epoch": epoch + 1,
            "lr": lr,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "observed_train_mask_rate": observed_mask_rate,
            "observed_train_masked_count": aug_totals["masked"],
            "observed_train_unmasked_count": aug_totals["unmasked"],
            "observed_train_clinical_subset_count": aug_totals["clinical_subset"],
            "observed_train_random_subset_count": aug_totals["random_subset"],
        }
        history.append(row)

        print(
            f"{name} | Epoch {epoch + 1:03d}/{EPOCHS} | lr={lr:.3e} | "
            f"train: acc={row['train_accuracy']:.4f}, macro_f1={row['train_macro_f1']:.4f}, mask_rate={observed_mask_rate:.3f} | "
            f"val-full: acc={row['val_accuracy']:.4f}, bal_acc={row['val_balanced_accuracy']:.4f}, macro_f1={row['val_macro_f1']:.4f}"
        )

        if row["val_macro_f1"] > best_val_macro_f1 + 1e-6:
            best_val_macro_f1 = row["val_macro_f1"]
            best_epoch = epoch + 1
            patience = 0
            save_checkpoint(best_path, model, epoch + 1, best_val_macro_f1, variant)
        else:
            patience += 1

        save_history_csv(history, OUTPUT_DIR / f"history_{name}.csv")

        if patience >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping {name} at epoch {epoch + 1}; best epoch={best_epoch}.")
            break

    runtime_minutes = (time.time() - start) / 60.0
    return {
        "variant_name": name,
        "variant_display_name": display_name,
        "lead_mask_prob": float(variant["lead_mask_prob"]),
        "mask_policy": variant["mask_policy"],
        "clinical_subset_prob": float(variant["clinical_subset_prob"]),
        "checkpoint": str(best_path),
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_macro_f1),
        "runtime_minutes": float(runtime_minutes),
    }


def evaluate_variant_grid(training_result: Dict[str, Any]) -> List[Dict[str, Any]]:
    variant_name = training_result["variant_name"]
    display_name = training_result["variant_display_name"]
    ckpt_path = Path(training_result["checkpoint"])

    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    load_checkpoint(ckpt_path, model)
    model.eval()

    rows: List[Dict[str, Any]] = []
    print("\n" + "=" * 155)
    print(f"REDUCED-LEAD GRID: {display_name}")
    print("=" * 155)

    for lead_key, lead_info in LEAD_CONFIGS.items():
        lead_display = lead_info["display_name"]
        lead_indices = lead_info["lead_indices"]
        kept_names = [LEAD_NAMES[i] for i in lead_indices]
        print(f"Evaluating {display_name} | {lead_display} | kept={kept_names}")
        loader = make_test_loader(lead_indices)
        metrics, pred_df = evaluate_model(model, loader, criterion=None)
        save_eval_artifacts(variant_name, lead_key, metrics, pred_df)
        print(
            f"  ACC={metrics['accuracy']:.4f} | Bal-ACC={metrics['balanced_accuracy']:.4f} | "
            f"Macro-F1={metrics['macro_f1']:.4f} | Weighted-F1={metrics['weighted_f1']:.4f}"
        )

        rows.append({
            "variant_name": variant_name,
            "variant_display_name": display_name,
            "lead_mask_prob": training_result["lead_mask_prob"],
            "mask_policy": training_result["mask_policy"],
            "clinical_subset_prob": training_result["clinical_subset_prob"],
            "lead_key": lead_key,
            "lead_display_name": lead_display,
            "kept_leads": ",".join(kept_names),
            "n_kept_leads": len(lead_indices),
            "accuracy": metrics["accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "per_class_recall_json": json.dumps(metrics["per_class_recall"]),
            "per_class_f1_json": json.dumps(metrics["per_class_f1"]),
        })
    return rows


def convert_existing_step10_p060_mixed() -> Optional[pd.DataFrame]:
    if not EXISTING_P060_MIXED_RESULTS_CSV.exists():
        print(f"Existing Step 10 p=0.60 mixed results not found: {EXISTING_P060_MIXED_RESULTS_CSV}")
        return None

    df = pd.read_csv(EXISTING_P060_MIXED_RESULTS_CSV)
    needed = df[df["model_key"] == "supervised_masked_plain_ce"].copy()
    if needed.empty:
        print("Existing Step 10 CSV found, but supervised_masked_plain_ce rows were not present.")
        return None

    converted = pd.DataFrame({
        "variant_name": "p060_mixed_existing",
        "variant_display_name": "p=0.60 Mixed [Step 10]",
        "lead_mask_prob": 0.60,
        "mask_policy": "mixed",
        "clinical_subset_prob": 0.50,
        "lead_key": needed["lead_key"].values,
        "lead_display_name": needed["lead_display_name"].values,
        "kept_leads": needed["kept_leads"].values,
        "n_kept_leads": needed["n_kept_leads"].values,
        "accuracy": needed["accuracy"].values,
        "balanced_accuracy": needed["balanced_accuracy"].values,
        "macro_f1": needed["macro_f1"].values,
        "weighted_f1": needed["weighted_f1"].values,
        "per_class_recall_json": needed["per_class_recall_json"].values,
        "per_class_f1_json": needed["per_class_f1_json"].values,
    })
    return converted


def add_retention_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    full = out[out["lead_key"] == "12_lead_full"].copy()
    acc_map = dict(zip(full["variant_name"], full["accuracy"]))
    macro_map = dict(zip(full["variant_name"], full["macro_f1"]))
    bal_map = dict(zip(full["variant_name"], full["balanced_accuracy"]))

    out["accuracy_retention_vs_own_12lead"] = out.apply(
        lambda r: float(r["accuracy"] / acc_map[r["variant_name"]]) if acc_map[r["variant_name"]] > 0 else np.nan,
        axis=1,
    )
    out["macro_f1_retention_vs_own_12lead"] = out.apply(
        lambda r: float(r["macro_f1"] / macro_map[r["variant_name"]]) if macro_map[r["variant_name"]] > 0 else np.nan,
        axis=1,
    )
    out["balanced_accuracy_retention_vs_own_12lead"] = out.apply(
        lambda r: float(r["balanced_accuracy"] / bal_map[r["variant_name"]]) if bal_map[r["variant_name"]] > 0 else np.nan,
        axis=1,
    )
    return out


def save_final_summary(training_results: List[Dict[str, Any]], new_eval_rows: List[Dict[str, Any]]) -> None:
    training_df = pd.DataFrame(training_results)
    new_eval_df = pd.DataFrame(new_eval_rows)
    new_eval_df = add_retention_columns(new_eval_df)

    training_df.to_csv(OUTPUT_DIR / "ablation_training_results.csv", index=False)
    new_eval_df.to_csv(OUTPUT_DIR / "ablation_reduced_lead_results_new_only.csv", index=False)

    existing_df = convert_existing_step10_p060_mixed()
    if existing_df is not None:
        combined_df = pd.concat([existing_df, new_eval_df.drop(columns=[c for c in [
            "accuracy_retention_vs_own_12lead",
            "macro_f1_retention_vs_own_12lead",
            "balanced_accuracy_retention_vs_own_12lead",
        ] if c in new_eval_df.columns])], ignore_index=True)
        combined_df = add_retention_columns(combined_df)
    else:
        combined_df = new_eval_df.copy()

    combined_df.to_csv(OUTPUT_DIR / "ablation_reduced_lead_results_with_existing_p060_mixed.csv", index=False)

    lead_order = [v["display_name"] for v in LEAD_CONFIGS.values()]
    macro_wide = combined_df.pivot(index="lead_display_name", columns="variant_display_name", values="macro_f1").reindex(lead_order)
    acc_wide = combined_df.pivot(index="lead_display_name", columns="variant_display_name", values="accuracy").reindex(lead_order)
    ret_macro_wide = combined_df.pivot(index="lead_display_name", columns="variant_display_name", values="macro_f1_retention_vs_own_12lead").reindex(lead_order)

    macro_wide.to_csv(OUTPUT_DIR / "ablation_macro_f1_wide_with_existing.csv")
    acc_wide.to_csv(OUTPUT_DIR / "ablation_accuracy_wide_with_existing.csv")
    ret_macro_wide.to_csv(OUTPUT_DIR / "ablation_macro_f1_retention_wide_with_existing.csv")

    reduced_only = combined_df[combined_df["lead_key"] != "12_lead_full"].copy()
    avg_macro = reduced_only.groupby("variant_display_name", as_index=False)["macro_f1"].mean().sort_values("macro_f1", ascending=False)
    avg_ret = reduced_only.groupby("variant_display_name", as_index=False)["macro_f1_retention_vs_own_12lead"].mean().sort_values("macro_f1_retention_vs_own_12lead", ascending=False)

    lines: List[str] = []
    lines.append("=" * 160)
    lines.append("STEP 11 — LEAD-MASKING POLICY ABLATION SUMMARY")
    lines.append("=" * 160)
    lines.append("Goal: justify the lead-masking probability and the clinical/random subset policy before manuscript writing.")
    lines.append("")
    lines.append("TRAINING RESULTS FOR NEW VARIANTS")
    for _, row in training_df.iterrows():
        lines.append(
            f"  {row['variant_display_name']:<30s} | best_epoch={int(row['best_epoch']):<3d} | "
            f"best_val_macro_f1={row['best_val_macro_f1']:.6f} | runtime={row['runtime_minutes']:.2f} min"
        )
    lines.append("")
    lines.append("ABSOLUTE MACRO-F1 ACROSS LEAD SETTINGS")
    lines.append(macro_wide.to_string(float_format=lambda x: f"{x:.6f}"))
    lines.append("")
    lines.append("ABSOLUTE ACCURACY ACROSS LEAD SETTINGS")
    lines.append(acc_wide.to_string(float_format=lambda x: f"{x:.6f}"))
    lines.append("")
    lines.append("MACRO-F1 RETENTION VS OWN 12-LEAD PERFORMANCE")
    lines.append(ret_macro_wide.to_string(float_format=lambda x: f"{x:.6f}"))
    lines.append("")
    lines.append("AVERAGE REDUCED-LEAD MACRO-F1 (excluding 12-lead full)")
    for _, row in avg_macro.iterrows():
        lines.append(f"  {row['variant_display_name']:<30s} | avg_reduced_macro_f1={row['macro_f1']:.6f}")
    lines.append("")
    lines.append("AVERAGE REDUCED-LEAD MACRO-F1 RETENTION (excluding 12-lead full)")
    for _, row in avg_ret.iterrows():
        lines.append(f"  {row['variant_display_name']:<30s} | avg_retention={row['macro_f1_retention_vs_own_12lead']:.6f}")
    lines.append("")
    lines.append("BEST VARIANT BY ABSOLUTE MACRO-F1 FOR EACH LEAD CONFIGURATION")
    for lead_key, lead_info in LEAD_CONFIGS.items():
        subset = combined_df[combined_df["lead_key"] == lead_key].sort_values("macro_f1", ascending=False)
        if len(subset) > 0:
            row = subset.iloc[0]
            lines.append(
                f"  {lead_info['display_name']:<30s} -> {row['variant_display_name']:<30s} "
                f"Macro-F1={row['macro_f1']:.6f}, ACC={row['accuracy']:.6f}"
            )
    lines.append("")
    lines.append(f"Artifacts saved in: {OUTPUT_DIR}")
    lines.append("=" * 160)

    summary_text = "\n".join(lines)
    print("\n" + summary_text)
    with open(OUTPUT_DIR / "lead_masking_policy_ablation_summary.txt", "w", encoding="utf-8") as f:
        f.write(summary_text)


if __name__ == "__main__":
    print("=" * 160)
    print("STEP 11 — LEAD-MASKING POLICY ABLATION")
    print("=" * 160)
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Data directory: {DATA_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"New variants to train: {[v['name'] for v in ABLATION_VARIANTS]}")
    print(f"Existing p=0.60 mixed Step 10 CSV will be used if found: {EXISTING_P060_MIXED_RESULTS_CSV}")
    print("=" * 160)

    training_results: List[Dict[str, Any]] = []
    all_new_eval_rows: List[Dict[str, Any]] = []

    for variant in ABLATION_VARIANTS:
        tr = train_ablation_variant(variant)
        training_results.append(tr)
        rows = evaluate_variant_grid(tr)
        all_new_eval_rows.extend(rows)

    save_final_summary(training_results, all_new_eval_rows)

02_ptbxl_bootstrap_perclass_analysis

In [ ]:
from __future__ import annotations

import os
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)

DATA_DIR = Path(os.getenv("DATA_DIR", Path.cwd() / "Data"))

BASELINE_PRED_DIR = DATA_DIR / "reduced_lead_robustness_evaluation"
FINAL_METHOD_PRED_DIR = DATA_DIR / "lead_masking_policy_ablation"
OUTPUT_DIR = DATA_DIR / "paper_ready_statistical_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_BOOTSTRAPS = 2000
CI_LOW = 2.5
CI_HIGH = 97.5

CLASS_NAMES = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES = len(CLASS_NAMES)

LEAD_CONFIGS = [
    ("12_lead_full", "12-lead full"),
    ("6_limb", "6 limb leads"),
    ("6_precordial", "6 precordial leads"),
    ("3_limb_I_II_III", "3 limb leads (I, II, III)"),
    ("lead_II_only", "Lead II only"),
    ("V5_only", "V5 only"),
]

MODEL_CONFIGS = {
    "supervised_plain_ce": {
        "display_name": "Standard Supervised",
        "pred_dir": BASELINE_PRED_DIR,
        "filename_template": "predictions_supervised_plain_ce_{lead_key}.csv",
    },
    "p060_clinical_only": {
        "display_name": "Clinical Lead-Masked p=0.60",
        "pred_dir": FINAL_METHOD_PRED_DIR,
        "filename_template": "predictions_p060_clinical_only_{lead_key}.csv",
    },
}

METRIC_NAMES = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]

rng = np.random.default_rng(SEED)
random.seed(SEED)
np.random.seed(SEED)

def compute_global_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

def compute_per_class_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Dict[str, float]]:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0,
    )
    return {
        CLASS_NAMES[i]: {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
        for i in range(NUM_CLASSES)
    }

def bootstrap_metric_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_bootstraps: int = N_BOOTSTRAPS,
) -> Dict[str, Tuple[float, float, float]]:
    n = len(y_true)
    point = compute_global_metrics(y_true, y_pred)
    draws = {m: [] for m in METRIC_NAMES}

    for _ in tqdm(range(n_bootstraps), desc="Bootstrap metric CI", leave=False):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        metrics = compute_global_metrics(yt, yp)
        for m in METRIC_NAMES:
            draws[m].append(metrics[m])

    out = {}
    for m in METRIC_NAMES:
        arr = np.asarray(draws[m], dtype=np.float64)
        out[m] = (
            float(point[m]),
            float(np.percentile(arr, CI_LOW)),
            float(np.percentile(arr, CI_HIGH)),
        )
    return out

def paired_bootstrap_gain_ci(
    y_true: np.ndarray,
    baseline_pred: np.ndarray,
    final_pred: np.ndarray,
    n_bootstraps: int = N_BOOTSTRAPS,
) -> Dict[str, Tuple[float, float, float, float]]:
    n = len(y_true)
    base_point = compute_global_metrics(y_true, baseline_pred)
    final_point = compute_global_metrics(y_true, final_pred)
    point_gain = {m: final_point[m] - base_point[m] for m in METRIC_NAMES}
    draws = {m: [] for m in METRIC_NAMES}

    for _ in tqdm(range(n_bootstraps), desc="Paired bootstrap gain CI", leave=False):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yb = baseline_pred[idx]
        yf = final_pred[idx]
        mb = compute_global_metrics(yt, yb)
        mf = compute_global_metrics(yt, yf)
        for m in METRIC_NAMES:
            draws[m].append(mf[m] - mb[m])

    out = {}
    for m in METRIC_NAMES:
        arr = np.asarray(draws[m], dtype=np.float64)
        out[m] = (
            float(point_gain[m]),
            float(np.percentile(arr, CI_LOW)),
            float(np.percentile(arr, CI_HIGH)),
            float(np.mean(arr <= 0.0)),
        )
    return out

def bootstrap_retention_ci(
    y_true_full: np.ndarray,
    y_pred_full: np.ndarray,
    y_true_reduced: np.ndarray,
    y_pred_reduced: np.ndarray,
    metric_name: str,
    n_bootstraps: int = N_BOOTSTRAPS,
) -> Tuple[float, float, float]:
    if not np.array_equal(y_true_full, y_true_reduced):
        raise ValueError("Full-lead and reduced-lead y_true arrays are not aligned.")

    point_full = compute_global_metrics(y_true_full, y_pred_full)[metric_name]
    point_reduced = compute_global_metrics(y_true_reduced, y_pred_reduced)[metric_name]
    point_retention = float(point_reduced / point_full) if point_full > 0 else np.nan

    n = len(y_true_full)
    draws = []
    for _ in tqdm(range(n_bootstraps), desc=f"Bootstrap retention {metric_name}", leave=False):
        idx = rng.integers(0, n, size=n)
        yt = y_true_full[idx]
        pf = y_pred_full[idx]
        pr = y_pred_reduced[idx]
        mf = compute_global_metrics(yt, pf)[metric_name]
        mr = compute_global_metrics(yt, pr)[metric_name]
        if mf > 0:
            draws.append(mr / mf)

    arr = np.asarray(draws, dtype=np.float64)
    return (
        point_retention,
        float(np.percentile(arr, CI_LOW)),
        float(np.percentile(arr, CI_HIGH)),
    )

def load_prediction_df(model_key: str, lead_key: str) -> pd.DataFrame:
    cfg = MODEL_CONFIGS[model_key]
    path = cfg["pred_dir"] / cfg["filename_template"].format(lead_key=lead_key)
    if not path.exists():
        raise FileNotFoundError(f"Missing prediction file: {path}")
    df = pd.read_csv(path)
    required = {"record_idx", "ecg_id", "patient_id", "y_true", "y_pred"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")
    return df.sort_values("record_idx").reset_index(drop=True)

def assert_alignment(df_a: pd.DataFrame, df_b: pd.DataFrame, context: str) -> None:
    for col in ["record_idx", "ecg_id", "patient_id", "y_true"]:
        if not np.array_equal(df_a[col].to_numpy(), df_b[col].to_numpy()):
            raise ValueError(f"Alignment mismatch for column '{col}' in {context}")

def run_analysis() -> None:
    print("=" * 160)
    print("STEP 12 — PAPER-READY PER-CLASS ANALYSIS + BOOTSTRAP CONFIDENCE INTERVALS")
    print("=" * 160)
    print(f"Baseline prediction dir: {BASELINE_PRED_DIR}")
    print(f"Final-method prediction dir: {FINAL_METHOD_PRED_DIR}")
    print(f"Output dir: {OUTPUT_DIR}")
    print(f"Bootstrap resamples: {N_BOOTSTRAPS}")
    print("=" * 160)

    metric_ci_rows: List[Dict[str, Any]] = []
    paired_gain_rows: List[Dict[str, Any]] = []
    per_class_recall_rows: List[Dict[str, Any]] = []
    per_class_f1_rows: List[Dict[str, Any]] = []
    per_class_gain_rows: List[Dict[str, Any]] = []
    retention_rows: List[Dict[str, Any]] = []

    pred_cache: Dict[Tuple[str, str], pd.DataFrame] = {}

    for lead_key, lead_display in LEAD_CONFIGS:
        baseline_df = load_prediction_df("supervised_plain_ce", lead_key)
        final_df = load_prediction_df("p060_clinical_only", lead_key)
        assert_alignment(baseline_df, final_df, context=f"lead={lead_key}")
        pred_cache[("supervised_plain_ce", lead_key)] = baseline_df
        pred_cache[("p060_clinical_only", lead_key)] = final_df

        y_true = baseline_df["y_true"].to_numpy(dtype=np.int64)
        y_base = baseline_df["y_pred"].to_numpy(dtype=np.int64)
        y_final = final_df["y_pred"].to_numpy(dtype=np.int64)

        print(f"\nAnalyzing lead configuration: {lead_display}")

        for model_key, y_pred in [
            ("supervised_plain_ce", y_base),
            ("p060_clinical_only", y_final),
        ]:
            model_display = MODEL_CONFIGS[model_key]["display_name"]
            metric_cis = bootstrap_metric_ci(y_true, y_pred)
            for metric_name, (point, low, high) in metric_cis.items():
                metric_ci_rows.append({
                    "lead_key": lead_key,
                    "lead_display_name": lead_display,
                    "model_key": model_key,
                    "model_display_name": model_display,
                    "metric": metric_name,
                    "point_estimate": point,
                    "ci95_low": low,
                    "ci95_high": high,
                })

        gain_cis = paired_bootstrap_gain_ci(y_true, y_base, y_final)
        for metric_name, (point, low, high, p_nonpositive) in gain_cis.items():
            paired_gain_rows.append({
                "lead_key": lead_key,
                "lead_display_name": lead_display,
                "comparison": "Clinical Lead-Masked p=0.60 minus Standard Supervised",
                "metric": metric_name,
                "point_gain": point,
                "ci95_low": low,
                "ci95_high": high,
                "bootstrap_fraction_gain_leq_zero": p_nonpositive,
            })

        per_base = compute_per_class_metrics(y_true, y_base)
        per_final = compute_per_class_metrics(y_true, y_final)

        for cls in CLASS_NAMES:
            per_class_recall_rows.append({
                "lead_key": lead_key,
                "lead_display_name": lead_display,
                "class_name": cls,
                "standard_supervised_recall": per_base[cls]["recall"],
                "clinical_masked_recall": per_final[cls]["recall"],
                "recall_gain_final_minus_standard": per_final[cls]["recall"] - per_base[cls]["recall"],
                "support": per_base[cls]["support"],
            })
            per_class_f1_rows.append({
                "lead_key": lead_key,
                "lead_display_name": lead_display,
                "class_name": cls,
                "standard_supervised_f1": per_base[cls]["f1"],
                "clinical_masked_f1": per_final[cls]["f1"],
                "f1_gain_final_minus_standard": per_final[cls]["f1"] - per_base[cls]["f1"],
                "support": per_base[cls]["support"],
            })
            per_class_gain_rows.append({
                "lead_key": lead_key,
                "lead_display_name": lead_display,
                "class_name": cls,
                "recall_gain_final_minus_standard": per_final[cls]["recall"] - per_base[cls]["recall"],
                "f1_gain_final_minus_standard": per_final[cls]["f1"] - per_base[cls]["f1"],
                "support": per_base[cls]["support"],
            })

    for model_key in MODEL_CONFIGS.keys():
        model_display = MODEL_CONFIGS[model_key]["display_name"]
        full_df = pred_cache[(model_key, "12_lead_full")]
        y_true_full = full_df["y_true"].to_numpy(dtype=np.int64)
        y_pred_full = full_df["y_pred"].to_numpy(dtype=np.int64)

        for lead_key, lead_display in LEAD_CONFIGS:
            reduced_df = pred_cache[(model_key, lead_key)]
            y_true_reduced = reduced_df["y_true"].to_numpy(dtype=np.int64)
            y_pred_reduced = reduced_df["y_pred"].to_numpy(dtype=np.int64)

            for metric_name in ["accuracy", "macro_f1", "balanced_accuracy"]:
                point, low, high = bootstrap_retention_ci(
                    y_true_full=y_true_full,
                    y_pred_full=y_pred_full,
                    y_true_reduced=y_true_reduced,
                    y_pred_reduced=y_pred_reduced,
                    metric_name=metric_name,
                )
                retention_rows.append({
                    "model_key": model_key,
                    "model_display_name": model_display,
                    "lead_key": lead_key,
                    "lead_display_name": lead_display,
                    "metric": metric_name,
                    "retention_point_estimate": point,
                    "ci95_low": low,
                    "ci95_high": high,
                })

    metric_ci_df = pd.DataFrame(metric_ci_rows)
    paired_gain_df = pd.DataFrame(paired_gain_rows)
    recall_df = pd.DataFrame(per_class_recall_rows)
    f1_df = pd.DataFrame(per_class_f1_rows)
    gain_df = pd.DataFrame(per_class_gain_rows)
    retention_df = pd.DataFrame(retention_rows)

    metric_ci_df.to_csv(OUTPUT_DIR / "metric_summary_with_bootstrap_ci.csv", index=False)
    paired_gain_df.to_csv(OUTPUT_DIR / "paired_gain_bootstrap_ci_final_vs_supervised.csv", index=False)
    recall_df.to_csv(OUTPUT_DIR / "per_class_recall_comparison.csv", index=False)
    f1_df.to_csv(OUTPUT_DIR / "per_class_f1_comparison.csv", index=False)
    gain_df.to_csv(OUTPUT_DIR / "per_class_gain_final_minus_supervised.csv", index=False)
    retention_df.to_csv(OUTPUT_DIR / "retention_summary_with_ci.csv", index=False)

    summary_lines: List[str] = []
    summary_lines.append("=" * 170)
    summary_lines.append("STEP 12 — PAPER-READY STATISTICAL SUMMARY")
    summary_lines.append("=" * 170)
    summary_lines.append("Comparison: Clinical Lead-Masked p=0.60 vs Standard Supervised")
    summary_lines.append(f"Bootstrap resamples: {N_BOOTSTRAPS}")
    summary_lines.append("")

    summary_lines.append("PAIRED BOOTSTRAP MACRO-F1 GAINS")
    macro_gain = paired_gain_df[paired_gain_df["metric"] == "macro_f1"].copy()
    for _, row in macro_gain.iterrows():
        summary_lines.append(
            f"  {row['lead_display_name']:<32s} | gain={row['point_gain']:+.6f} "
            f"[95% CI {row['ci95_low']:+.6f}, {row['ci95_high']:+.6f}] | "
            f"Pr(gain<=0)={row['bootstrap_fraction_gain_leq_zero']:.4f}"
        )
    summary_lines.append("")

    summary_lines.append("PAIRED BOOTSTRAP ACCURACY GAINS")
    acc_gain = paired_gain_df[paired_gain_df["metric"] == "accuracy"].copy()
    for _, row in acc_gain.iterrows():
        summary_lines.append(
            f"  {row['lead_display_name']:<32s} | gain={row['point_gain']:+.6f} "
            f"[95% CI {row['ci95_low']:+.6f}, {row['ci95_high']:+.6f}] | "
            f"Pr(gain<=0)={row['bootstrap_fraction_gain_leq_zero']:.4f}"
        )
    summary_lines.append("")

    summary_lines.append("TOP PER-CLASS F1 GAINS UNDER REDUCED-LEAD SETTINGS")
    reduced_gain_df = gain_df[gain_df["lead_key"] != "12_lead_full"].copy()
    top_gains = reduced_gain_df.sort_values("f1_gain_final_minus_standard", ascending=False).head(15)
    for _, row in top_gains.iterrows():
        summary_lines.append(
            f"  {row['lead_display_name']:<32s} | {row['class_name']:<5s} | "
            f"F1 gain={row['f1_gain_final_minus_standard']:+.6f}, "
            f"Recall gain={row['recall_gain_final_minus_standard']:+.6f}, support={int(row['support'])}"
        )
    summary_lines.append("")

    summary_lines.append("OUTPUT FILES")
    for fn in [
        "metric_summary_with_bootstrap_ci.csv",
        "paired_gain_bootstrap_ci_final_vs_supervised.csv",
        "per_class_recall_comparison.csv",
        "per_class_f1_comparison.csv",
        "per_class_gain_final_minus_supervised.csv",
        "retention_summary_with_ci.csv",
    ]:
        summary_lines.append(f"  - {OUTPUT_DIR / fn}")
    summary_lines.append("=" * 170)

    summary_text = "\n".join(summary_lines)
    print("\n" + summary_text)
    with open(OUTPUT_DIR / "paper_ready_statistical_summary.txt", "w", encoding="utf-8") as f:
        f.write(summary_text)

if __name__ == "__main__":
    run_analysis()


03_chapman_standard_clinical_training

In [ ]:
from __future__ import annotations

import csv
import os
import json
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

DATA_DIR = Path(os.getenv("DATA_DIR", "data"))

OUT_DIR = DATA_DIR / "chapman_external_validation_step14"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

CLASS_NAMES = ["SB", "AFIB", "GSVT", "SR"]
NUM_CLASSES = len(CLASS_NAMES)

TARGET_LENGTH = 5000
N_LEADS = 12
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

BATCH_SIZE = 128
NUM_WORKERS = 0                                                                
PIN_MEMORY = torch.cuda.is_available()

INPUT_CHANNELS = 12
BASE_FILTERS = 64
RESNET_BLOCKS = [2, 2, 2, 2]
EMBED_DIM = 512
DROPOUT = 0.10

EPOCHS = 70
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
MIN_LR_RATIO = 0.02
GRAD_CLIP_NORM = 5.0
EARLY_STOPPING_PATIENCE = 15

CLINICAL_MASK_PROB = 0.60
CLINICAL_LEAD_SETS: Tuple[Tuple[int, ...], ...] = (
    tuple(range(12)),                          
    (0, 1, 2, 3, 4, 5),                        
    (6, 7, 8, 9, 10, 11),                            
    (0, 1, 2),                               
    (1,),                                      
    (10,),                                
)

MODEL_VARIANTS = [
    {
        "model_key": "chapman_standard_supervised",
        "display_name": "Chapman Standard Supervised",
        "use_clinical_lead_masking": False,
        "lead_mask_prob": 0.00,
    },
    {
        "model_key": "chapman_clinical_masked_p060",
        "display_name": "Chapman Clinical Lead-Masked p=0.60",
        "use_clinical_lead_masking": True,
        "lead_mask_prob": CLINICAL_MASK_PROB,
    },
]

LEAD_CONFIGS = {
    "12_lead_full": {
        "display_name": "12-lead full",
        "lead_indices": list(range(12)),
    },
    "6_limb": {
        "display_name": "6 limb leads",
        "lead_indices": [0, 1, 2, 3, 4, 5],
    },
    "6_precordial": {
        "display_name": "6 precordial leads",
        "lead_indices": [6, 7, 8, 9, 10, 11],
    },
    "3_limb_I_II_III": {
        "display_name": "3 limb leads (I, II, III)",
        "lead_indices": [0, 1, 2],
    },
    "lead_II_only": {
        "display_name": "Lead II only",
        "lead_indices": [1],
    },
    "V5_only": {
        "display_name": "V5 only",
        "lead_indices": [10],
    },
}

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

def verify_required_preprocessed_files() -> None:
    required_files = [
        DATA_DIR / "train_signals.npy",
        DATA_DIR / "val_signals.npy",
        DATA_DIR / "test_signals.npy",
        DATA_DIR / "train_labels.npy",
        DATA_DIR / "val_labels.npy",
        DATA_DIR / "test_labels.npy",
        DATA_DIR / "train_metadata.csv",
        DATA_DIR / "val_metadata.csv",
        DATA_DIR / "test_metadata.csv",
        DATA_DIR / "chapman_4class_class_mapping.json",
        DATA_DIR / "chapman_preprocessing_summary.txt",
    ]
    missing = [str(p) for p in required_files if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required Step 13 Chapman preprocessing outputs:\n" + "\n".join(missing))

class ClinicalLeadMaskAugment:

    def __init__(self, lead_mask_prob: float = CLINICAL_MASK_PROB):
        self.lead_mask_prob = float(lead_mask_prob)

    def __call__(self, signal: np.ndarray) -> Tuple[np.ndarray, Dict[str, Any]]:
        x = np.asarray(signal, dtype=np.float32).copy()           
        if x.ndim != 2 or x.shape[1] != N_LEADS:
            raise ValueError(f"Expected ECG shape (T, {N_LEADS}), got {x.shape}")

        meta = {
            "lead_mask_applied": False,
            "mask_type": "none",
            "kept_leads": list(range(N_LEADS)),
            "kept_lead_count": N_LEADS,
        }

        if random.random() >= self.lead_mask_prob:
            return x, meta

        kept = tuple(sorted(random.choice(CLINICAL_LEAD_SETS)))
        dropped = [i for i in range(N_LEADS) if i not in kept]
        if dropped:
            x[:, dropped] = 0.0

        meta = {
            "lead_mask_applied": True,
            "mask_type": "clinical_subset",
            "kept_leads": list(map(int, kept)),
            "kept_lead_count": int(len(kept)),
        }
        return x, meta

class ChapmanTrainDataset(Dataset):

    def __init__(
        self,
        signals_file: Path,
        labels_file: Path,
        metadata_file: Path,
        use_clinical_lead_masking: bool,
        lead_mask_prob: float = CLINICAL_MASK_PROB,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(signals_file, mmap_mode=mmap_mode)
        self.labels = np.load(labels_file).astype(np.int64)
        self.metadata = pd.read_csv(metadata_file)
        self.use_clinical_lead_masking = bool(use_clinical_lead_masking)
        self.augmenter = ClinicalLeadMaskAugment(lead_mask_prob=lead_mask_prob) if self.use_clinical_lead_masking else None
        self._validate()

    def _validate(self) -> None:
        if len(self.signals) != len(self.labels):
            raise ValueError(f"Signals/labels mismatch: {len(self.signals)} vs {len(self.labels)}")
        if len(self.signals) != len(self.metadata):
            raise ValueError(f"Signals/metadata mismatch: {len(self.signals)} vs {len(self.metadata)}")
        if self.signals.ndim != 3 or self.signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
            raise ValueError(f"Expected train signals shape (N, {TARGET_LENGTH}, {N_LEADS}), got {self.signals.shape}")
        labels_seen = sorted(np.unique(self.labels).tolist())
        if labels_seen != list(range(NUM_CLASSES)):
            raise ValueError(f"Expected labels [0..{NUM_CLASSES-1}], got {labels_seen}")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32).copy()              
        y = int(self.labels[idx])

        if self.augmenter is not None:
            x, aug_meta = self.augmenter(x)
        else:
            aug_meta = {
                "lead_mask_applied": False,
                "mask_type": "none",
                "kept_leads": list(range(N_LEADS)),
                "kept_lead_count": N_LEADS,
            }

        x = torch.from_numpy(x.T.copy()).float()                      
        row = self.metadata.iloc[idx]

        return {
            "signal": x,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(row["record_idx"]) if "record_idx" in row else int(idx),
            "file_name": str(row["FileName_csv"]) if "FileName_csv" in row else "",
            "patient_id": str(row["patient_id"]) if "patient_id" in row else "",
            "aug_meta": aug_meta,
        }

class ChapmanFixedLeadEvalDataset(Dataset):

    def __init__(
        self,
        signals_file: Path,
        labels_file: Path,
        metadata_file: Path,
        lead_indices_to_keep: Optional[List[int]] = None,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(signals_file, mmap_mode=mmap_mode)
        self.labels = np.load(labels_file).astype(np.int64)
        self.metadata = pd.read_csv(metadata_file)
        self.lead_indices_to_keep = list(range(N_LEADS)) if lead_indices_to_keep is None else sorted(list(map(int, lead_indices_to_keep)))
        self.lead_indices_to_zero = [i for i in range(N_LEADS) if i not in self.lead_indices_to_keep]
        self._validate()

    def _validate(self) -> None:
        if len(self.signals) != len(self.labels):
            raise ValueError(f"Signals/labels mismatch: {len(self.signals)} vs {len(self.labels)}")
        if len(self.signals) != len(self.metadata):
            raise ValueError(f"Signals/metadata mismatch: {len(self.signals)} vs {len(self.metadata)}")
        if self.signals.ndim != 3 or self.signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
            raise ValueError(f"Expected eval signals shape (N, {TARGET_LENGTH}, {N_LEADS}), got {self.signals.shape}")
        if not self.lead_indices_to_keep:
            raise ValueError("At least one lead must be kept.")
        if min(self.lead_indices_to_keep) < 0 or max(self.lead_indices_to_keep) >= N_LEADS:
            raise ValueError(f"Invalid lead indices: {self.lead_indices_to_keep}")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32).copy()
        y = int(self.labels[idx])

        if self.lead_indices_to_zero:
            x[:, self.lead_indices_to_zero] = 0.0

        x = torch.from_numpy(x.T.copy()).float()
        row = self.metadata.iloc[idx]

        return {
            "signal": x,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(row["record_idx"]) if "record_idx" in row else int(idx),
            "file_name": str(row["FileName_csv"]) if "FileName_csv" in row else "",
            "patient_id": str(row["patient_id"]) if "patient_id" in row else "",
        }

def train_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "signal": torch.stack([b["signal"] for b in batch], dim=0),
        "label": torch.stack([b["label"] for b in batch], dim=0),
        "record_idx": torch.tensor([b["record_idx"] for b in batch], dtype=torch.long),
        "file_name": [b["file_name"] for b in batch],
        "patient_id": [b["patient_id"] for b in batch],
        "aug_meta": [b["aug_meta"] for b in batch],
    }

def eval_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "signal": torch.stack([b["signal"] for b in batch], dim=0),
        "label": torch.stack([b["label"] for b in batch], dim=0),
        "record_idx": torch.tensor([b["record_idx"] for b in batch], dtype=torch.long),
        "file_name": [b["file_name"] for b in batch],
        "patient_id": [b["patient_id"] for b in batch],
    }

def make_train_val_loaders(variant: Dict[str, Any]) -> Tuple[ChapmanTrainDataset, DataLoader, DataLoader]:
    train_ds = ChapmanTrainDataset(
        signals_file=DATA_DIR / "train_signals.npy",
        labels_file=DATA_DIR / "train_labels.npy",
        metadata_file=DATA_DIR / "train_metadata.csv",
        use_clinical_lead_masking=variant["use_clinical_lead_masking"],
        lead_mask_prob=variant["lead_mask_prob"],
        mmap_mode="r",
    )
    val_ds = ChapmanFixedLeadEvalDataset(
        signals_file=DATA_DIR / "val_signals.npy",
        labels_file=DATA_DIR / "val_labels.npy",
        metadata_file=DATA_DIR / "val_metadata.csv",
        lead_indices_to_keep=list(range(N_LEADS)),
        mmap_mode="r",
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=train_collate_fn,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=eval_collate_fn,
    )
    return train_ds, train_loader, val_loader

def make_test_loader(lead_indices_to_keep: List[int]) -> DataLoader:
    test_ds = ChapmanFixedLeadEvalDataset(
        signals_file=DATA_DIR / "test_signals.npy",
        labels_file=DATA_DIR / "test_labels.npy",
        metadata_file=DATA_DIR / "test_metadata.csv",
        lead_indices_to_keep=lead_indices_to_keep,
        mmap_mode="r",
    )
    return DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=eval_collate_fn,
    )

class BasicBlock1D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        kernel_size: int = 7,
        dropout: float = 0.0,
    ):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(identity)
        out = out + identity
        out = self.relu(out)
        return out

class ResNet1DEncoder(nn.Module):
    def __init__(
        self,
        input_channels: int = INPUT_CHANNELS,
        base_filters: int = BASE_FILTERS,
        block_counts: List[int] = RESNET_BLOCKS,
        embedding_dim: int = EMBED_DIM,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(input_channels, base_filters, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        channel_plan = [base_filters, base_filters * 2, base_filters * 4, base_filters * 8]
        in_channels = base_filters
        stages = []
        for stage_idx, (out_channels, n_blocks) in enumerate(zip(channel_plan, block_counts)):
            stride = 1 if stage_idx == 0 else 2
            stage, in_channels = self._make_stage(in_channels, out_channels, n_blocks, stride, dropout)
            stages.append(stage)
        self.backbone = nn.Sequential(*stages)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )
        self.embedding_dim = embedding_dim
        self._init_weights()

    @staticmethod
    def _make_stage(
        in_channels: int,
        out_channels: int,
        n_blocks: int,
        first_stride: int,
        dropout: float,
    ) -> Tuple[nn.Sequential, int]:
        blocks = [BasicBlock1D(in_channels, out_channels, stride=first_stride, kernel_size=7, dropout=dropout)]
        for _ in range(1, n_blocks):
            blocks.append(BasicBlock1D(out_channels, out_channels, stride=1, kernel_size=7, dropout=dropout))
        return nn.Sequential(*blocks), out_channels

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.backbone(x)
        x = self.global_pool(x)
        x = self.embedding_head(x)
        return x

class ECGClassifier(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.encoder = ResNet1DEncoder()
        self.classifier = nn.Linear(self.encoder.embedding_dim, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.encoder(x)
        logits = self.classifier(emb)
        return logits

def metrics_from_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
    metrics: Dict[str, Any] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0,
    )
    metrics["per_class_precision"] = {CLASS_NAMES[i]: float(precision[i]) for i in range(NUM_CLASSES)}
    metrics["per_class_recall"] = {CLASS_NAMES[i]: float(recall[i]) for i in range(NUM_CLASSES)}
    metrics["per_class_f1"] = {CLASS_NAMES[i]: float(f1[i]) for i in range(NUM_CLASSES)}
    metrics["per_class_support"] = {CLASS_NAMES[i]: int(support[i]) for i in range(NUM_CLASSES)}
    metrics["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    return metrics

def summarize_aug_batch(aug_meta_list: List[Dict[str, Any]]) -> Dict[str, int]:
    summary = {
        "masked": 0,
        "unmasked": 0,
        "clinical_subset": 0,
    }
    for meta in aug_meta_list:
        if meta["lead_mask_applied"]:
            summary["masked"] += 1
            summary["clinical_subset"] += 1
        else:
            summary["unmasked"] += 1
    return summary

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
) -> Dict[str, Any]:
    model.train()
    total_loss = 0.0
    total_items = 0
    all_true: List[int] = []
    all_pred: List[int] = []
    aug_totals = {"masked": 0, "unmasked": 0, "clinical_subset": 0}

    for batch in tqdm(loader, desc="Training", leave=False):
        x = batch["signal"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        batch_aug = summarize_aug_batch(batch["aug_meta"])
        for k, v in batch_aug.items():
            aug_totals[k] += int(v)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss encountered: {loss.item()}")

        scaler.scale(loss).backward()
        if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits, dim=1)
        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        total_items += bs
        all_true.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())

    y_true = np.asarray(all_true, dtype=np.int64)
    y_pred = np.asarray(all_pred, dtype=np.int64)
    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_items)
    metrics["aug_totals"] = aug_totals
    return metrics

def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    criterion: Optional[nn.Module] = None,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_items = 0

    all_true: List[int] = []
    all_pred: List[int] = []
    all_prob: List[np.ndarray] = []
    all_record_idx: List[int] = []
    all_file_name: List[str] = []
    all_patient_id: List[str] = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            x = batch["signal"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)

            with autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y) if criterion is not None else None

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            bs = y.size(0)
            if loss is not None:
                total_loss += float(loss.item()) * bs
            total_items += bs

            all_true.extend(y.detach().cpu().numpy().tolist())
            all_pred.extend(preds.detach().cpu().numpy().tolist())
            all_prob.extend(probs.detach().cpu().numpy())
            all_record_idx.extend(batch["record_idx"].cpu().numpy().tolist())
            all_file_name.extend(batch["file_name"])
            all_patient_id.extend(batch["patient_id"])

    y_true = np.asarray(all_true, dtype=np.int64)
    y_pred = np.asarray(all_pred, dtype=np.int64)
    probs_arr = np.asarray(all_prob, dtype=np.float32)

    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_items) if criterion is not None else float("nan")

    pred_df = pd.DataFrame({
        "record_idx": all_record_idx,
        "file_name": all_file_name,
        "patient_id": all_patient_id,
        "y_true": y_true,
        "y_true_name": [CLASS_NAMES[i] for i in y_true],
        "y_pred": y_pred,
        "y_pred_name": [CLASS_NAMES[i] for i in y_pred],
    })
    for i, cls in enumerate(CLASS_NAMES):
        pred_df[f"prob_{cls}"] = probs_arr[:, i]

    return metrics, pred_df

def set_warmup_cosine_lr(
    optimizer: torch.optim.Optimizer,
    epoch: int,
    total_epochs: int,
    base_lr: float,
    warmup_epochs: int,
    min_lr_ratio: float,
) -> float:
    if epoch < warmup_epochs:
        lr = base_lr * float(epoch + 1) / float(max(1, warmup_epochs))
    else:
        progress = (epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr = base_lr * (min_lr_ratio + (1.0 - min_lr_ratio) * cosine)
    for group in optimizer.param_groups:
        group["lr"] = lr
    return lr

def save_history_csv(history: List[Dict[str, Any]], path: Path) -> None:
    if not history:
        return
    fields = list(history[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(history)

def save_checkpoint(
    path: Path,
    model: nn.Module,
    epoch: int,
    best_val_macro_f1: float,
    variant: Dict[str, Any],
) -> None:
    torch.save({
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "encoder_state_dict": model.encoder.state_dict(),
        "classifier_state_dict": model.classifier.state_dict(),
        "best_val_macro_f1": float(best_val_macro_f1),
        "variant": variant,
        "class_names": CLASS_NAMES,
    }, path)

def load_checkpoint(path: Path, model: nn.Module) -> Dict[str, Any]:
    ckpt = torch.load(path, map_location="cpu")
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    if missing or unexpected:
        raise RuntimeError(f"Checkpoint mismatch. Missing={missing}, Unexpected={unexpected}")
    return ckpt

def save_eval_artifacts(
    model_key: str,
    lead_key: str,
    metrics: Dict[str, Any],
    pred_df: pd.DataFrame,
) -> None:
    pred_df.to_csv(OUT_DIR / f"predictions_{model_key}_{lead_key}.csv", index=False)
    np.save(OUT_DIR / f"confusion_{model_key}_{lead_key}.npy", metrics["confusion_matrix"])
    report = classification_report(
        pred_df["y_true"].to_numpy(),
        pred_df["y_pred"].to_numpy(),
        labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
    with open(OUT_DIR / f"classification_report_{model_key}_{lead_key}.txt", "w", encoding="utf-8") as f:
        f.write(report)

def train_variant(variant: Dict[str, Any]) -> Dict[str, Any]:
    model_key = variant["model_key"]
    display_name = variant["display_name"]

    print("=" * 150)
    print(f"TRAINING CHAPMAN VARIANT: {display_name}")
    print("=" * 150)
    print(f"Clinical lead masking during training: {variant['use_clinical_lead_masking']}")
    print(f"Lead mask probability                 : {variant['lead_mask_prob']}")
    print("Loss                                  : Plain CrossEntropyLoss")
    print("Validation selection metric           : Full 12-lead validation Macro-F1")
    print("=" * 150)

    seed_everything(SEED)
    train_ds, train_loader, val_loader = make_train_val_loaders(variant)

    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(device="cuda", enabled=USE_AMP)

    history: List[Dict[str, Any]] = []
    best_val_macro_f1 = -1.0
    best_epoch = -1
    patience = 0
    start = time.time()
    best_ckpt_path = OUT_DIR / f"best_{model_key}.pt"

    for epoch in range(EPOCHS):
        lr = set_warmup_cosine_lr(
            optimizer=optimizer,
            epoch=epoch,
            total_epochs=EPOCHS,
            base_lr=BASE_LR,
            warmup_epochs=WARMUP_EPOCHS,
            min_lr_ratio=MIN_LR_RATIO,
        )

        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_metrics, _ = evaluate_model(model, val_loader, criterion)

        aug_totals = train_metrics["aug_totals"]
        total_seen = aug_totals["masked"] + aug_totals["unmasked"]
        observed_mask_rate = aug_totals["masked"] / max(1, total_seen)

        row = {
            "epoch": epoch + 1,
            "lr": lr,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "observed_train_mask_rate": observed_mask_rate,
            "observed_train_masked_count": aug_totals["masked"],
            "observed_train_unmasked_count": aug_totals["unmasked"],
        }
        history.append(row)

        print(
            f"{model_key} | Epoch {epoch + 1:03d}/{EPOCHS} | lr={lr:.3e} | "
            f"train: acc={row['train_accuracy']:.4f}, macro_f1={row['train_macro_f1']:.4f}, mask_rate={observed_mask_rate:.3f} | "
            f"val-full: acc={row['val_accuracy']:.4f}, bal_acc={row['val_balanced_accuracy']:.4f}, macro_f1={row['val_macro_f1']:.4f}"
        )

        if row["val_macro_f1"] > best_val_macro_f1 + 1e-6:
            best_val_macro_f1 = row["val_macro_f1"]
            best_epoch = epoch + 1
            patience = 0
            save_checkpoint(best_ckpt_path, model, epoch + 1, best_val_macro_f1, variant)
        else:
            patience += 1

        save_history_csv(history, OUT_DIR / f"history_{model_key}.csv")

        if patience >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping {model_key} at epoch {epoch + 1}; best epoch={best_epoch}.")
            break

    runtime_minutes = (time.time() - start) / 60.0

    return {
        "model_key": model_key,
        "display_name": display_name,
        "best_checkpoint": str(best_ckpt_path),
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_macro_f1),
        "runtime_minutes": float(runtime_minutes),
        "use_clinical_lead_masking": bool(variant["use_clinical_lead_masking"]),
        "lead_mask_prob": float(variant["lead_mask_prob"]),
    }

def evaluate_variant_grid(training_result: Dict[str, Any]) -> List[Dict[str, Any]]:
    model_key = training_result["model_key"]
    display_name = training_result["display_name"]
    ckpt_path = Path(training_result["best_checkpoint"])

    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    load_checkpoint(ckpt_path, model)
    model.eval()

    rows: List[Dict[str, Any]] = []
    print("\n" + "=" * 150)
    print(f"CHAPMAN REDUCED-LEAD TEST GRID: {display_name}")
    print("=" * 150)

    for lead_key, lead_info in LEAD_CONFIGS.items():
        lead_display = lead_info["display_name"]
        lead_indices = lead_info["lead_indices"]
        kept_names = [LEAD_NAMES[i] for i in lead_indices]

        print(f"Evaluating {display_name} | {lead_display} | kept={kept_names}")
        test_loader = make_test_loader(lead_indices)
        metrics, pred_df = evaluate_model(model, test_loader, criterion=None)
        save_eval_artifacts(model_key, lead_key, metrics, pred_df)

        print(
            f"  ACC={metrics['accuracy']:.4f} | "
            f"Bal-ACC={metrics['balanced_accuracy']:.4f} | "
            f"Macro-F1={metrics['macro_f1']:.4f} | "
            f"Weighted-F1={metrics['weighted_f1']:.4f}"
        )

        rows.append({
            "model_key": model_key,
            "model_display_name": display_name,
            "lead_key": lead_key,
            "lead_display_name": lead_display,
            "kept_leads": ",".join(kept_names),
            "n_kept_leads": int(len(lead_indices)),
            "accuracy": float(metrics["accuracy"]),
            "balanced_accuracy": float(metrics["balanced_accuracy"]),
            "macro_f1": float(metrics["macro_f1"]),
            "weighted_f1": float(metrics["weighted_f1"]),
            "per_class_recall_json": json.dumps(metrics["per_class_recall"]),
            "per_class_f1_json": json.dumps(metrics["per_class_f1"]),
        })

    return rows

def finalize_results(training_results: List[Dict[str, Any]], eval_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    training_df = pd.DataFrame(training_results)
    results_df = pd.DataFrame(eval_rows)

    full_df = results_df[results_df["lead_key"] == "12_lead_full"].copy()
    full_acc_map = dict(zip(full_df["model_key"], full_df["accuracy"]))
    full_macro_map = dict(zip(full_df["model_key"], full_df["macro_f1"]))
    full_bal_map = dict(zip(full_df["model_key"], full_df["balanced_accuracy"]))

    results_df["accuracy_retention_vs_own_12lead"] = results_df.apply(
        lambda r: float(r["accuracy"] / full_acc_map[r["model_key"]]) if full_acc_map[r["model_key"]] > 0 else np.nan,
        axis=1,
    )
    results_df["macro_f1_retention_vs_own_12lead"] = results_df.apply(
        lambda r: float(r["macro_f1"] / full_macro_map[r["model_key"]]) if full_macro_map[r["model_key"]] > 0 else np.nan,
        axis=1,
    )
    results_df["balanced_accuracy_retention_vs_own_12lead"] = results_df.apply(
        lambda r: float(r["balanced_accuracy"] / full_bal_map[r["model_key"]]) if full_bal_map[r["model_key"]] > 0 else np.nan,
        axis=1,
    )

    training_df.to_csv(OUT_DIR / "chapman_external_validation_training_results.csv", index=False)
    results_df.to_csv(OUT_DIR / "chapman_external_validation_reduced_lead_results.csv", index=False)

    lead_order = [v["display_name"] for v in LEAD_CONFIGS.values()]
    accuracy_wide = results_df.pivot(index="lead_display_name", columns="model_display_name", values="accuracy").reindex(lead_order)
    macro_wide = results_df.pivot(index="lead_display_name", columns="model_display_name", values="macro_f1").reindex(lead_order)
    acc_ret_wide = results_df.pivot(index="lead_display_name", columns="model_display_name", values="accuracy_retention_vs_own_12lead").reindex(lead_order)
    macro_ret_wide = results_df.pivot(index="lead_display_name", columns="model_display_name", values="macro_f1_retention_vs_own_12lead").reindex(lead_order)

    accuracy_wide.to_csv(OUT_DIR / "chapman_external_accuracy_wide.csv")
    macro_wide.to_csv(OUT_DIR / "chapman_external_macro_f1_wide.csv")
    acc_ret_wide.to_csv(OUT_DIR / "chapman_external_accuracy_retention_wide.csv")
    macro_ret_wide.to_csv(OUT_DIR / "chapman_external_macro_f1_retention_wide.csv")

    baseline_key = "chapman_standard_supervised"
    masked_key = "chapman_clinical_masked_p060"
    gain_lines: List[str] = []
    for lead_key, lead_info in LEAD_CONFIGS.items():
        base_row = results_df[(results_df["model_key"] == baseline_key) & (results_df["lead_key"] == lead_key)].iloc[0]
        mask_row = results_df[(results_df["model_key"] == masked_key) & (results_df["lead_key"] == lead_key)].iloc[0]
        gain_lines.append(
            f"  {lead_info['display_name']:<32s} | "
            f"ACC gain={mask_row['accuracy'] - base_row['accuracy']:+.6f} | "
            f"Macro-F1 gain={mask_row['macro_f1'] - base_row['macro_f1']:+.6f}"
        )

    lines: List[str] = []
    lines.append("=" * 165)
    lines.append("STEP 14 — CHAPMAN EXTERNAL VALIDATION SUMMARY")
    lines.append("=" * 165)
    lines.append("External dataset: Chapman raw ECGData, corrected 4-class grouped rhythm task")
    lines.append("Classes: SB, AFIB, GSVT, SR")
    lines.append("Comparison: Standard supervised vs Clinical lead-masked p=0.60 supervised training")
    lines.append("")
    lines.append("TRAINING RESULTS")
    for _, row in training_df.iterrows():
        lines.append(
            f"  {row['display_name']:<42s} | best_epoch={int(row['best_epoch']):<3d} | "
            f"best_val_macro_f1={row['best_val_macro_f1']:.6f} | runtime={row['runtime_minutes']:.2f} min"
        )
    lines.append("")
    lines.append("ABSOLUTE ACCURACY")
    lines.append(accuracy_wide.to_string(float_format=lambda x: f"{x:.6f}"))
    lines.append("")
    lines.append("ABSOLUTE MACRO-F1")
    lines.append(macro_wide.to_string(float_format=lambda x: f"{x:.6f}"))
    lines.append("")
    lines.append("ACCURACY RETENTION VS OWN 12-LEAD")
    lines.append(acc_ret_wide.to_string(float_format=lambda x: f"{x:.6f}"))
    lines.append("")
    lines.append("MACRO-F1 RETENTION VS OWN 12-LEAD")
    lines.append(macro_ret_wide.to_string(float_format=lambda x: f"{x:.6f}"))
    lines.append("")
    lines.append("CLINICAL LEAD-MASKED p=0.60 MINUS STANDARD SUPERVISED GAINS")
    lines.extend(gain_lines)
    lines.append("")
    lines.append(f"Artifacts saved in: {OUT_DIR}")
    lines.append("=" * 165)

    summary_text = "\n".join(lines)
    print("\n" + summary_text)
    with open(OUT_DIR / "chapman_external_validation_summary.txt", "w", encoding="utf-8") as f:
        f.write(summary_text)

    return results_df

def main() -> None:
    print("=" * 165)
    print("STEP 14 — CHAPMAN EXTERNAL VALIDATION TRAINING + REDUCED-LEAD EVALUATION")
    print("=" * 165)
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Preprocessed Chapman data: {DATA_DIR}")
    print(f"Output directory: {OUT_DIR}")
    print(f"Models to train: {[v['model_key'] for v in MODEL_VARIANTS]}")
    print("=" * 165)

    verify_required_preprocessed_files()

    training_results: List[Dict[str, Any]] = []
    all_eval_rows: List[Dict[str, Any]] = []

    for variant in MODEL_VARIANTS:
        tr = train_variant(variant)
        training_results.append(tr)
        eval_rows = evaluate_variant_grid(tr)
        all_eval_rows.extend(eval_rows)

    finalize_results(training_results, all_eval_rows)

if __name__ == "__main__":
    main()


04_chapman_random_training_corrected

In [ ]:
from __future__ import annotations

import csv
import gc
import json
import math
import random
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

                                                                                                            
          
                                                                                                            

PROJECT_ROOT = Path("your_project_root")

DATA_DIR = PROJECT_ROOT / "processed_chapman_4class_raw_ecgdata"

OUT_DIR = (
    PROJECT_ROOT
    / "lead_masking_final"
    / "step21c_chapman_random_only_p060_step14_protocol_matched"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT_PATH = OUT_DIR / "best_chapman_random_only_p060_step14_protocol_matched.pt"
LATEST_CKPT_PATH = OUT_DIR / "latest_chapman_random_only_p060_step14_protocol_matched.pt"
HISTORY_CSV = OUT_DIR / "history_chapman_random_only_p060_step14_protocol_matched.csv"
TRAINING_RESULTS_CSV = OUT_DIR / "chapman_random_only_p060_training_results.csv"
REDUCED_LEAD_RESULTS_CSV = OUT_DIR / "chapman_random_only_p060_reduced_lead_results.csv"
MASK_CARDINALITY_HISTORY_CSV = OUT_DIR / "chapman_random_only_p060_mask_cardinality_by_epoch.csv"
SUMMARY_TXT = OUT_DIR / "chapman_random_only_p060_protocol_matched_summary.txt"
CONFIG_JSON = OUT_DIR / "chapman_random_only_p060_protocol_matched_config.json"

                                                                   
CHAPMAN_STANDARD_REFERENCE_CKPT = (
    DATA_DIR
    / "chapman_external_validation_step14"
    / "best_chapman_standard_supervised.pt"
)

CHAPMAN_CLINICAL_REFERENCE_CKPT = (
    DATA_DIR
    / "chapman_external_validation_step14"
    / "best_chapman_clinical_masked_p060.pt"
)

                                                                                                            
                               
                                                                                                            

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

                                                                                                            
                         
                                                                                                            

CLASS_NAMES = ["SB", "AFIB", "GSVT", "SR"]
NUM_CLASSES = len(CLASS_NAMES)

N_LEADS = 12
TARGET_LENGTH = 5000
INPUT_CHANNELS = 12

LEAD_NAMES = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

                                            
LEAD_CONFIGS = {
    "12_lead_full": {
        "display_name": "12-lead full",
        "lead_indices": list(range(12)),
    },
    "6_limb": {
        "display_name": "6 limb leads",
        "lead_indices": [0, 1, 2, 3, 4, 5],
    },
    "6_precordial": {
        "display_name": "6 precordial leads",
        "lead_indices": [6, 7, 8, 9, 10, 11],
    },
    "3_limb_I_II_III": {
        "display_name": "3 limb leads (I, II, III)",
        "lead_indices": [0, 1, 2],
    },
    "lead_II_only": {
        "display_name": "Lead II only",
        "lead_indices": [1],
    },
    "V5_only": {
        "display_name": "V5 only",
        "lead_indices": [10],
    },
}

                                                                                                            
                                              
                                                                     
                                                                                                            

BATCH_SIZE = 128
EPOCHS = 70
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
MIN_LR_RATIO = 0.02
GRAD_CLIP_NORM = 5.0
EARLY_STOPPING_PATIENCE = 15

                                                   
BASE_FILTERS = 64
RESNET_BLOCKS = [2, 2, 2, 2]
EMBED_DIM = 512
DROPOUT = 0.10

                                                                
RANDOM_MASK_PROB = 0.60
MIN_RANDOM_KEPT_LEADS = 1
MAX_RANDOM_KEPT_LEADS = 12

RESUME_IF_AVAILABLE = True

                                                                                                            
                                 
                                                                                                            

EXPECTED_PROTOCOL = {
    "BATCH_SIZE": 128,
    "EPOCHS": 70,
    "BASE_LR": 3e-4,
    "WEIGHT_DECAY": 1e-4,
    "WARMUP_EPOCHS": 3,
    "MIN_LR_RATIO": 0.02,
    "GRAD_CLIP_NORM": 5.0,
    "EARLY_STOPPING_PATIENCE": 15,
    "DROPOUT": 0.10,
    "LOSS": "Plain CrossEntropyLoss",
    "TRAIN_LOADER": "shuffle=True",
    "RANDOM_POLICY": "k uniform over 1..12 retained leads",
    "RANDOM_MASK_PROB": 0.60,
}

def run_protocol_lock_check() -> None:
    checks = {
        "BATCH_SIZE": BATCH_SIZE,
        "EPOCHS": EPOCHS,
        "BASE_LR": BASE_LR,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "WARMUP_EPOCHS": WARMUP_EPOCHS,
        "MIN_LR_RATIO": MIN_LR_RATIO,
        "GRAD_CLIP_NORM": GRAD_CLIP_NORM,
        "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
        "DROPOUT": DROPOUT,
        "RANDOM_MASK_PROB": RANDOM_MASK_PROB,
    }

    expected_numeric = {
        k: v for k, v in EXPECTED_PROTOCOL.items()
        if isinstance(v, (int, float))
    }

    mismatches = []
    for key, expected in expected_numeric.items():
        got = checks[key]
        if float(got) != float(expected):
            mismatches.append(f"{key}: expected {expected}, got {got}")

    if mismatches:
        raise RuntimeError(
            "Protocol lock failed. Hyperparameters do not match the required Step 14 setup:\n"
            + "\n".join(mismatches)
        )

    print("Protocol lock check passed ")
    print("  - Hyperparameters match Chapman Step 14 Standard/Clinical.")
    print("  - Random policy matches PTB-XL Random-only definition.")
    print("  - Loss = Plain CrossEntropyLoss.")
    print("  - Train loader = shuffle=True.")

                                                                                                            
                        
                                                                                                            

def require_exists(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")

def verify_required_files() -> None:
    required_files = [
        DATA_DIR / "train_signals.npy",
        DATA_DIR / "val_signals.npy",
        DATA_DIR / "test_signals.npy",
        DATA_DIR / "train_labels.npy",
        DATA_DIR / "val_labels.npy",
        DATA_DIR / "test_labels.npy",
        DATA_DIR / "train_metadata.csv",
        DATA_DIR / "val_metadata.csv",
        DATA_DIR / "test_metadata.csv",
        CHAPMAN_STANDARD_REFERENCE_CKPT,
        CHAPMAN_CLINICAL_REFERENCE_CKPT,
    ]

    missing = [str(p) for p in required_files if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required files:\n" + "\n".join(missing)
        )

                                                                                                            
                               
                                                   
                                                                                                            

class RandomLeadMaskAugment:

    def __init__(
        self,
        lead_mask_prob: float = RANDOM_MASK_PROB,
        min_random_kept_leads: int = MIN_RANDOM_KEPT_LEADS,
        max_random_kept_leads: int = MAX_RANDOM_KEPT_LEADS,
    ):
        self.lead_mask_prob = float(lead_mask_prob)
        self.min_random_kept_leads = int(min_random_kept_leads)
        self.max_random_kept_leads = int(max_random_kept_leads)

        if not (0.0 <= self.lead_mask_prob <= 1.0):
            raise ValueError("lead_mask_prob must be within [0,1].")
        if self.min_random_kept_leads < 1:
            raise ValueError("min_random_kept_leads must be >= 1.")
        if self.max_random_kept_leads > N_LEADS:
            raise ValueError("max_random_kept_leads must be <= 12.")
        if self.min_random_kept_leads > self.max_random_kept_leads:
            raise ValueError("Invalid kept-lead range.")

    def __call__(self, signal: np.ndarray) -> Tuple[np.ndarray, Dict[str, Any]]:
        x = np.asarray(signal, dtype=np.float32).copy()           

        if x.ndim != 2 or x.shape[1] != N_LEADS:
            raise ValueError(f"Expected ECG shape (T, {N_LEADS}), got {x.shape}")

        default_meta = {
            "lead_mask_applied": False,
            "mask_type": "none",
            "kept_leads": list(range(N_LEADS)),
            "kept_lead_count": N_LEADS,
        }

        if random.random() >= self.lead_mask_prob:
            return x, default_meta

        k = random.randint(
            self.min_random_kept_leads,
            self.max_random_kept_leads,
        )

        kept = tuple(sorted(random.sample(range(N_LEADS), k=k)))
        dropped = [i for i in range(N_LEADS) if i not in kept]

        if dropped:
            x[:, dropped] = 0.0

        meta = {
            "lead_mask_applied": True,
            "mask_type": "random_subset",
            "kept_leads": list(map(int, kept)),
            "kept_lead_count": int(len(kept)),
        }

        return x, meta

                                                                                                            
             
                                                                                                            

class ChapmanRandomTrainDataset(Dataset):

    def __init__(
        self,
        signals_file: Path,
        labels_file: Path,
        metadata_file: Path,
        lead_mask_prob: float = RANDOM_MASK_PROB,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(signals_file, mmap_mode=mmap_mode)
        self.labels = np.load(labels_file).astype(np.int64)
        self.metadata = pd.read_csv(metadata_file)
        self.augmenter = RandomLeadMaskAugment(
            lead_mask_prob=lead_mask_prob,
            min_random_kept_leads=MIN_RANDOM_KEPT_LEADS,
            max_random_kept_leads=MAX_RANDOM_KEPT_LEADS,
        )
        self._validate()

    def _validate(self) -> None:
        if len(self.signals) != len(self.labels):
            raise ValueError(
                f"Signals/labels mismatch: {len(self.signals)} vs {len(self.labels)}"
            )
        if len(self.signals) != len(self.metadata):
            raise ValueError(
                f"Signals/metadata mismatch: {len(self.signals)} vs {len(self.metadata)}"
            )
        if self.signals.ndim != 3 or self.signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
            raise ValueError(
                f"Expected train signals shape (N,{TARGET_LENGTH},{N_LEADS}), got {self.signals.shape}"
            )
        labels_seen = sorted(np.unique(self.labels).tolist())
        if labels_seen != list(range(NUM_CLASSES)):
            raise ValueError(
                f"Expected labels [0..{NUM_CLASSES-1}], got {labels_seen}"
            )

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32).copy()              
        y = int(self.labels[idx])

        x, aug_meta = self.augmenter(x)

        x = torch.from_numpy(x.T.copy()).float()             
        row = self.metadata.iloc[idx]

        return {
            "signal": x,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(row["record_idx"]) if "record_idx" in row else int(idx),
            "file_name": str(row["FileName_csv"]) if "FileName_csv" in row else "",
            "patient_id": str(row["patient_id"]) if "patient_id" in row else "",
            "aug_meta": aug_meta,
        }

class ChapmanFixedLeadEvalDataset(Dataset):

    def __init__(
        self,
        signals_file: Path,
        labels_file: Path,
        metadata_file: Path,
        lead_indices_to_keep: Optional[List[int]] = None,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(signals_file, mmap_mode=mmap_mode)
        self.labels = np.load(labels_file).astype(np.int64)
        self.metadata = pd.read_csv(metadata_file)
        self.lead_indices_to_keep = (
            list(range(N_LEADS))
            if lead_indices_to_keep is None
            else sorted(list(map(int, lead_indices_to_keep)))
        )
        self.lead_indices_to_zero = [
            i for i in range(N_LEADS)
            if i not in self.lead_indices_to_keep
        ]
        self._validate()

    def _validate(self) -> None:
        if len(self.signals) != len(self.labels):
            raise ValueError(
                f"Signals/labels mismatch: {len(self.signals)} vs {len(self.labels)}"
            )
        if len(self.signals) != len(self.metadata):
            raise ValueError(
                f"Signals/metadata mismatch: {len(self.signals)} vs {len(self.metadata)}"
            )
        if self.signals.ndim != 3 or self.signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
            raise ValueError(
                f"Expected eval signals shape (N,{TARGET_LENGTH},{N_LEADS}), got {self.signals.shape}"
            )
        if not self.lead_indices_to_keep:
            raise ValueError("At least one lead must be retained.")
        if min(self.lead_indices_to_keep) < 0 or max(self.lead_indices_to_keep) >= N_LEADS:
            raise ValueError(f"Invalid lead indices: {self.lead_indices_to_keep}")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32).copy()
        y = int(self.labels[idx])

        if self.lead_indices_to_zero:
            x[:, self.lead_indices_to_zero] = 0.0

        x = torch.from_numpy(x.T.copy()).float()

        row = self.metadata.iloc[idx]

        return {
            "signal": x,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(row["record_idx"]) if "record_idx" in row else int(idx),
            "file_name": str(row["FileName_csv"]) if "FileName_csv" in row else "",
            "patient_id": str(row["patient_id"]) if "patient_id" in row else "",
        }

                                                                                                            
                      
                                                                                                            

def train_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "signal": torch.stack([b["signal"] for b in batch], dim=0),
        "label": torch.stack([b["label"] for b in batch], dim=0),
        "record_idx": torch.tensor([b["record_idx"] for b in batch], dtype=torch.long),
        "file_name": [b["file_name"] for b in batch],
        "patient_id": [b["patient_id"] for b in batch],
        "aug_meta": [b["aug_meta"] for b in batch],
    }

def eval_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "signal": torch.stack([b["signal"] for b in batch], dim=0),
        "label": torch.stack([b["label"] for b in batch], dim=0),
        "record_idx": torch.tensor([b["record_idx"] for b in batch], dtype=torch.long),
        "file_name": [b["file_name"] for b in batch],
        "patient_id": [b["patient_id"] for b in batch],
    }

                                                                                                            
                                        
                                                                                                            

def make_train_val_loaders() -> Tuple[
    ChapmanRandomTrainDataset,
    DataLoader,
    DataLoader,
]:
    train_ds = ChapmanRandomTrainDataset(
        signals_file=DATA_DIR / "train_signals.npy",
        labels_file=DATA_DIR / "train_labels.npy",
        metadata_file=DATA_DIR / "train_metadata.csv",
        lead_mask_prob=RANDOM_MASK_PROB,
        mmap_mode="r",
    )

    val_ds = ChapmanFixedLeadEvalDataset(
        signals_file=DATA_DIR / "val_signals.npy",
        labels_file=DATA_DIR / "val_labels.npy",
        metadata_file=DATA_DIR / "val_metadata.csv",
        lead_indices_to_keep=list(range(N_LEADS)),
        mmap_mode="r",
    )

                                                                   
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=train_collate_fn,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=eval_collate_fn,
    )

    return train_ds, train_loader, val_loader

def make_test_loader(lead_indices_to_keep: List[int]) -> DataLoader:
    test_ds = ChapmanFixedLeadEvalDataset(
        signals_file=DATA_DIR / "test_signals.npy",
        labels_file=DATA_DIR / "test_labels.npy",
        metadata_file=DATA_DIR / "test_metadata.csv",
        lead_indices_to_keep=lead_indices_to_keep,
        mmap_mode="r",
    )

    return DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=eval_collate_fn,
    )

                                                                                                            
                                                       
                                                                                                            

class BasicBlock1D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        kernel_size: int = 7,
        dropout: float = 0.0,
    ):
        super().__init__()

        padding = kernel_size // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            bias=False,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(identity)

        out = out + identity
        out = self.relu(out)

        return out

class ResNet1DEncoder(nn.Module):
    def __init__(
        self,
        input_channels: int = INPUT_CHANNELS,
        base_filters: int = BASE_FILTERS,
        block_counts: List[int] = RESNET_BLOCKS,
        embedding_dim: int = EMBED_DIM,
        dropout: float = DROPOUT,
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                input_channels,
                base_filters,
                kernel_size=15,
                stride=2,
                padding=7,
                bias=False,
            ),
            nn.BatchNorm1d(base_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )

        channel_plan = [
            base_filters,
            base_filters * 2,
            base_filters * 4,
            base_filters * 8,
        ]

        in_channels = base_filters
        stages = []

        for stage_idx, (out_channels, n_blocks) in enumerate(
            zip(channel_plan, block_counts)
        ):
            stride = 1 if stage_idx == 0 else 2
            stage, in_channels = self._make_stage(
                in_channels,
                out_channels,
                n_blocks,
                stride,
                dropout,
            )
            stages.append(stage)

        self.backbone = nn.Sequential(*stages)
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )

        self.embedding_dim = embedding_dim
        self._init_weights()

    @staticmethod
    def _make_stage(
        in_channels: int,
        out_channels: int,
        n_blocks: int,
        first_stride: int,
        dropout: float,
    ) -> Tuple[nn.Sequential, int]:
        blocks = [
            BasicBlock1D(
                in_channels,
                out_channels,
                stride=first_stride,
                kernel_size=7,
                dropout=dropout,
            )
        ]

        for _ in range(1, n_blocks):
            blocks.append(
                BasicBlock1D(
                    out_channels,
                    out_channels,
                    stride=1,
                    kernel_size=7,
                    dropout=dropout,
                )
            )

        return nn.Sequential(*blocks), out_channels

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(
                    m.weight,
                    mode="fan_out",
                    nonlinearity="relu",
                )
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.backbone(x)
        x = self.global_pool(x)
        x = self.embedding_head(x)
        return x

class ECGClassifier(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.encoder = ResNet1DEncoder()
        self.classifier = nn.Linear(
            self.encoder.embedding_dim,
            num_classes,
        )

        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.encoder(x)
        logits = self.classifier(emb)
        return logits

                                                                                                            
                                   
                                                                                                            

def extract_state_dict_from_checkpoint(ckpt_obj: Any) -> Dict[str, torch.Tensor]:
    if isinstance(ckpt_obj, dict):
        if "model_state_dict" in ckpt_obj:
            return ckpt_obj["model_state_dict"]

        for key in ["state_dict", "model", "net"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                return ckpt_obj[key]

        if all(isinstance(k, str) for k in ckpt_obj.keys()):
            return ckpt_obj

    raise ValueError("Unable to locate model state dict in checkpoint.")

def strict_architecture_preflight() -> Dict[str, Any]:
    results = {}

    for name, ckpt_path in {
        "Chapman_Standard": CHAPMAN_STANDARD_REFERENCE_CKPT,
        "Chapman_Clinical": CHAPMAN_CLINICAL_REFERENCE_CKPT,
    }.items():
        ckpt = torch.load(ckpt_path, map_location="cpu")
        state = extract_state_dict_from_checkpoint(ckpt)

        model = ECGClassifier(num_classes=NUM_CLASSES)

        missing, unexpected = model.load_state_dict(
            state,
            strict=False,
        )

        if missing or unexpected:
            raise RuntimeError(
                f"Strict architecture preflight failed for {name}.\n"
                f"Missing keys: {missing}\n"
                f"Unexpected keys: {unexpected}"
            )

        model.load_state_dict(state, strict=True)

        results[name] = {
            "checkpoint": str(ckpt_path),
            "strict_load_passed": True,
            "num_state_keys": int(len(state)),
        }

    return results

                                                                                                            
             
                                                                                                            

def metrics_from_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, Any]:
    metrics: Dict[str, Any] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "weighted_f1": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
    }

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0,
    )

    metrics["per_class_precision"] = {
        CLASS_NAMES[i]: float(precision[i])
        for i in range(NUM_CLASSES)
    }
    metrics["per_class_recall"] = {
        CLASS_NAMES[i]: float(recall[i])
        for i in range(NUM_CLASSES)
    }
    metrics["per_class_f1"] = {
        CLASS_NAMES[i]: float(f1[i])
        for i in range(NUM_CLASSES)
    }
    metrics["per_class_support"] = {
        CLASS_NAMES[i]: int(support[i])
        for i in range(NUM_CLASSES)
    }
    metrics["confusion_matrix"] = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
    )

    return metrics

def summarize_aug_batch(
    aug_meta_list: List[Dict[str, Any]],
) -> Dict[str, Any]:
    summary = {
        "masked": 0,
        "unmasked": 0,
        "random_subset": 0,
        "kept_count_distribution": {str(k): 0 for k in range(1, 13)},
    }

    for meta in aug_meta_list:
        if meta["lead_mask_applied"]:
            summary["masked"] += 1
            summary["random_subset"] += 1
            k = int(meta["kept_lead_count"])
            summary["kept_count_distribution"][str(k)] += 1
        else:
            summary["unmasked"] += 1

    return summary

                                                                                                            
                                     
                                                                                                            

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
) -> Dict[str, Any]:
    model.train()

    total_loss = 0.0
    total_items = 0
    all_true: List[int] = []
    all_pred: List[int] = []

    aug_totals = {
        "masked": 0,
        "unmasked": 0,
        "random_subset": 0,
        "kept_count_distribution": {str(k): 0 for k in range(1, 13)},
    }

    for batch in tqdm(loader, desc="Training", leave=False):
        x = batch["signal"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        batch_aug = summarize_aug_batch(batch["aug_meta"])
        aug_totals["masked"] += int(batch_aug["masked"])
        aug_totals["unmasked"] += int(batch_aug["unmasked"])
        aug_totals["random_subset"] += int(batch_aug["random_subset"])

        for k in range(1, 13):
            aug_totals["kept_count_distribution"][str(k)] += int(
                batch_aug["kept_count_distribution"][str(k)]
            )

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite training loss encountered: {loss.item()}"
            )

        scaler.scale(loss).backward()

        if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_NORM,
            )

        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits, dim=1)

        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        total_items += bs

        all_true.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())

    y_true = np.asarray(all_true, dtype=np.int64)
    y_pred = np.asarray(all_pred, dtype=np.int64)

    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_items)
    metrics["aug_totals"] = aug_totals

    return metrics

def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    criterion: Optional[nn.Module] = None,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    model.eval()

    total_loss = 0.0
    total_items = 0

    all_true: List[int] = []
    all_pred: List[int] = []
    all_prob: List[np.ndarray] = []
    all_record_idx: List[int] = []
    all_file_name: List[str] = []
    all_patient_id: List[str] = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            x = batch["signal"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)

            with autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y) if criterion is not None else None

            probs = torch.softmax(logits.float(), dim=1)
            preds = torch.argmax(probs, dim=1)

            bs = y.size(0)

            if loss is not None:
                total_loss += float(loss.item()) * bs

            total_items += bs

            all_true.extend(y.detach().cpu().numpy().tolist())
            all_pred.extend(preds.detach().cpu().numpy().tolist())
            all_prob.extend(probs.detach().cpu().numpy())
            all_record_idx.extend(batch["record_idx"].cpu().numpy().tolist())
            all_file_name.extend(batch["file_name"])
            all_patient_id.extend(batch["patient_id"])

    y_true = np.asarray(all_true, dtype=np.int64)
    y_pred = np.asarray(all_pred, dtype=np.int64)
    probs_arr = np.asarray(all_prob, dtype=np.float32)

    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = (
        total_loss / max(1, total_items)
        if criterion is not None
        else float("nan")
    )

    pred_df = pd.DataFrame({
        "record_idx": all_record_idx,
        "file_name": all_file_name,
        "patient_id": all_patient_id,
        "y_true": y_true,
        "y_true_name": [CLASS_NAMES[i] for i in y_true],
        "y_pred": y_pred,
        "y_pred_name": [CLASS_NAMES[i] for i in y_pred],
    })

    for i, cls in enumerate(CLASS_NAMES):
        pred_df[f"prob_{cls}"] = probs_arr[:, i]

    return metrics, pred_df

                                                                                                            
                                
                                                                                                            

def set_warmup_cosine_lr(
    optimizer: torch.optim.Optimizer,
    epoch: int,
    total_epochs: int,
    base_lr: float,
    warmup_epochs: int,
    min_lr_ratio: float,
) -> float:
    if epoch < warmup_epochs:
        lr = base_lr * float(epoch + 1) / float(max(1, warmup_epochs))
    else:
        progress = (
            epoch - warmup_epochs
        ) / float(max(1, total_epochs - warmup_epochs))

        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr = base_lr * (
            min_lr_ratio + (1.0 - min_lr_ratio) * cosine
        )

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr

def save_history_csv(
    history: List[Dict[str, Any]],
    path: Path,
) -> None:
    if not history:
        return

    fields = list(history[0].keys())

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(history)

def save_checkpoint(
    path: Path,
    model: nn.Module,
    epoch: int,
    best_val_macro_f1: float,
    variant: Dict[str, Any],
    history: List[Dict[str, Any]],
) -> None:
    torch.save({
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "encoder_state_dict": model.encoder.state_dict(),
        "classifier_state_dict": model.classifier.state_dict(),
        "best_val_macro_f1": float(best_val_macro_f1),
        "variant": variant,
        "class_names": CLASS_NAMES,
        "history": history,
    }, path)

def load_checkpoint(
    path: Path,
    model: nn.Module,
) -> Dict[str, Any]:
    ckpt = torch.load(path, map_location="cpu")
    missing, unexpected = model.load_state_dict(
        ckpt["model_state_dict"],
        strict=False,
    )

    if missing or unexpected:
        raise RuntimeError(
            f"Checkpoint mismatch. Missing={missing}, Unexpected={unexpected}"
        )

    return ckpt

                                                                                                            
                                
                                                                                                            

def save_eval_artifacts(
    model_key: str,
    lead_key: str,
    metrics: Dict[str, Any],
    pred_df: pd.DataFrame,
) -> None:
    pred_df.to_csv(
        OUT_DIR / f"predictions_{model_key}_{lead_key}.csv",
        index=False,
    )

    np.save(
        OUT_DIR / f"confusion_{model_key}_{lead_key}.npy",
        metrics["confusion_matrix"],
    )

    report = classification_report(
        pred_df["y_true"].to_numpy(),
        pred_df["y_pred"].to_numpy(),
        labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    with open(
        OUT_DIR / f"classification_report_{model_key}_{lead_key}.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(report)

                                                                                                            
                                   
                                                                                                            

def save_mask_cardinality_history(
    cardinality_rows: List[Dict[str, Any]],
) -> None:
    if cardinality_rows:
        pd.DataFrame(cardinality_rows).to_csv(
            MASK_CARDINALITY_HISTORY_CSV,
            index=False,
        )

                                                                                                            
                   
                                                                                                            

def train_chapman_random_variant() -> Dict[str, Any]:
    model_key = "chapman_random_only_p060_step14_protocol_matched"
    display_name = "Chapman Random-Only p=0.60 — Step 14 Protocol-Matched"

    print("=" * 170)
    print(f"TRAINING VARIANT: {display_name}")
    print("=" * 170)
    print("Random lead masking during training : True")
    print(f"Lead mask probability               : {RANDOM_MASK_PROB}")
    print("Random retained-lead policy         : k ~ Uniform{1,...,12}")
    print("Loss                                : Plain CrossEntropyLoss")
    print("Training loader                     : shuffle=True")
    print("Validation selection metric         : Full 12-lead validation Macro-F1")
    print("=" * 170)

    train_ds, train_loader, val_loader = make_train_val_loaders()

    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)

                                   
    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY,
    )

    scaler = GradScaler(
        device="cuda",
        enabled=USE_AMP,
    )

    history: List[Dict[str, Any]] = []
    cardinality_rows: List[Dict[str, Any]] = []

    best_val_macro_f1 = -1.0
    best_epoch = -1
    patience = 0
    start_epoch = 0

                     
    if RESUME_IF_AVAILABLE and LATEST_CKPT_PATH.exists():
        ckpt = load_checkpoint(LATEST_CKPT_PATH, model)
        start_epoch = int(ckpt.get("epoch", -1)) + 1
        best_val_macro_f1 = float(ckpt.get("best_val_macro_f1", -1.0))
        history = list(ckpt.get("history", []))
        best_epoch = max(
            [int(row["epoch"]) for row in history if row.get("is_best", False)],
            default=-1,
        )
        print(f"Resumed from latest checkpoint at epoch {start_epoch}.")

    training_start_time = time.time()

    for epoch in range(start_epoch, EPOCHS):
        lr = set_warmup_cosine_lr(
            optimizer=optimizer,
            epoch=epoch,
            total_epochs=EPOCHS,
            base_lr=BASE_LR,
            warmup_epochs=WARMUP_EPOCHS,
            min_lr_ratio=MIN_LR_RATIO,
        )

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
        )

        val_metrics, _ = evaluate_model(
            model=model,
            loader=val_loader,
            criterion=criterion,
        )

        aug_totals = train_metrics["aug_totals"]
        total_seen = (
            aug_totals["masked"]
            + aug_totals["unmasked"]
        )
        observed_mask_rate = (
            aug_totals["masked"] / max(1, total_seen)
        )

        is_best = (
            val_metrics["macro_f1"] > best_val_macro_f1
        )

        if is_best:
            best_val_macro_f1 = float(val_metrics["macro_f1"])
            best_epoch = epoch + 1
            patience = 0

            save_checkpoint(
                path=BEST_CKPT_PATH,
                model=model,
                epoch=epoch,
                best_val_macro_f1=best_val_macro_f1,
                variant={
                    "model_key": model_key,
                    "display_name": display_name,
                    "mask_policy": "random_only",
                    "lead_mask_prob": RANDOM_MASK_PROB,
                    "random_kept_lead_range": [
                        MIN_RANDOM_KEPT_LEADS,
                        MAX_RANDOM_KEPT_LEADS,
                    ],
                    "protocol": "Step14 matched",
                },
                history=history,
            )
        else:
            patience += 1

        row = {
            "epoch": epoch + 1,
            "lr": lr,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "observed_train_mask_rate": observed_mask_rate,
            "observed_train_masked_count": aug_totals["masked"],
            "observed_train_unmasked_count": aug_totals["unmasked"],
            "observed_train_random_subset_count": aug_totals["random_subset"],
            "is_best": bool(is_best),
            "patience_counter": int(patience),
        }

        history.append(row)
        save_history_csv(history, HISTORY_CSV)

        cardinality_row = {
            "epoch": epoch + 1,
            "masked_count": int(aug_totals["masked"]),
            "unmasked_count": int(aug_totals["unmasked"]),
        }

        for k in range(1, 13):
            cardinality_row[f"kept_{k}_lead_count"] = int(
                aug_totals["kept_count_distribution"][str(k)]
            )

        cardinality_rows.append(cardinality_row)
        save_mask_cardinality_history(cardinality_rows)

        save_checkpoint(
            path=LATEST_CKPT_PATH,
            model=model,
            epoch=epoch,
            best_val_macro_f1=best_val_macro_f1,
            variant={
                "model_key": model_key,
                "display_name": display_name,
                "mask_policy": "random_only",
                "lead_mask_prob": RANDOM_MASK_PROB,
                "random_kept_lead_range": [
                    MIN_RANDOM_KEPT_LEADS,
                    MAX_RANDOM_KEPT_LEADS,
                ],
                "protocol": "Step14 matched",
            },
            history=history,
        )

        print(
            f"Epoch {epoch + 1:03d}/{EPOCHS} | "
            f"lr={lr:.3e} | "
            f"train loss={train_metrics['loss']:.5f}, "
            f"train Macro-F1={train_metrics['macro_f1']:.5f} | "
            f"val loss={val_metrics['loss']:.5f}, "
            f"val Macro-F1={val_metrics['macro_f1']:.5f} | "
            f"mask rate={observed_mask_rate:.4f} | "
            f"best={is_best} | "
            f"patience={patience}/{EARLY_STOPPING_PATIENCE}"
        )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if patience >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping triggered at epoch {epoch + 1}. "
                f"Best epoch = {best_epoch}, "
                f"Best Val Macro-F1 = {best_val_macro_f1:.6f}"
            )
            break

    total_training_minutes = (
        time.time() - training_start_time
    ) / 60.0

                                                                    
    best_model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    load_checkpoint(BEST_CKPT_PATH, best_model)
    best_model.eval()

    reduced_results = []

    for lead_key, config in LEAD_CONFIGS.items():
        test_loader = make_test_loader(config["lead_indices"])

        test_metrics, pred_df = evaluate_model(
            model=best_model,
            loader=test_loader,
            criterion=None,
        )

        save_eval_artifacts(
            model_key=model_key,
            lead_key=lead_key,
            metrics=test_metrics,
            pred_df=pred_df,
        )

        row = {
            "model_key": model_key,
            "display_name": display_name,
            "lead_key": lead_key,
            "lead_display_name": config["display_name"],
            "accuracy": test_metrics["accuracy"],
            "balanced_accuracy": test_metrics["balanced_accuracy"],
            "macro_f1": test_metrics["macro_f1"],
            "weighted_f1": test_metrics["weighted_f1"],
        }

        for cls in CLASS_NAMES:
            row[f"recall_{cls}"] = test_metrics["per_class_recall"][cls]
            row[f"f1_{cls}"] = test_metrics["per_class_f1"][cls]

        reduced_results.append(row)

        print(
            f"TEST | {lead_key:<22s} | "
            f"Macro-F1={test_metrics['macro_f1']:.6f} | "
            f"Weighted-F1={test_metrics['weighted_f1']:.6f} | "
            f"BalancedAcc={test_metrics['balanced_accuracy']:.6f}"
        )

    pd.DataFrame(reduced_results).to_csv(
        REDUCED_LEAD_RESULTS_CSV,
        index=False,
    )

    training_result = {
        "model_key": model_key,
        "display_name": display_name,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_macro_f1),
        "total_training_minutes": float(total_training_minutes),
        "best_checkpoint": str(BEST_CKPT_PATH),
        "latest_checkpoint": str(LATEST_CKPT_PATH),
    }

    pd.DataFrame([training_result]).to_csv(
        TRAINING_RESULTS_CSV,
        index=False,
    )

    return {
        "training_result": training_result,
        "reduced_results": reduced_results,
    }

                                                                                                            
                 
                                                                                                            

def save_config_json(
    preflight_results: Dict[str, Any],
) -> None:
    payload = {
        "experiment": "STEP 21C Chapman Random-only p=0.60, Step14-protocol-matched",
        "paths": {
            "project_root": str(PROJECT_ROOT),
            "data_dir": str(DATA_DIR),
            "out_dir": str(OUT_DIR),
            "best_ckpt": str(BEST_CKPT_PATH),
            "latest_ckpt": str(LATEST_CKPT_PATH),
        },
        "protocol_lock": EXPECTED_PROTOCOL,
        "architecture_preflight": preflight_results,
        "task": {
            "class_names": CLASS_NAMES,
            "num_classes": NUM_CLASSES,
            "signal_shape": [TARGET_LENGTH, N_LEADS],
        },
        "random_mask_policy": {
            "mask_probability": RANDOM_MASK_PROB,
            "random_kept_lead_count_distribution": "Uniform integer k in [1,12]",
            "min_random_kept_leads": MIN_RANDOM_KEPT_LEADS,
            "max_random_kept_leads": MAX_RANDOM_KEPT_LEADS,
        },
        "hyperparameters": {
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "base_lr": BASE_LR,
            "weight_decay": WEIGHT_DECAY,
            "warmup_epochs": WARMUP_EPOCHS,
            "min_lr_ratio": MIN_LR_RATIO,
            "grad_clip_norm": GRAD_CLIP_NORM,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "dropout": DROPOUT,
            "loss": "Plain CrossEntropyLoss",
            "train_loader": "shuffle=True",
        },
    }

    with open(CONFIG_JSON, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

                                                                                                            
                    
                                                                                                            

def write_final_summary(
    preflight_results: Dict[str, Any],
    output: Dict[str, Any],
) -> str:
    training = output["training_result"]
    results = output["reduced_results"]

    lines = []

    lines.append("=" * 180)
    lines.append("STEP 21C — FINAL CORRECTED CHAPMAN RANDOM-ONLY p=0.60 SUMMARY")
    lines.append("=" * 180)
    lines.append("Purpose:")
    lines.append("  Retrain the Chapman Random-only comparator so it is:")
    lines.append("    (1) architecture-matched to Chapman Standard/Clinical,")
    lines.append("    (2) training-protocol-matched to Chapman Step 14,")
    lines.append("    (3) random-policy-matched to PTB-XL Random-only.")
    lines.append("")

    lines.append("STRICT ARCHITECTURE PREFLIGHT")
    for key, value in preflight_results.items():
        lines.append(
            f"  {key:<22s} | "
            f"strict_load_passed={value['strict_load_passed']} | "
            f"num_state_keys={value['num_state_keys']} | "
            f"{value['checkpoint']}"
        )
    lines.append("")

    lines.append("PROTOCOL MATCH")
    lines.append(f"  Batch size                    : {BATCH_SIZE}")
    lines.append(f"  Epochs                        : {EPOCHS}")
    lines.append(f"  Base LR                       : {BASE_LR}")
    lines.append(f"  Weight decay                  : {WEIGHT_DECAY}")
    lines.append(f"  Warmup epochs                 : {WARMUP_EPOCHS}")
    lines.append(f"  Min LR ratio                  : {MIN_LR_RATIO}")
    lines.append(f"  Early-stop patience           : {EARLY_STOPPING_PATIENCE}")
    lines.append(f"  Dropout                       : {DROPOUT}")
    lines.append("  Loss                          : Plain CrossEntropyLoss")
    lines.append("  Train loader                  : shuffle=True")
    lines.append("")

    lines.append("RANDOM MASKING POLICY")
    lines.append(f"  Mask probability              : {RANDOM_MASK_PROB}")
    lines.append("  Retained lead count           : k ~ Uniform integer in {1,...,12}")
    lines.append("  Lead subset                   : uniformly random subset of size k")
    lines.append("")

    lines.append("TRAINING RESULT")
    lines.append(f"  Best epoch                    : {training['best_epoch']}")
    lines.append(f"  Best validation Macro-F1      : {training['best_val_macro_f1']:.6f}")
    lines.append(f"  Training time (minutes)       : {training['total_training_minutes']:.2f}")
    lines.append(f"  Best checkpoint               : {training['best_checkpoint']}")
    lines.append("")

    lines.append("KNOWN LEAD-CONDITION TEST RESULTS")
    for row in results:
        lines.append(
            f"  {row['lead_key']:<22s} | "
            f"Macro-F1={row['macro_f1']:.6f} | "
            f"Weighted-F1={row['weighted_f1']:.6f} | "
            f"BalancedAcc={row['balanced_accuracy']:.6f}"
        )
    lines.append("")

    lines.append("FILES SAVED")
    for p in [
        BEST_CKPT_PATH,
        LATEST_CKPT_PATH,
        HISTORY_CSV,
        TRAINING_RESULTS_CSV,
        REDUCED_LEAD_RESULTS_CSV,
        MASK_CARDINALITY_HISTORY_CSV,
        CONFIG_JSON,
        SUMMARY_TXT,
    ]:
        lines.append(f"  - {p}")
    lines.append("")

    lines.append("NEXT REQUIRED ACTION")
    lines.append("  1) Update Step 22 Chapman_Random checkpoint path to the new Step 21C checkpoint.")
    lines.append("  2) Rerun Step 22 final evaluation.")
    lines.append("  3) Rerun Step 23 figure generation.")
    lines.append("  4) Revise manuscript wording so Random masking is described as")
    lines.append("     a broad random subset policy, not a matched-cardinality structure-isolation baseline.")
    lines.append("=" * 180)

    summary = "\n".join(lines)

    with open(SUMMARY_TXT, "w", encoding="utf-8") as f:
        f.write(summary)

    return summary

                                                                                                            
          
                                                                                                            

def main() -> None:
    print("=" * 180)
    print("STEP 21C — CHAPMAN RANDOM-ONLY p=0.60, FULL PROTOCOL-CORRECTED RUN")
    print("=" * 180)
    print(f"Device       : {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"Data dir     : {DATA_DIR}")
    print(f"Output dir   : {OUT_DIR}")
    print("=" * 180)

    verify_required_files()
    run_protocol_lock_check()

    print("\nRunning strict architecture preflight...")
    preflight_results = strict_architecture_preflight()
    print("Strict architecture preflight passed ")

    save_config_json(preflight_results)

    output = train_chapman_random_variant()

    summary = write_final_summary(
        preflight_results=preflight_results,
        output=output,
    )

    print("\n" + summary)

if __name__ == "__main__":
    main()


05_final_evaluation_bootstrap_mi

In [ ]:

from __future__ import annotations

import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm



PROJECT_ROOT = Path(
    "YOUR_PROJECT_PATH"
)

PTBXL_PROCESSED_DIR = PROJECT_ROOT / "Data"

CHAPMAN_PROCESSED_DIR = PROJECT_ROOT / "processed_chapman_4class_raw_ecgdata"

OUT_DIR = PROJECT_ROOT / "lead_masking_final" / "step22_final_evaluation_bootstrap_mi"

PREDICTION_DIR = OUT_DIR / "predictions"
REPORT_DIR = OUT_DIR / "reports"
TABLE_DIR = OUT_DIR / "tables"

for d in [OUT_DIR, PREDICTION_DIR, REPORT_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_CONFIG_JSON = REPORT_DIR / "step22_run_config.json"
FINAL_SUMMARY_TXT = REPORT_DIR / "step22_final_summary.txt"

CONSOLIDATED_LONG_CSV = TABLE_DIR / "step22_all_methods_all_conditions_metrics_long.csv"
CONSOLIDATED_MASTER_CSV = TABLE_DIR / "step22_consolidated_master_macro_f1_table.csv"
BOOTSTRAP_RESULTS_CSV = TABLE_DIR / "step22_pairwise_bootstrap_macro_f1.csv"
UNSEEN_TRANSFER_SUMMARY_CSV = TABLE_DIR / "step22_unseen_transfer_level_assessment.csv"
PTBXL_MI_ANALYSIS_CSV = TABLE_DIR / "step22_ptbxl_mi_precision_recall_f1.csv"
PTBXL_MI_COMPARISON_CSV = TABLE_DIR / "step22_ptbxl_mi_comparison_summary.csv"

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()
NUM_WORKERS = 0
BATCH_SIZE = 128

N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 2026
CI_LOW = 2.5
CI_HIGH = 97.5

N_LEADS = 12
SIGNAL_LENGTH = 5000

LEAD_NAMES = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]
LEAD_TO_IDX = {lead: i for i, lead in enumerate(LEAD_NAMES)}

LEAD_CONDITIONS: Dict[str, Dict[str, Any]] = {
    "12lead_full": {
        "leads": LEAD_NAMES,
        "group": "known",
        "description": "Full 12-lead input",
    },
    "6_limb": {
        "leads": ["I", "II", "III", "aVR", "aVL", "aVF"],
        "group": "known",
        "description": "Six limb leads",
    },
    "6_precordial": {
        "leads": ["V1", "V2", "V3", "V4", "V5", "V6"],
        "group": "known",
        "description": "Six precordial leads",
    },
    "3_limb": {
        "leads": ["I", "II", "III"],
        "group": "known",
        "description": "Three limb leads",
    },
    "lead_II_only": {
        "leads": ["II"],
        "group": "known",
        "description": "Lead II only",
    },
    "V5_only": {
        "leads": ["V5"],
        "group": "known",
        "description": "V5 only",
    },

    "lead_I_only_unseen": {
        "leads": ["I"],
        "group": "unseen",
        "description": "Lead I only; unseen single-limb configuration",
    },
    "V1_only_unseen": {
        "leads": ["V1"],
        "group": "unseen",
        "description": "V1 only; unseen single-precordial configuration",
    },
    "I_II_unseen": {
        "leads": ["I", "II"],
        "group": "unseen",
        "description": "I+II paired limb-lead configuration",
    },
    "V1_V5_unseen": {
        "leads": ["V1", "V5"],
        "group": "unseen",
        "description": "V1+V5 paired precordial configuration",
    },
}

UNSEEN_CONDITIONS = [
    "lead_I_only_unseen",
    "V1_only_unseen",
    "I_II_unseen",
    "V1_V5_unseen",
]

KNOWN_CONDITIONS = [
    "12lead_full",
    "6_limb",
    "6_precordial",
    "3_limb",
    "lead_II_only",
    "V5_only",
]

DATASET_CONFIGS: Dict[str, Dict[str, Any]] = {
    "PTBXL": {
        "processed_dir": PTBXL_PROCESSED_DIR,
        "test_signals": PTBXL_PROCESSED_DIR / "test_signals.npy",
        "test_labels": PTBXL_PROCESSED_DIR / "test_labels.npy",
        "class_names": ["NORM", "MI", "STTC", "CD", "HYP"],
        "num_classes": 5,
        "mi_class_index": 1,
    },
    "Chapman": {
        "processed_dir": CHAPMAN_PROCESSED_DIR,
        "test_signals": CHAPMAN_PROCESSED_DIR / "test_signals.npy",
        "test_labels": CHAPMAN_PROCESSED_DIR / "test_labels.npy",
        "class_names": ["SB", "AFIB", "GSVT", "SR"],
        "num_classes": 4,
        "mi_class_index": None,
    },
}


CHECKPOINT_OVERRIDES: Dict[str, Optional[Path]] = {
    "PTBXL_Standard": Path(
        "YOUR_PROJECT_PATH"
    ),

    "PTBXL_Random": Path(
        "YOUR_PROJECT_PATH"
    ),

    "PTBXL_Clinical": Path(
        "YOUR_PROJECT_PATH"
    ),

    "Chapman_Standard": Path(
        "YOUR_PROJECT_PATH"
    ),

    "Chapman_Random": Path(
        "YOUR_PROJECT_PATH"
    ),

    "Chapman_Clinical": Path(
        "YOUR_PROJECT_PATH"
    ),
}

AUTO_DISCOVER_CHECKPOINTS = False

CHECKPOINT_SEARCH_HINTS: Dict[str, List[str]] = {
    "PTBXL_Standard": ["ptb", "standard", "best"],
    "PTBXL_Random": ["ptb", "random", "best"],
    "PTBXL_Clinical": ["ptb", "clinical", "best"],
    "Chapman_Standard": ["chapman", "standard", "best"],
    "Chapman_Random": ["chapman", "random", "p060", "best"],
    "Chapman_Clinical": ["chapman", "clinical", "best"],
}

BASE_FILTERS = 64
KERNEL_SIZE = 7
DROPOUT = 0.25



def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


def require_exists(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def save_json(payload: Dict[str, Any], path: Path) -> None:
    def convert(obj: Any) -> Any:
        if isinstance(obj, Path):
            return str(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if torch.is_tensor(obj):
            if obj.numel() == 1:
                return obj.detach().cpu().item()
            return obj.detach().cpu().tolist()
        return obj

    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, default=convert)


def condition_keep_indices(condition_name: str) -> List[int]:
    leads = LEAD_CONDITIONS[condition_name]["leads"]
    return [LEAD_TO_IDX[lead] for lead in leads]


def apply_keep_indices_mask(signal_ch_time: np.ndarray, keep_indices: Sequence[int]) -> np.ndarray:
    x = np.asarray(signal_ch_time, dtype=np.float32).copy()
    keep_set = set(int(i) for i in keep_indices)

    for lead_idx in range(N_LEADS):
        if lead_idx not in keep_set:
            x[lead_idx, :] = 0.0

    return x


def format_float(x: float) -> str:
    return f"{x:.6f}"



def list_all_checkpoint_files(root: Path) -> List[Path]:
    files = []
    for ext in ["*.pt", "*.pth", "*.ckpt"]:
        files.extend(root.rglob(ext))
    return sorted(set(files))


def checkpoint_score(path: Path, hints: List[str]) -> int:
    text = str(path).lower()
    return sum(1 for hint in hints if hint.lower() in text)


def auto_discover_checkpoint(method_key: str) -> Path:
    hints = CHECKPOINT_SEARCH_HINTS[method_key]
    candidates = list_all_checkpoint_files(PROJECT_ROOT)

    scored = []
    for path in candidates:
        score = checkpoint_score(path, hints)
        if score > 0:
            scored.append((score, path))

    scored = sorted(scored, key=lambda x: (-x[0], str(x[1])))

    if not scored:
        raise FileNotFoundError(
            f"No checkpoint candidates found for {method_key} using hints {hints}"
        )

    best_score = scored[0][0]
    top = [p for s, p in scored if s == best_score]

    if len(top) == 1:
        print(f"[AUTO] {method_key} -> {top[0]}")
        return top[0]

    print(f"\nAmbiguous checkpoint discovery for: {method_key}")
    print(f"Hints: {hints}")
    print("Top candidates:")
    for p in top[:20]:
        print("  ", p)

    raise RuntimeError(
        f"Multiple equally likely checkpoint candidates found for {method_key}. "
        f"Paste the correct path into CHECKPOINT_OVERRIDES['{method_key}']."
    )


def resolve_checkpoint_paths() -> Dict[str, Path]:
    resolved: Dict[str, Path] = {}

    for method_key, override in CHECKPOINT_OVERRIDES.items():
        if override is not None:
            require_exists(override, f"checkpoint override for {method_key}")
            resolved[method_key] = override
            print(f"[OVERRIDE] {method_key} -> {override}")
        elif AUTO_DISCOVER_CHECKPOINTS:
            resolved[method_key] = auto_discover_checkpoint(method_key)
        else:
            raise FileNotFoundError(
                f"No checkpoint override supplied for {method_key} and auto-discovery disabled."
            )

    return resolved



class FixedLeadTestDataset(Dataset):
    def __init__(self, signals_path: Path, labels_path: Path, condition_name: str):
        self.signals = np.load(signals_path, mmap_mode="r")
        self.labels = np.load(labels_path, mmap_mode="r").astype(np.int64)
        self.condition_name = condition_name
        self.keep_indices = condition_keep_indices(condition_name)

    def __len__(self) -> int:
        return int(len(self.labels))

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = np.asarray(self.signals[int(idx)], dtype=np.float32).T  # [12,5000]
        x = apply_keep_indices_mask(x, self.keep_indices)
        y = int(self.labels[int(idx)])
        return torch.from_numpy(x.copy()).float(), torch.tensor(y, dtype=torch.long)



EMBED_DIM = 512


class BasicBlock1D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = KERNEL_SIZE,
        stride: int = 1,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        padding = kernel_size // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            bias=False,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.dropout = nn.Dropout(dropout)

        self.downsample = None

        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        if self.downsample is not None:
            identity = self.downsample(x)

        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out = F.relu(out + identity, inplace=True)

        return out


class LegacyECGEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                N_LEADS,
                BASE_FILTERS,
                kernel_size=15,
                stride=2,
                padding=7,
                bias=False,
            ),
            nn.BatchNorm1d(BASE_FILTERS),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )

        self.backbone = nn.Sequential(
            self._make_stage(BASE_FILTERS, BASE_FILTERS, blocks=2, stride=1),
            self._make_stage(BASE_FILTERS, BASE_FILTERS * 2, blocks=2, stride=2),
            self._make_stage(BASE_FILTERS * 2, BASE_FILTERS * 4, blocks=2, stride=2),
            self._make_stage(BASE_FILTERS * 4, BASE_FILTERS * 8, blocks=2, stride=2),
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(BASE_FILTERS * 8, EMBED_DIM),
            nn.BatchNorm1d(EMBED_DIM),
            nn.ReLU(inplace=True),
        )

    def _make_stage(
        self,
        in_channels: int,
        out_channels: int,
        blocks: int,
        stride: int,
    ) -> nn.Sequential:
        layers: List[nn.Module] = [
            BasicBlock1D(
                in_channels=in_channels,
                out_channels=out_channels,
                stride=stride,
            )
        ]

        for _ in range(1, blocks):
            layers.append(
                BasicBlock1D(
                    in_channels=out_channels,
                    out_channels=out_channels,
                    stride=1,
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.backbone(x)
        x = self.global_pool(x)
        x = self.embedding_head(x)
        return x


class LegacyECGClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.encoder = LegacyECGEncoder()
        self.classifier = nn.Linear(EMBED_DIM, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        logits = self.classifier(z)
        return logits


def extract_state_dict_from_checkpoint(ckpt_obj: Any) -> Dict[str, torch.Tensor]:
    if isinstance(ckpt_obj, dict):
        for key in ["model_state_dict", "state_dict", "model", "net"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                state = ckpt_obj[key]
                break
        else:
            if all(isinstance(k, str) for k in ckpt_obj.keys()):
                state = ckpt_obj
            else:
                raise ValueError("Unable to locate model state dict in checkpoint.")
    else:
        raise ValueError("Unexpected checkpoint object type.")

    cleaned = {}
    for k, v in state.items():
        new_k = k[7:] if k.startswith("module.") else k
        cleaned[new_k] = v

    return cleaned


def load_model_for_method(dataset_name: str, checkpoint_path: Path) -> nn.Module:
    num_classes = DATASET_CONFIGS[dataset_name]["num_classes"]

    model = LegacyECGClassifier(num_classes=num_classes).to(DEVICE)

    ckpt = torch.load(checkpoint_path, map_location="cpu")
    state = extract_state_dict_from_checkpoint(ckpt)

    model.load_state_dict(state, strict=True)
    model.to(DEVICE)
    model.eval()

    return model



@torch.no_grad()
def evaluate_model_condition(
    model: nn.Module,
    dataset_name: str,
    condition_name: str,
) -> Dict[str, Any]:
    cfg = DATASET_CONFIGS[dataset_name]

    dataset = FixedLeadTestDataset(
        signals_path=cfg["test_signals"],
        labels_path=cfg["test_labels"],
        condition_name=condition_name,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    all_probs: List[np.ndarray] = []
    all_preds: List[np.ndarray] = []
    all_targets: List[np.ndarray] = []

    for x, y in tqdm(
        loader,
        desc=f"{dataset_name} | {condition_name}",
        leave=False,
    ):
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)

        probs = torch.softmax(logits.float(), dim=1)
        preds = torch.argmax(probs, dim=1)

        all_probs.append(probs.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.numpy())

    y_true = np.concatenate(all_targets).astype(int)
    y_pred = np.concatenate(all_preds).astype(int)
    y_prob = np.concatenate(all_probs)

    class_names = cfg["class_names"]
    num_classes = cfg["num_classes"]

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(num_classes),
        average=None,
        zero_division=0,
    )

    metrics = {
        "dataset": dataset_name,
        "lead_condition": condition_name,
        "condition_group": LEAD_CONDITIONS[condition_name]["group"],
        "n_test": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=np.arange(num_classes),
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=np.arange(num_classes),
                average="weighted",
                zero_division=0,
            )
        ),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred,
            labels=np.arange(num_classes),
        ),
        "per_class_precision": {
            class_names[i]: float(precision[i]) for i in range(num_classes)
        },
        "per_class_recall": {
            class_names[i]: float(recall[i]) for i in range(num_classes)
        },
        "per_class_f1": {
            class_names[i]: float(f1[i]) for i in range(num_classes)
        },
        "per_class_support": {
            class_names[i]: int(support[i]) for i in range(num_classes)
        },
    }

    return metrics


def save_prediction_file(
    dataset_name: str,
    method_name: str,
    condition_name: str,
    metrics: Dict[str, Any],
) -> Path:
    class_names = DATASET_CONFIGS[dataset_name]["class_names"]

    df = pd.DataFrame({
        "sample_idx": np.arange(len(metrics["y_true"]), dtype=int),
        "y_true": metrics["y_true"],
        "y_pred": metrics["y_pred"],
    })

    for idx, cname in enumerate(class_names):
        df[f"prob_{cname}"] = metrics["y_prob"][:, idx]

    out_path = (
        PREDICTION_DIR
        / f"{dataset_name.lower()}_{method_name.lower()}_{condition_name}_predictions.csv"
    )
    df.to_csv(out_path, index=False)
    return out_path



def metrics_to_long_row(
    dataset_name: str,
    method_name: str,
    condition_name: str,
    metrics: Dict[str, Any],
) -> Dict[str, Any]:
    row = {
        "dataset": dataset_name,
        "method": method_name,
        "lead_condition": condition_name,
        "condition_group": LEAD_CONDITIONS[condition_name]["group"],
        "n_test": metrics["n_test"],
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
    }

    for cname, value in metrics["per_class_precision"].items():
        row[f"precision_{cname}"] = value

    for cname, value in metrics["per_class_recall"].items():
        row[f"recall_{cname}"] = value

    for cname, value in metrics["per_class_f1"].items():
        row[f"f1_{cname}"] = value

    for cname, value in metrics["per_class_support"].items():
        row[f"support_{cname}"] = value

    return row


def build_consolidated_master_table(metrics_long_df: pd.DataFrame) -> pd.DataFrame:
    pivot = metrics_long_df.pivot_table(
        index=["dataset", "lead_condition", "condition_group"],
        columns="method",
        values="macro_f1",
        aggfunc="first",
    ).reset_index()

    rename_map = {
        "Standard": "macro_f1_standard",
        "Random": "macro_f1_random",
        "Clinical": "macro_f1_clinical",
    }
    pivot = pivot.rename(columns=rename_map)

    if {"macro_f1_standard", "macro_f1_clinical"}.issubset(pivot.columns):
        pivot["clinical_minus_standard"] = (
            pivot["macro_f1_clinical"] - pivot["macro_f1_standard"]
        )

    if {"macro_f1_random", "macro_f1_clinical"}.issubset(pivot.columns):
        pivot["clinical_minus_random"] = (
            pivot["macro_f1_clinical"] - pivot["macro_f1_random"]
        )

    if {"macro_f1_standard", "macro_f1_random"}.issubset(pivot.columns):
        pivot["random_minus_standard"] = (
            pivot["macro_f1_random"] - pivot["macro_f1_standard"]
        )

    lead_order = {name: i for i, name in enumerate(LEAD_CONDITIONS.keys())}
    pivot["lead_order"] = pivot["lead_condition"].map(lead_order)
    pivot = pivot.sort_values(["dataset", "lead_order"]).drop(columns=["lead_order"])

    return pivot



def macro_f1_fixed_labels(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    num_classes: int,
) -> float:
    return float(
        f1_score(
            y_true,
            y_pred,
            labels=np.arange(num_classes),
            average="macro",
            zero_division=0,
        )
    )


def paired_bootstrap_macro_f1_diff(
    y_true: np.ndarray,
    y_pred_a: np.ndarray,
    y_pred_b: np.ndarray,
    num_classes: int,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = BOOTSTRAP_SEED,
) -> Dict[str, Any]:
    rng = np.random.default_rng(seed)
    n = len(y_true)

    observed_a = macro_f1_fixed_labels(y_true, y_pred_a, num_classes)
    observed_b = macro_f1_fixed_labels(y_true, y_pred_b, num_classes)
    observed_diff = observed_a - observed_b

    diffs = np.empty(n_bootstrap, dtype=np.float64)

    for i in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        diff_i = (
            macro_f1_fixed_labels(y_true[idx], y_pred_a[idx], num_classes)
            - macro_f1_fixed_labels(y_true[idx], y_pred_b[idx], num_classes)
        )
        diffs[i] = diff_i

    lower = float(np.percentile(diffs, CI_LOW))
    upper = float(np.percentile(diffs, CI_HIGH))
    p_le_zero = float(np.mean(diffs <= 0.0))

    return {
        "observed_a": observed_a,
        "observed_b": observed_b,
        "observed_diff": float(observed_diff),
        "bootstrap_mean_diff": float(np.mean(diffs)),
        "ci_lower": lower,
        "ci_upper": upper,
        "bootstrap_fraction_diff_le_0": p_le_zero,
    }


def run_all_pairwise_bootstraps(
    prediction_store: Dict[Tuple[str, str, str], Dict[str, Any]]
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    comparisons = [
        ("Clinical", "Standard"),
        ("Clinical", "Random"),
        ("Random", "Standard"),
    ]

    for dataset_name, cfg in DATASET_CONFIGS.items():
        num_classes = cfg["num_classes"]

        for condition_name in LEAD_CONDITIONS.keys():
            for method_a, method_b in comparisons:
                key_a = (dataset_name, method_a, condition_name)
                key_b = (dataset_name, method_b, condition_name)

                if key_a not in prediction_store or key_b not in prediction_store:
                    continue

                pred_a = prediction_store[key_a]
                pred_b = prediction_store[key_b]

                y_true_a = pred_a["y_true"]
                y_true_b = pred_b["y_true"]

                if not np.array_equal(y_true_a, y_true_b):
                    raise ValueError(
                        f"y_true mismatch for {dataset_name}, {condition_name}, "
                        f"{method_a} vs {method_b}"
                    )

                boot = paired_bootstrap_macro_f1_diff(
                    y_true=y_true_a,
                    y_pred_a=pred_a["y_pred"],
                    y_pred_b=pred_b["y_pred"],
                    num_classes=num_classes,
                    n_bootstrap=N_BOOTSTRAP,
                    seed=BOOTSTRAP_SEED,
                )

                rows.append({
                    "dataset": dataset_name,
                    "lead_condition": condition_name,
                    "condition_group": LEAD_CONDITIONS[condition_name]["group"],
                    "comparison": f"{method_a} - {method_b}",
                    "method_a": method_a,
                    "method_b": method_b,
                    **boot,
                })

    df = pd.DataFrame(rows)
    return df



def bootstrap_average_gain_across_unseen_conditions(
    prediction_store: Dict[Tuple[str, str, str], Dict[str, Any]],
    dataset_name: str,
    method_a: str,
    method_b: str,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = BOOTSTRAP_SEED + 77,
) -> Dict[str, Any]:
    cfg = DATASET_CONFIGS[dataset_name]
    num_classes = cfg["num_classes"]

    first_key = (dataset_name, method_a, UNSEEN_CONDITIONS[0])
    y_true_ref = prediction_store[first_key]["y_true"]
    n = len(y_true_ref)

    rng = np.random.default_rng(seed)
    diffs = np.empty(n_bootstrap, dtype=np.float64)

    observed_condition_diffs = []

    for condition_name in UNSEEN_CONDITIONS:
        pred_a = prediction_store[(dataset_name, method_a, condition_name)]
        pred_b = prediction_store[(dataset_name, method_b, condition_name)]

        if not np.array_equal(pred_a["y_true"], pred_b["y_true"]):
            raise ValueError("y_true mismatch during unseen transfer assessment.")

        obs_diff = (
            macro_f1_fixed_labels(pred_a["y_true"], pred_a["y_pred"], num_classes)
            - macro_f1_fixed_labels(pred_b["y_true"], pred_b["y_pred"], num_classes)
        )
        observed_condition_diffs.append(obs_diff)

    observed_avg_gain = float(np.mean(observed_condition_diffs))
    positive_condition_count = int(np.sum(np.array(observed_condition_diffs) > 0))

    for b in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        condition_diffs = []

        for condition_name in UNSEEN_CONDITIONS:
            pred_a = prediction_store[(dataset_name, method_a, condition_name)]
            pred_b = prediction_store[(dataset_name, method_b, condition_name)]

            diff_i = (
                macro_f1_fixed_labels(
                    pred_a["y_true"][idx],
                    pred_a["y_pred"][idx],
                    num_classes,
                )
                - macro_f1_fixed_labels(
                    pred_b["y_true"][idx],
                    pred_b["y_pred"][idx],
                    num_classes,
                )
            )
            condition_diffs.append(diff_i)

        diffs[b] = float(np.mean(condition_diffs))

    ci_lower = float(np.percentile(diffs, CI_LOW))
    ci_upper = float(np.percentile(diffs, CI_HIGH))
    p_le_zero = float(np.mean(diffs <= 0.0))

    if positive_condition_count >= 3 and observed_avg_gain > 0 and ci_lower > 0:
        transfer_level = "Level A — Strong unseen-subset transfer"
    elif positive_condition_count >= 2 or observed_avg_gain > 0:
        transfer_level = "Level B — Partial unseen-subset transfer"
    else:
        transfer_level = "Level C — Limited unseen-subset transfer"

    return {
        "dataset": dataset_name,
        "comparison": f"{method_a} - {method_b}",
        "positive_unseen_condition_count": positive_condition_count,
        "observed_unseen_condition_diffs": json.dumps(
            {UNSEEN_CONDITIONS[i]: float(v) for i, v in enumerate(observed_condition_diffs)}
        ),
        "observed_average_unseen_gain": observed_avg_gain,
        "bootstrap_mean_average_unseen_gain": float(np.mean(diffs)),
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "bootstrap_fraction_average_gain_le_0": p_le_zero,
        "transfer_level": transfer_level,
    }


def build_unseen_transfer_assessment(
    prediction_store: Dict[Tuple[str, str, str], Dict[str, Any]]
) -> pd.DataFrame:
    rows = []

    for dataset_name in DATASET_CONFIGS.keys():
        rows.append(
            bootstrap_average_gain_across_unseen_conditions(
                prediction_store=prediction_store,
                dataset_name=dataset_name,
                method_a="Clinical",
                method_b="Random",
            )
        )
        rows.append(
            bootstrap_average_gain_across_unseen_conditions(
                prediction_store=prediction_store,
                dataset_name=dataset_name,
                method_a="Clinical",
                method_b="Standard",
            )
        )

    return pd.DataFrame(rows)



def build_ptbxl_mi_analysis(
    metrics_long_df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    ptb = metrics_long_df[metrics_long_df["dataset"] == "PTBXL"].copy()

    cols = [
        "dataset",
        "method",
        "lead_condition",
        "condition_group",
        "precision_MI",
        "recall_MI",
        "f1_MI",
        "support_MI",
    ]

    mi_df = ptb[cols].copy()

    pivot = mi_df.pivot_table(
        index=["lead_condition", "condition_group"],
        columns="method",
        values=["precision_MI", "recall_MI", "f1_MI"],
        aggfunc="first",
    )

    pivot.columns = [
        f"{metric}_{method.lower()}"
        for metric, method in pivot.columns
    ]
    pivot = pivot.reset_index()

    for metric in ["precision_MI", "recall_MI", "f1_MI"]:
        clinical_col = f"{metric}_clinical"
        random_col = f"{metric}_random"
        standard_col = f"{metric}_standard"

        if clinical_col in pivot.columns and random_col in pivot.columns:
            pivot[f"{metric}_clinical_minus_random"] = (
                pivot[clinical_col] - pivot[random_col]
            )

        if clinical_col in pivot.columns and standard_col in pivot.columns:
            pivot[f"{metric}_clinical_minus_standard"] = (
                pivot[clinical_col] - pivot[standard_col]
            )

        if random_col in pivot.columns and standard_col in pivot.columns:
            pivot[f"{metric}_random_minus_standard"] = (
                pivot[random_col] - pivot[standard_col]
            )

    lead_order = {name: i for i, name in enumerate(LEAD_CONDITIONS.keys())}
    pivot["lead_order"] = pivot["lead_condition"].map(lead_order)
    pivot = pivot.sort_values("lead_order").drop(columns=["lead_order"])

    return mi_df, pivot



def main() -> None:
    seed_everything(SEED)

    print("=" * 170)
    print("STEP 22 — FINAL EVALUATION: UNSEEN SUBSETS + BOOTSTRAP + PTB-XL MI ANALYSIS")
    print("=" * 170)
    print(f"Device                : {DEVICE}")
    print(f"Project root          : {PROJECT_ROOT}")
    print(f"Output directory      : {OUT_DIR}")
    print(f"Bootstrap replicates  : {N_BOOTSTRAP}")
    print("=" * 170)

    for dataset_name, cfg in DATASET_CONFIGS.items():
        require_exists(cfg["test_signals"], f"{dataset_name} test signals")
        require_exists(cfg["test_labels"], f"{dataset_name} test labels")

        labels = np.load(cfg["test_labels"], mmap_mode="r").astype(int)
        counts = Counter(labels.tolist())
        print(
            f"{dataset_name} test class counts: "
            f"{ {cfg['class_names'][k]: int(v) for k, v in sorted(counts.items())} }"
        )

    checkpoint_paths = resolve_checkpoint_paths()

    methods = {
        "PTBXL": {
            "Standard": checkpoint_paths["PTBXL_Standard"],
            "Random": checkpoint_paths["PTBXL_Random"],
            "Clinical": checkpoint_paths["PTBXL_Clinical"],
        },
        "Chapman": {
            "Standard": checkpoint_paths["Chapman_Standard"],
            "Random": checkpoint_paths["Chapman_Random"],
            "Clinical": checkpoint_paths["Chapman_Clinical"],
        },
    }

    save_json(
        {
            "device": str(DEVICE),
            "project_root": str(PROJECT_ROOT),
            "out_dir": str(OUT_DIR),
            "lead_conditions": LEAD_CONDITIONS,
            "unseen_conditions": UNSEEN_CONDITIONS,
            "known_conditions": KNOWN_CONDITIONS,
            "bootstrap": {
                "n_bootstrap": N_BOOTSTRAP,
                "seed": BOOTSTRAP_SEED,
                "ci_low": CI_LOW,
                "ci_high": CI_HIGH,
            },
            "checkpoint_paths": {k: str(v) for k, v in checkpoint_paths.items()},
        },
        RUN_CONFIG_JSON,
    )

    metrics_rows: List[Dict[str, Any]] = []
    prediction_store: Dict[Tuple[str, str, str], Dict[str, Any]] = {}

    for dataset_name, method_dict in methods.items():
        print("\n" + "=" * 170)
        print(f"EVALUATING DATASET: {dataset_name}")
        print("=" * 170)

        for method_name, ckpt_path in method_dict.items():
            print(f"\nLoading {dataset_name} | {method_name}")
            print(f"Checkpoint: {ckpt_path}")

            model = load_model_for_method(dataset_name, ckpt_path)

            for condition_name in LEAD_CONDITIONS.keys():
                metrics = evaluate_model_condition(
                    model=model,
                    dataset_name=dataset_name,
                    condition_name=condition_name,
                )

                metrics_rows.append(
                    metrics_to_long_row(
                        dataset_name=dataset_name,
                        method_name=method_name,
                        condition_name=condition_name,
                        metrics=metrics,
                    )
                )

                prediction_store[
                    (dataset_name, method_name, condition_name)
                ] = {
                    "y_true": metrics["y_true"],
                    "y_pred": metrics["y_pred"],
                    "y_prob": metrics["y_prob"],
                }

                save_prediction_file(
                    dataset_name=dataset_name,
                    method_name=method_name,
                    condition_name=condition_name,
                    metrics=metrics,
                )

                print(
                    f"  {condition_name:<22s} | "
                    f"Macro-F1={metrics['macro_f1']:.6f} | "
                    f"Weighted-F1={metrics['weighted_f1']:.6f}"
                )

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    metrics_long_df = pd.DataFrame(metrics_rows)
    metrics_long_df.to_csv(CONSOLIDATED_LONG_CSV, index=False)

    master_df = build_consolidated_master_table(metrics_long_df)
    master_df.to_csv(CONSOLIDATED_MASTER_CSV, index=False)

    print("\nRunning paired bootstrap comparisons...")
    bootstrap_df = run_all_pairwise_bootstraps(prediction_store)
    bootstrap_df.to_csv(BOOTSTRAP_RESULTS_CSV, index=False)

    print("\nRunning unseen transfer assessment...")
    transfer_df = build_unseen_transfer_assessment(prediction_store)
    transfer_df.to_csv(UNSEEN_TRANSFER_SUMMARY_CSV, index=False)

    print("\nBuilding PTB-XL MI precision/recall/F1 tables...")
    mi_long_df, mi_comparison_df = build_ptbxl_mi_analysis(metrics_long_df)
    mi_long_df.to_csv(PTBXL_MI_ANALYSIS_CSV, index=False)
    mi_comparison_df.to_csv(PTBXL_MI_COMPARISON_CSV, index=False)

    lines: List[str] = []
    lines.append("=" * 180)
    lines.append("STEP 22 — FINAL EVALUATION SUMMARY")
    lines.append("=" * 180)
    lines.append("Purpose: finalize the paper-ready experimental evidence after Chapman random-only training.")
    lines.append("")

    lines.append("CHECKPOINTS USED")
    for k, v in checkpoint_paths.items():
        lines.append(f"  {k:<22s}: {v}")

    lines.append("")
    lines.append("CONSOLIDATED MASTER TABLE — MACRO-F1")
    for _, row in master_df.iterrows():
        lines.append(
            f"  {row['dataset']:<8s} | {row['lead_condition']:<22s} | "
            f"Std={row.get('macro_f1_standard', np.nan):.6f} | "
            f"Rnd={row.get('macro_f1_random', np.nan):.6f} | "
            f"Clin={row.get('macro_f1_clinical', np.nan):.6f} | "
            f"Clin-Std={row.get('clinical_minus_standard', np.nan):+.6f} | "
            f"Clin-Rnd={row.get('clinical_minus_random', np.nan):+.6f}"
        )

    lines.append("")
    lines.append("UNSEEN TRANSFER ASSESSMENT")
    for _, row in transfer_df.iterrows():
        lines.append(
            f"  {row['dataset']:<8s} | {row['comparison']:<22s} | "
            f"wins={int(row['positive_unseen_condition_count'])}/4 | "
            f"avg_gain={row['observed_average_unseen_gain']:+.6f} | "
            f"CI=[{row['ci_lower']:+.6f}, {row['ci_upper']:+.6f}] | "
            f"{row['transfer_level']}"
        )

    lines.append("")
    lines.append("PTB-XL MI CLINICAL ANALYSIS")
    for _, row in mi_comparison_df.iterrows():
        lines.append(
            f"  {row['lead_condition']:<22s} | "
            f"Recall Clin={row.get('recall_MI_clinical', np.nan):.6f}, "
            f"Rnd={row.get('recall_MI_random', np.nan):.6f}, "
            f"Std={row.get('recall_MI_standard', np.nan):.6f} | "
            f"F1 Clin={row.get('f1_MI_clinical', np.nan):.6f}, "
            f"Rnd={row.get('f1_MI_random', np.nan):.6f}, "
            f"Std={row.get('f1_MI_standard', np.nan):.6f}"
        )

    lines.append("")
    lines.append("FILES SAVED")
    for p in [
        CONSOLIDATED_LONG_CSV,
        CONSOLIDATED_MASTER_CSV,
        BOOTSTRAP_RESULTS_CSV,
        UNSEEN_TRANSFER_SUMMARY_CSV,
        PTBXL_MI_ANALYSIS_CSV,
        PTBXL_MI_COMPARISON_CSV,
        RUN_CONFIG_JSON,
        FINAL_SUMMARY_TXT,
    ]:
        lines.append(f"  - {p}")

    lines.append("")
    lines.append("NEXT CONTEXT")
    lines.append("  After this output is reviewed, experiments are complete.")
    lines.append("  The next stage is paper writing: title, abstract, contributions, tables, and Results section.")
    lines.append("=" * 180)

    final_summary = "\n".join(lines)

    with open(FINAL_SUMMARY_TXT, "w", encoding="utf-8") as f:
        f.write(final_summary)

    print("\n" + final_summary)


if __name__ == "__main__":
    main()

audit_checkpoint_signatures

In [ ]:

from pathlib import Path
import torch
import json
from collections import Counter

CHECKPOINTS = {
    "PTBXL_Standard": Path(
        r"PROJECT_PATH"
    ),

    "PTBXL_Random": Path(
        r"PROJECT_PATH"
    ),

    "PTBXL_Clinical": Path(
        r"PROJECT_PATH"
    ),

    "Chapman_Standard": Path(
        r"PROJECT_PATH"
    ),

    "Chapman_Random_Step21": Path(
        r"PROJECT_PATH"
    ),

    "Chapman_Clinical": Path(
        r"PROJECT_PATH"
    ),
}


def require_exists(path: Path, name: str):
    if not path.exists():
        raise FileNotFoundError(f"Missing checkpoint for {name}: {path}")


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for candidate in ["model_state_dict", "state_dict", "model", "net"]:
            if candidate in ckpt and isinstance(ckpt[candidate], dict):
                return ckpt[candidate], candidate

        if all(isinstance(k, str) for k in ckpt.keys()):
            return ckpt, "raw_state_dict"

    raise ValueError("Could not identify state_dict container.")


def strip_module_prefix(state):
    cleaned = {}
    for k, v in state.items():
        nk = k[7:] if k.startswith("module.") else k
        cleaned[nk] = v
    return cleaned


def infer_architecture_signature(keys):
    keys = list(keys)

    if any(k.startswith("encoder.stem.") for k in keys) and any(k.startswith("encoder.backbone.") for k in keys):
        return "LEGACY_ENCODER_BACKBONE"

    if any(k.startswith("stem.") for k in keys) and any(k.startswith("layer1.") for k in keys):
        return "STEP21_FLAT_RESNET"

    if any(k.startswith("encoder.") for k in keys):
        return "OTHER_ENCODER_STYLE"

    return "UNKNOWN_STYLE"


def summarize_checkpoint(name: str, path: Path):
    require_exists(path, name)

    ckpt = torch.load(path, map_location="cpu")
    state, container_key = extract_state_dict(ckpt)
    state = strip_module_prefix(state)

    keys = list(state.keys())
    signature = infer_architecture_signature(keys)

    shape_items = []
    for k in keys:
        v = state[k]
        if hasattr(v, "shape"):
            shape_items.append((k, tuple(v.shape)))

    head_like = [
        (k, tuple(state[k].shape))
        for k in keys
        if any(token in k.lower() for token in ["head", "classifier", "fc", "linear"])
        and hasattr(state[k], "shape")
    ]

    first_token_counts = Counter(k.split(".")[0] for k in keys)

    first_keys = keys[:12]
    last_keys = keys[-12:]

    summary = {
        "name": name,
        "path": str(path),
        "state_container": container_key,
        "num_state_keys": len(keys),
        "architecture_signature": signature,
        "top_level_prefix_counts": dict(first_token_counts),
        "first_12_keys": first_keys,
        "last_12_keys": last_keys,
        "head_like_keys_and_shapes": head_like[:30],
    }

    return summary


summaries = []

print("=" * 170)
print("STEP 22B — CHECKPOINT ARCHITECTURE SIGNATURE AUDIT")
print("=" * 170)

for name, path in CHECKPOINTS.items():
    summary = summarize_checkpoint(name, path)
    summaries.append(summary)

    print()
    print("-" * 170)
    print(name)
    print("-" * 170)
    print(f"Path                    : {summary['path']}")
    print(f"State container         : {summary['state_container']}")
    print(f"Number of state keys    : {summary['num_state_keys']}")
    print(f"Architecture signature  : {summary['architecture_signature']}")
    print(f"Top-level prefixes      : {summary['top_level_prefix_counts']}")

    print("\nFirst 12 keys:")
    for k in summary["first_12_keys"]:
        print("  ", k)

    print("\nLast 12 keys:")
    for k in summary["last_12_keys"]:
        print("  ", k)

    print("\nHead/classifier-like keys and shapes:")
    if summary["head_like_keys_and_shapes"]:
        for k, shape in summary["head_like_keys_and_shapes"]:
            print(f"  {k:<60s} {shape}")
    else:
        print("  None detected.")

print()
print("=" * 170)
print("COMPACT ARCHITECTURE SUMMARY")
print("=" * 170)

for summary in summaries:
    print(f"{summary['name']:<28s} -> {summary['architecture_signature']}")

print("=" * 170)
print("DONE. Send this full output back.")
print("=" * 170)

06_publication_figures

In [ ]:

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

PROJECT_ROOT = Path(r"PROJECT_PATH")

STEP22_DIR = PROJECT_ROOT / "lead_masking_final" / "step22_final_evaluation_bootstrap_mi"
TABLE_DIR = STEP22_DIR / "tables"

OUT_DIR = PROJECT_ROOT / "lead_masking_final" / "step23_publication_ready_figures"
PNG_DIR = OUT_DIR / "png"
PDF_DIR = OUT_DIR / "pdf"
SVG_DIR = OUT_DIR / "svg"
REPORT_DIR = OUT_DIR / "reports"

for d in [OUT_DIR, PNG_DIR, PDF_DIR, SVG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LONG_CSV = TABLE_DIR / "step22_all_methods_all_conditions_metrics_long.csv"
MASTER_CSV = TABLE_DIR / "step22_consolidated_master_macro_f1_table.csv"
BOOTSTRAP_CSV = TABLE_DIR / "step22_pairwise_bootstrap_macro_f1.csv"
UNSEEN_TRANSFER_CSV = TABLE_DIR / "step22_unseen_transfer_level_assessment.csv"
MI_LONG_CSV = TABLE_DIR / "step22_ptbxl_mi_precision_recall_f1.csv"
MI_COMPARE_CSV = TABLE_DIR / "step22_ptbxl_mi_comparison_summary.csv"

CONSISTENCY_REPORT_TXT = REPORT_DIR / "step23_data_consistency_report.txt"
CONSISTENCY_REPORT_JSON = REPORT_DIR / "step23_data_consistency_report.json"

COLOR_STANDARD = "#AEB6BF"   # gray
COLOR_RANDOM   = "#7FA6D6"   # light blue
COLOR_CLINICAL = "#2F5D9B"   # strong blue
COLOR_WHITE    = "#FFFFFF"
COLOR_BLACK    = "#000000"
GRID_COLOR     = "#D9DDE3"

METHOD_COLORS = {
    "Standard": COLOR_STANDARD,
    "Random": COLOR_RANDOM,
    "Clinical": COLOR_CLINICAL,
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 1.2,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "font.weight": "bold",
    "font.size": 14,
    "axes.titlesize": 19,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "legend.frameon": True,
    "legend.edgecolor": "black",
    "legend.facecolor": "white",
    "grid.color": GRID_COLOR,
    "grid.linewidth": 0.8,
    "grid.alpha": 0.9,
    "savefig.bbox": "tight",
})

KNOWN_CONDITIONS = [
    "12lead_full",
    "6_limb",
    "6_precordial",
    "3_limb",
    "lead_II_only",
    "V5_only",
]

UNSEEN_CONDITIONS = [
    "lead_I_only_unseen",
    "V1_only_unseen",
    "I_II_unseen",
    "V1_V5_unseen",
]

ALL_CONDITIONS = KNOWN_CONDITIONS + UNSEEN_CONDITIONS

CONDITION_LABELS = {
    "12lead_full": "12-Lead",
    "6_limb": "6 Limb",
    "6_precordial": "6 Precordial",
    "3_limb": "3 Limb",
    "lead_II_only": "Lead II",
    "V5_only": "V5",
    "lead_I_only_unseen": "Lead I†",
    "V1_only_unseen": "V1†",
    "I_II_unseen": "I+II†",
    "V1_V5_unseen": "V1+V5†",
}

UNSEEN_LABELS = {
    "lead_I_only_unseen": "Lead I",
    "V1_only_unseen": "V1",
    "I_II_unseen": "I+II",
    "V1_V5_unseen": "V1+V5",
}

METHOD_ORDER = ["Standard", "Random", "Clinical"]
DATASET_ORDER = ["PTBXL", "Chapman"]

def require_exists(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")

def save_multi_format(fig, base_name: str) -> None:
    fig.savefig(PNG_DIR / f"{base_name}.png", dpi=600, facecolor="white")
    fig.savefig(PDF_DIR / f"{base_name}.pdf", dpi=600, facecolor="white")
    fig.savefig(SVG_DIR / f"{base_name}.svg", dpi=600, facecolor="white")

def add_value_labels(ax, bars, fmt="{:.3f}", fontsize=10, rotation=90, y_offset=0.01):
    for b in bars:
        h = b.get_height()
        ax.text(
            b.get_x() + b.get_width() / 2,
            h + y_offset,
            fmt.format(h),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            fontweight="bold",
            rotation=rotation,
            color="black"
        )

def style_axes(ax, ylim=(0, 1.0), ylabel="Macro-F1"):
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel, fontweight="bold")
    ax.grid(axis="y")
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.2)

def pivot_metric(long_df: pd.DataFrame, dataset: str, metric: str) -> pd.DataFrame:
    sub = long_df[long_df["dataset"] == dataset].copy()
    p = sub.pivot_table(
        index="lead_condition",
        columns="method",
        values=metric,
        aggfunc="first"
    )
    p = p.reindex(ALL_CONDITIONS)
    p = p.reindex(columns=METHOD_ORDER)
    return p

def build_heatmap_matrix(master_df: pd.DataFrame, dataset: str, col_name: str) -> Tuple[np.ndarray, List[str]]:
    sub = master_df[master_df["dataset"] == dataset].copy()
    sub["lead_condition"] = pd.Categorical(sub["lead_condition"], categories=ALL_CONDITIONS, ordered=True)
    sub = sub.sort_values("lead_condition")
    arr = sub[col_name].values.astype(float).reshape(1, -1)
    labels = [CONDITION_LABELS[c] for c in sub["lead_condition"].tolist()]
    return arr, labels

def make_blue_gray_white_cmap():
    return LinearSegmentedColormap.from_list(
        "blue_gray_white",
        ["#E2E6EB", "#FFFFFF", "#2F5D9B"]
    )

def load_tables():
    require_exists(LONG_CSV, "Step 22 long metrics CSV")
    require_exists(MASTER_CSV, "Step 22 master table CSV")
    require_exists(BOOTSTRAP_CSV, "Step 22 bootstrap CSV")
    require_exists(UNSEEN_TRANSFER_CSV, "Step 22 unseen transfer CSV")
    require_exists(MI_LONG_CSV, "Step 22 PTB-XL MI long CSV")
    require_exists(MI_COMPARE_CSV, "Step 22 PTB-XL MI comparison CSV")

    long_df = pd.read_csv(LONG_CSV)
    master_df = pd.read_csv(MASTER_CSV)
    bootstrap_df = pd.read_csv(BOOTSTRAP_CSV)
    unseen_df = pd.read_csv(UNSEEN_TRANSFER_CSV)
    mi_long_df = pd.read_csv(MI_LONG_CSV)
    mi_compare_df = pd.read_csv(MI_COMPARE_CSV)

    return long_df, master_df, bootstrap_df, unseen_df, mi_long_df, mi_compare_df

def run_triple_consistency_check(
    long_df: pd.DataFrame,
    master_df: pd.DataFrame,
    bootstrap_df: pd.DataFrame,
    unseen_df: pd.DataFrame,
    mi_long_df: pd.DataFrame,
    mi_compare_df: pd.DataFrame
):
    report_lines = []
    report = {
        "check_1_schema_and_inventory": {},
        "check_2_master_vs_long": {},
        "check_3_mi_and_delta_consistency": {},
        "overall_status": "PASS"
    }

    report_lines.append("=" * 120)
    report_lines.append("CHECK 1 — SCHEMA + INVENTORY")
    report_lines.append("=" * 120)

    required_long_cols = {"dataset", "method", "lead_condition", "macro_f1", "weighted_f1"}
    missing_long = sorted(required_long_cols - set(long_df.columns))
    if missing_long:
        raise RuntimeError(f"Missing required columns in long_df: {missing_long}")

    datasets_present = sorted(long_df["dataset"].dropna().unique().tolist())
    methods_present = sorted(long_df["method"].dropna().unique().tolist())
    conds_present = sorted(long_df["lead_condition"].dropna().unique().tolist())

    report["check_1_schema_and_inventory"]["datasets_present"] = datasets_present
    report["check_1_schema_and_inventory"]["methods_present"] = methods_present
    report["check_1_schema_and_inventory"]["conditions_present_count"] = len(conds_present)

    if set(DATASET_ORDER) - set(datasets_present):
        raise RuntimeError(f"Missing datasets in long_df: {set(DATASET_ORDER)-set(datasets_present)}")

    if set(METHOD_ORDER) - set(methods_present):
        raise RuntimeError(f"Missing methods in long_df: {set(METHOD_ORDER)-set(methods_present)}")

    missing_conditions = set(ALL_CONDITIONS) - set(conds_present)
    if missing_conditions:
        raise RuntimeError(f"Missing lead conditions in long_df: {missing_conditions}")

    expected_rows = len(DATASET_ORDER) * len(METHOD_ORDER) * len(ALL_CONDITIONS)
    actual_rows = len(long_df)
    if actual_rows != expected_rows:
        raise RuntimeError(f"Unexpected long_df row count. Expected {expected_rows}, got {actual_rows}")

    report_lines.append(f"Datasets present          : {datasets_present}")
    report_lines.append(f"Methods present           : {methods_present}")
    report_lines.append(f"Lead conditions present   : {len(conds_present)} / {len(ALL_CONDITIONS)}")
    report_lines.append(f"Expected long rows        : {expected_rows}")
    report_lines.append(f"Actual long rows          : {actual_rows}")
    report_lines.append("CHECK 1 STATUS            : PASS")
    report_lines.append("")

    report_lines.append("=" * 120)
    report_lines.append("CHECK 2 — MASTER TABLE VS LONG TABLE")
    report_lines.append("=" * 120)

    max_abs_diff_metric = 0.0
    max_abs_diff_delta = 0.0

    for dataset in DATASET_ORDER:
        p = pivot_metric(long_df, dataset=dataset, metric="macro_f1").reset_index()
        p.columns = ["lead_condition", "macro_f1_standard", "macro_f1_random", "macro_f1_clinical"]

        p["clinical_minus_standard"] = p["macro_f1_clinical"] - p["macro_f1_standard"]
        p["clinical_minus_random"] = p["macro_f1_clinical"] - p["macro_f1_random"]

        m = master_df[master_df["dataset"] == dataset].copy()
        m = m.sort_values("lead_condition").reset_index(drop=True)
        p = p.sort_values("lead_condition").reset_index(drop=True)

        merge = pd.merge(
            m,
            p,
            on="lead_condition",
            suffixes=("_master", "_recomputed")
        )

        metric_cols = [
            ("macro_f1_standard_master", "macro_f1_standard_recomputed"),
            ("macro_f1_random_master", "macro_f1_random_recomputed"),
            ("macro_f1_clinical_master", "macro_f1_clinical_recomputed"),
        ]
        delta_cols = [
            ("clinical_minus_standard_master", "clinical_minus_standard_recomputed"),
            ("clinical_minus_random_master", "clinical_minus_random_recomputed"),
        ]

        for a, b in metric_cols:
            diff = np.abs(merge[a] - merge[b]).max()
            max_abs_diff_metric = max(max_abs_diff_metric, float(diff))

        for a, b in delta_cols:
            diff = np.abs(merge[a] - merge[b]).max()
            max_abs_diff_delta = max(max_abs_diff_delta, float(diff))

    report["check_2_master_vs_long"]["max_abs_diff_metric"] = max_abs_diff_metric
    report["check_2_master_vs_long"]["max_abs_diff_delta"] = max_abs_diff_delta

    if max_abs_diff_metric > 1e-9 or max_abs_diff_delta > 1e-9:
        raise RuntimeError(
            f"Master table inconsistency detected. "
            f"Metric diff={max_abs_diff_metric}, Delta diff={max_abs_diff_delta}"
        )

    report_lines.append(f"Max abs diff (macro metrics) : {max_abs_diff_metric:.12f}")
    report_lines.append(f"Max abs diff (delta columns) : {max_abs_diff_delta:.12f}")
    report_lines.append("CHECK 2 STATUS               : PASS")
    report_lines.append("")

    report_lines.append("=" * 120)
    report_lines.append("CHECK 3 — MI SUMMARY + BOOTSTRAP INTEGRITY")
    report_lines.append("=" * 120)

    mi_pivot = mi_long_df.pivot_table(
        index="lead_condition",
        columns="method",
        values=["precision_MI", "recall_MI", "f1_MI"],
        aggfunc="first"
    )
    mi_pivot.columns = [f"{metric}_{method.lower()}" for metric, method in mi_pivot.columns]
    mi_pivot = mi_pivot.reset_index()

    mi_cmp = mi_compare_df.copy()
    merged_mi = pd.merge(mi_cmp, mi_pivot, on="lead_condition", how="inner", suffixes=("_cmp", "_pivot"))

    mi_check_cols = []
    for metric in ["precision_MI", "recall_MI", "f1_MI"]:
        for method in ["standard", "random", "clinical"]:
            cmp_col = f"{metric}_{method}"
            if cmp_col in merged_mi.columns and cmp_col in merged_mi.columns:
                mi_check_cols.append(cmp_col)

    mi_max_abs_diff = 0.0
    for col in mi_check_cols:
        pass

    for metric in ["precision_MI", "recall_MI", "f1_MI"]:
        for method in ["standard", "random", "clinical"]:
            col = f"{metric}_{method}"
            if col in mi_cmp.columns and col in mi_pivot.columns:
                a = mi_cmp.set_index("lead_condition")[col].sort_index()
                b = mi_pivot.set_index("lead_condition")[col].sort_index()
                common = a.index.intersection(b.index)
                diff = np.abs(a.loc[common].values - b.loc[common].values).max()
                mi_max_abs_diff = max(mi_max_abs_diff, float(diff))

    required_boot_cols = {
        "dataset",
        "lead_condition",
        "comparison",
        "observed_diff",
        "ci_lower",
        "ci_upper"
    }
    missing_boot_cols = sorted(required_boot_cols - set(bootstrap_df.columns))
    if missing_boot_cols:
        raise RuntimeError(f"Missing bootstrap columns: {missing_boot_cols}")

    if (bootstrap_df["ci_lower"] > bootstrap_df["ci_upper"]).any():
        raise RuntimeError("Bootstrap CI lower bound exceeds upper bound in some rows.")

    required_unseen_cols = {
        "dataset", "comparison", "positive_unseen_condition_count",
        "observed_average_unseen_gain", "ci_lower", "ci_upper", "transfer_level"
    }
    missing_unseen_cols = sorted(required_unseen_cols - set(unseen_df.columns))
    if missing_unseen_cols:
        raise RuntimeError(f"Missing unseen transfer columns: {missing_unseen_cols}")

    if unseen_df["positive_unseen_condition_count"].min() < 0 or unseen_df["positive_unseen_condition_count"].max() > 4:
        raise RuntimeError("Unseen positive count is outside valid range [0, 4].")

    report["check_3_mi_and_delta_consistency"]["mi_max_abs_diff"] = mi_max_abs_diff
    report["check_3_mi_and_delta_consistency"]["bootstrap_rows"] = int(len(bootstrap_df))
    report["check_3_mi_and_delta_consistency"]["unseen_rows"] = int(len(unseen_df))

    if mi_max_abs_diff > 1e-9:
        raise RuntimeError(f"MI summary inconsistency detected. Max diff = {mi_max_abs_diff}")

    report_lines.append(f"Max abs diff (MI tables)      : {mi_max_abs_diff:.12f}")
    report_lines.append(f"Bootstrap row count           : {len(bootstrap_df)}")
    report_lines.append(f"Unseen transfer row count     : {len(unseen_df)}")
    report_lines.append("CHECK 3 STATUS                : PASS")
    report_lines.append("")

    report_lines.append("=" * 120)
    report_lines.append("OVERALL CONSISTENCY STATUS: PASS")
    report_lines.append("=" * 120)

    with open(CONSISTENCY_REPORT_TXT, "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    with open(CONSISTENCY_REPORT_JSON, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print("\n".join(report_lines))

def plot_known_macro_f1(long_df: pd.DataFrame, dataset: str, base_name: str):
    p = pivot_metric(long_df, dataset=dataset, metric="macro_f1").loc[KNOWN_CONDITIONS]
    x = np.arange(len(KNOWN_CONDITIONS))
    width = 0.24

    fig, ax = plt.subplots(figsize=(14, 7))

    bars1 = ax.bar(
        x - width, p["Standard"].values, width,
        color=COLOR_STANDARD, edgecolor="black", linewidth=0.8, label="Standard"
    )
    bars2 = ax.bar(
        x, p["Random"].values, width,
        color=COLOR_RANDOM, edgecolor="black", linewidth=0.8, label="Random"
    )
    bars3 = ax.bar(
        x + width, p["Clinical"].values, width,
        color=COLOR_CLINICAL, edgecolor="black", linewidth=0.8, label="Clinical"
    )

    ax.set_xticks(x)
    ax.set_xticklabels([CONDITION_LABELS[c] for c in KNOWN_CONDITIONS], rotation=20, ha="right", fontweight="bold")
    ax.set_title(f"{dataset}: Known Lead Conditions (Macro-F1)", pad=14)
    style_axes(ax, ylim=(0, 1.0), ylabel="Macro-F1")
    ax.legend(loc="upper right")

    add_value_labels(ax, bars1, fontsize=9, rotation=90, y_offset=0.008)
    add_value_labels(ax, bars2, fontsize=9, rotation=90, y_offset=0.008)
    add_value_labels(ax, bars3, fontsize=9, rotation=90, y_offset=0.008)

    save_multi_format(fig, base_name)
    plt.close(fig)

def plot_unseen_macro_f1(long_df: pd.DataFrame, dataset: str, base_name: str):
    p = pivot_metric(long_df, dataset=dataset, metric="macro_f1").loc[UNSEEN_CONDITIONS]
    x = np.arange(len(UNSEEN_CONDITIONS))
    width = 0.24

    fig, ax = plt.subplots(figsize=(12, 7))

    bars1 = ax.bar(
        x - width, p["Standard"].values, width,
        color=COLOR_STANDARD, edgecolor="black", linewidth=0.8, label="Standard"
    )
    bars2 = ax.bar(
        x, p["Random"].values, width,
        color=COLOR_RANDOM, edgecolor="black", linewidth=0.8, label="Random"
    )
    bars3 = ax.bar(
        x + width, p["Clinical"].values, width,
        color=COLOR_CLINICAL, edgecolor="black", linewidth=0.8, label="Clinical"
    )

    ax.set_xticks(x)
    ax.set_xticklabels([UNSEEN_LABELS[c] for c in UNSEEN_CONDITIONS], rotation=0, fontweight="bold")
    ax.set_title(f"{dataset}: Unseen Lead-Subset Evaluation (Macro-F1)", pad=14)
    style_axes(ax, ylim=(0, 1.0), ylabel="Macro-F1")
    ax.legend(loc="upper right")

    add_value_labels(ax, bars1, fontsize=9, rotation=90, y_offset=0.008)
    add_value_labels(ax, bars2, fontsize=9, rotation=90, y_offset=0.008)
    add_value_labels(ax, bars3, fontsize=9, rotation=90, y_offset=0.008)

    ax.text(
        0.99, 0.02,
        "† Unseen relative to clinical masking policy",
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=11, fontweight="bold",
        bbox=dict(facecolor="white", edgecolor="black", linewidth=0.8, pad=4)
    )

    save_multi_format(fig, base_name)
    plt.close(fig)

def plot_gain_heatmap(master_df: pd.DataFrame, dataset: str, base_name: str):
    sub = master_df[master_df["dataset"] == dataset].copy()
    sub["lead_condition"] = pd.Categorical(sub["lead_condition"], categories=ALL_CONDITIONS, ordered=True)
    sub = sub.sort_values("lead_condition")

    mat = np.vstack([
        sub["clinical_minus_standard"].values.astype(float),
        sub["clinical_minus_random"].values.astype(float),
    ])

    row_labels = ["Clinical - Standard", "Clinical - Random"]
    col_labels = [CONDITION_LABELS[c] for c in sub["lead_condition"].tolist()]

    vmax = max(abs(mat.min()), abs(mat.max()))
    cmap = make_blue_gray_white_cmap()

    fig, ax = plt.subplots(figsize=(16, 4.8))
    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=-vmax, vmax=vmax)

    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=30, ha="right", fontweight="bold")
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels, fontweight="bold")
    ax.set_title(f"{dataset}: Clinical Gain Across Lead Conditions", pad=14)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(
                j, i, f"{mat[i, j]:+.3f}",
                ha="center", va="center",
                fontsize=11, fontweight="bold", color="black"
            )

    ax.set_xticks(np.arange(-.5, mat.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, mat.shape[0], 1), minor=True)
    ax.grid(which="minor", color="black", linestyle="-", linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = fig.colorbar(im, ax=ax, shrink=0.95)
    cbar.set_label("Macro-F1 Gain", fontweight="bold")
    cbar.outline.set_edgecolor("black")
    cbar.outline.set_linewidth(1.0)

    save_multi_format(fig, base_name)
    plt.close(fig)

def plot_unseen_transfer_summary(unseen_df: pd.DataFrame, base_name: str):
    

    plot_df = unseen_df.copy()

    desired_order = [
        ("PTBXL", "Clinical - Standard"),
        ("PTBXL", "Clinical - Random"),
        ("Chapman", "Clinical - Standard"),
        ("Chapman", "Clinical - Random"),
    ]

    plot_df["pair"] = list(zip(plot_df["dataset"], plot_df["comparison"]))

    available_pairs = set(plot_df["pair"].tolist())
    missing_pairs = [pair for pair in desired_order if pair not in available_pairs]

    if missing_pairs:
        raise RuntimeError(
            f"Unseen-transfer plotting cannot find expected rows: {missing_pairs}\n"
            f"Available pairs: {sorted(available_pairs)}"
        )

    plot_df = plot_df.set_index("pair").loc[desired_order].reset_index(drop=False)

    labels = [
        f"{dataset}\n{comparison}"
        for dataset, comparison in plot_df["pair"]
    ]

    vals = plot_df["observed_average_unseen_gain"].values.astype(float)
    lo = plot_df["ci_lower"].values.astype(float)
    hi = plot_df["ci_upper"].values.astype(float)

    yerr = np.vstack([
        vals - lo,
        hi - vals
    ])

    colors = []
    for _, comparison in plot_df["pair"]:
        if comparison == "Clinical - Standard":
            colors.append(COLOR_CLINICAL)
        elif comparison == "Clinical - Random":
            colors.append(COLOR_RANDOM)
        else:
            colors.append(COLOR_STANDARD)

    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(14, 7))

    bars = ax.bar(
        x,
        vals,
        color=colors,
        edgecolor="black",
        linewidth=0.8,
        yerr=yerr,
        capsize=6,
        ecolor="black"
    )

    ax.axhline(0, color="black", linewidth=1.3)

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        rotation=15,
        ha="right",
        fontweight="bold"
    )

    ax.set_title(
        "Average Transfer Gain on Unseen Lead Subsets",
        pad=14
    )

    lower_ylim = min(-0.08, float(lo.min()) - 0.04)
    upper_ylim = max(0.30, float(hi.max()) + 0.05)

    style_axes(
        ax,
        ylim=(lower_ylim, upper_ylim),
        ylabel="Average Macro-F1 Gain"
    )

    for idx, b in enumerate(bars):
        value = vals[idx]
        level = str(plot_df.iloc[idx]["transfer_level"])

        if "Level A" in level:
            level_text = "Level A"
        elif "Level B" in level:
            level_text = "Level B"
        elif "Level C" in level:
            level_text = "Level C"
        else:
            level_text = level

        y_pos = value + 0.018 if value >= 0 else value - 0.025
        va = "bottom" if value >= 0 else "top"

        ax.text(
            b.get_x() + b.get_width() / 2,
            y_pos,
            f"{value:+.3f}\n{level_text}",
            ha="center",
            va=va,
            fontsize=11,
            fontweight="bold",
            color="black",
            bbox=dict(
                facecolor="white",
                edgecolor="black",
                linewidth=0.6,
                pad=2.5
            )
        )

    legend_handles = [
        Patch(
            facecolor=COLOR_CLINICAL,
            edgecolor="black",
            linewidth=0.8,
            label="Clinical - Standard"
        ),
        Patch(
            facecolor=COLOR_RANDOM,
            edgecolor="black",
            linewidth=0.8,
            label="Clinical - Random"
        ),
    ]

    ax.legend(
        handles=legend_handles,
        loc="upper right"
    )

    save_multi_format(fig, base_name)
    plt.close(fig)

def plot_ptbxl_mi_metric(mi_compare_df: pd.DataFrame, metric: str, title_metric: str, base_name: str):
    df = mi_compare_df.copy()
    df["lead_condition"] = pd.Categorical(df["lead_condition"], categories=ALL_CONDITIONS, ordered=True)
    df = df.sort_values("lead_condition")

    xlabels = [CONDITION_LABELS[c] for c in df["lead_condition"].tolist()]
    x = np.arange(len(xlabels))
    width = 0.24

    vals_std = df[f"{metric}_standard"].values.astype(float)
    vals_rnd = df[f"{metric}_random"].values.astype(float)
    vals_cln = df[f"{metric}_clinical"].values.astype(float)

    fig, ax = plt.subplots(figsize=(16, 7))

    bars1 = ax.bar(
        x - width, vals_std, width,
        color=COLOR_STANDARD, edgecolor="black", linewidth=0.8, label="Standard"
    )
    bars2 = ax.bar(
        x, vals_rnd, width,
        color=COLOR_RANDOM, edgecolor="black", linewidth=0.8, label="Random"
    )
    bars3 = ax.bar(
        x + width, vals_cln, width,
        color=COLOR_CLINICAL, edgecolor="black", linewidth=0.8, label="Clinical"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=25, ha="right", fontweight="bold")
    ax.set_title(f"PTB-XL MI {title_metric} Across Lead Conditions", pad=14)
    style_axes(ax, ylim=(0, 1.0), ylabel=f"MI {title_metric}")
    ax.legend(loc="upper right")

    add_value_labels(ax, bars1, fontsize=8, rotation=90, y_offset=0.008)
    add_value_labels(ax, bars2, fontsize=8, rotation=90, y_offset=0.008)
    add_value_labels(ax, bars3, fontsize=8, rotation=90, y_offset=0.008)

    save_multi_format(fig, base_name)
    plt.close(fig)

def plot_known_weighted_f1(long_df: pd.DataFrame, dataset: str, base_name: str):
    p = pivot_metric(long_df, dataset=dataset, metric="weighted_f1").loc[KNOWN_CONDITIONS]
    x = np.arange(len(KNOWN_CONDITIONS))
    width = 0.24

    fig, ax = plt.subplots(figsize=(14, 7))

    bars1 = ax.bar(
        x - width, p["Standard"].values, width,
        color=COLOR_STANDARD, edgecolor="black", linewidth=0.8, label="Standard"
    )
    bars2 = ax.bar(
        x, p["Random"].values, width,
        color=COLOR_RANDOM, edgecolor="black", linewidth=0.8, label="Random"
    )
    bars3 = ax.bar(
        x + width, p["Clinical"].values, width,
        color=COLOR_CLINICAL, edgecolor="black", linewidth=0.8, label="Clinical"
    )

    ax.set_xticks(x)
    ax.set_xticklabels([CONDITION_LABELS[c] for c in KNOWN_CONDITIONS], rotation=20, ha="right", fontweight="bold")
    ax.set_title(f"{dataset}: Known Lead Conditions (Weighted-F1)", pad=14)
    style_axes(ax, ylim=(0, 1.0), ylabel="Weighted-F1")
    ax.legend(loc="upper right")

    add_value_labels(ax, bars1, fontsize=9, rotation=90, y_offset=0.008)
    add_value_labels(ax, bars2, fontsize=9, rotation=90, y_offset=0.008)
    add_value_labels(ax, bars3, fontsize=9, rotation=90, y_offset=0.008)

    save_multi_format(fig, base_name)
    plt.close(fig)

def generate_all_figures(
    long_df: pd.DataFrame,
    master_df: pd.DataFrame,
    bootstrap_df: pd.DataFrame,
    unseen_df: pd.DataFrame,
    mi_long_df: pd.DataFrame,
    mi_compare_df: pd.DataFrame
):
    plot_known_macro_f1(long_df, "PTBXL", "figure1_ptbxl_known_macro_f1")
    plot_known_macro_f1(long_df, "Chapman", "figure2_chapman_known_macro_f1")

    plot_unseen_macro_f1(long_df, "PTBXL", "figure3_ptbxl_unseen_macro_f1")
    plot_unseen_macro_f1(long_df, "Chapman", "figure4_chapman_unseen_macro_f1")

    plot_gain_heatmap(master_df, "PTBXL", "figure5_ptbxl_clinical_gain_heatmap")
    plot_gain_heatmap(master_df, "Chapman", "figure6_chapman_clinical_gain_heatmap")

    plot_unseen_transfer_summary(unseen_df, "figure7_unseen_transfer_summary")

    plot_ptbxl_mi_metric(mi_compare_df, "recall_MI", "Recall", "figure8_ptbxl_mi_recall")
    plot_ptbxl_mi_metric(mi_compare_df, "f1_MI", "F1-score", "figure9_ptbxl_mi_f1")

    plot_known_weighted_f1(long_df, "PTBXL", "supp_figure1_ptbxl_known_weighted_f1")
    plot_known_weighted_f1(long_df, "Chapman", "supp_figure2_chapman_known_weighted_f1")

def save_figure_index():
    lines = []
    lines.append("=" * 120)
    lines.append("STEP 23 — FIGURE INDEX")
    lines.append("=" * 120)
    lines.append(f"Output directory: {OUT_DIR}")
    lines.append("")
    lines.append("MAIN FIGURES")
    lines.append("  Figure 1  : PTB-XL known lead conditions (Macro-F1)")
    lines.append("  Figure 2  : Chapman known lead conditions (Macro-F1)")
    lines.append("  Figure 3  : PTB-XL unseen lead subsets (Macro-F1)")
    lines.append("  Figure 4  : Chapman unseen lead subsets (Macro-F1)")
    lines.append("  Figure 5  : PTB-XL clinical gain heatmap")
    lines.append("  Figure 6  : Chapman clinical gain heatmap")
    lines.append("  Figure 7  : Unseen transfer summary with CIs")
    lines.append("  Figure 8  : PTB-XL MI recall")
    lines.append("  Figure 9  : PTB-XL MI F1-score")
    lines.append("")
    lines.append("SUPPLEMENTARY FIGURES")
    lines.append("  Supp Figure 1 : PTB-XL known lead conditions (Weighted-F1)")
    lines.append("  Supp Figure 2 : Chapman known lead conditions (Weighted-F1)")
    lines.append("")
    lines.append("FORMATS SAVED")
    lines.append("  - PNG")
    lines.append("  - PDF")
    lines.append("  - SVG")
    lines.append("")
    lines.append("CONSISTENCY REPORTS")
    lines.append(f"  - {CONSISTENCY_REPORT_TXT}")
    lines.append(f"  - {CONSISTENCY_REPORT_JSON}")
    lines.append("=" * 120)

    index_path = REPORT_DIR / "step23_figure_index.txt"
    with open(index_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("\n".join(lines))

def main():
    print("=" * 120)
    print("STEP 23 — PUBLICATION-READY FIGURE GENERATION")
    print("=" * 120)
    print(f"Step 22 input dir : {STEP22_DIR}")
    print(f"Figure output dir : {OUT_DIR}")
    print("=" * 120)

    long_df, master_df, bootstrap_df, unseen_df, mi_long_df, mi_compare_df = load_tables()

    print("\nRunning triple consistency check...")
    run_triple_consistency_check(
        long_df=long_df,
        master_df=master_df,
        bootstrap_df=bootstrap_df,
        unseen_df=unseen_df,
        mi_long_df=mi_long_df,
        mi_compare_df=mi_compare_df
    )

    print("\nGenerating publication-ready figures...")
    generate_all_figures(
        long_df=long_df,
        master_df=master_df,
        bootstrap_df=bootstrap_df,
        unseen_df=unseen_df,
        mi_long_df=mi_long_df,
        mi_compare_df=mi_compare_df
    )

    save_figure_index()

    print("\nDONE. Publication-ready figures generated successfully.")
    print(f"PNG directory : {PNG_DIR}")
    print(f"PDF directory : {PDF_DIR}")
    print(f"SVG directory : {SVG_DIR}")
    print(f"Reports dir   : {REPORT_DIR}")

if __name__ == "__main__":
    main()

07_reviewer_supplementary_step24

In [ ]:

from __future__ import annotations

import os
import gc
import csv
import json
import math
import time
import random
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from tqdm.auto import tqdm



PROJECT_ROOT = Path(r"PROJECT_PATH")

PTBXL_DATA_DIR = PROJECT_ROOT / "Data"

STEP22_DIR = PROJECT_ROOT / "lead_masking_final" / "step22_final_evaluation_bootstrap_mi"
STEP22_TABLE_DIR = STEP22_DIR / "tables"

STEP24_OUT_DIR = PROJECT_ROOT / "lead_masking_final" / "step24_reviewer_requested_supplementary"
STEP24_OUT_DIR.mkdir(parents=True, exist_ok=True)

PMASK_SWEEP_DIR = STEP24_OUT_DIR / "step24a_pmask_sweep"
MULTISEED_DIR = STEP24_OUT_DIR / "step24b_multiseed_runs"
HYP_TABLE_DIR = STEP24_OUT_DIR / "step24c_hyp_perclass_table"
BOOT12_DIR = STEP24_OUT_DIR / "step24d_12lead_bootstrap_ci"

for d in [PMASK_SWEEP_DIR, MULTISEED_DIR, HYP_TABLE_DIR, BOOT12_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_24A_PMASK_SWEEP = False
RUN_24B_MULTI_SEED = True
RUN_24C_HYP_TABLE = False
RUN_24D_BOOTSTRAP_12LEAD = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

CLASS_NAMES = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES = len(CLASS_NAMES)
HYP_CLASS_INDEX = CLASS_NAMES.index("HYP")

LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
N_LEADS = 12
TARGET_LENGTH = 5000

LEAD_CONFIGS = {
    "12lead_full": {
        "display_name": "12-lead full",
        "lead_indices": list(range(12)),
        "known_unseen": "known",
    },
    "6_limb": {
        "display_name": "6 limb",
        "lead_indices": [0, 1, 2, 3, 4, 5],
        "known_unseen": "known",
    },
    "6_precordial": {
        "display_name": "6 precordial",
        "lead_indices": [6, 7, 8, 9, 10, 11],
        "known_unseen": "known",
    },
    "3_limb": {
        "display_name": "3 limb",
        "lead_indices": [0, 1, 2],
        "known_unseen": "known",
    },
    "lead_II_only": {
        "display_name": "Lead II only",
        "lead_indices": [1],
        "known_unseen": "known",
    },
    "V5_only": {
        "display_name": "V5 only",
        "lead_indices": [10],
        "known_unseen": "known",
    },
    "lead_I_only_unseen": {
        "display_name": "Lead I only†",
        "lead_indices": [0],
        "known_unseen": "unseen",
    },
    "V1_only_unseen": {
        "display_name": "V1 only†",
        "lead_indices": [6],
        "known_unseen": "unseen",
    },
    "I_II_unseen": {
        "display_name": "I+II†",
        "lead_indices": [0, 1],
        "known_unseen": "unseen",
    },
    "V1_V5_unseen": {
        "display_name": "V1+V5†",
        "lead_indices": [6, 10],
        "known_unseen": "unseen",
    },
}

KNOWN_REDUCED_KEYS = ["6_limb", "6_precordial", "3_limb", "lead_II_only", "V5_only"]
ALL_EVAL_KEYS = list(LEAD_CONFIGS.keys())

CLINICAL_SUBSETS = [
    [0, 1, 2, 3, 4, 5],       # 6 limb
    [6, 7, 8, 9, 10, 11],     # 6 precordial
    [0, 1, 2],                # 3 limb
    [1],                      # Lead II only
    [10],                     # V5 only
]

CARDINALITY_POOL = [6, 6, 3, 1, 1]

BATCH_SIZE = 128
MAX_EPOCHS = 70
EARLY_STOP_PATIENCE = 15
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
MIN_LR_RATIO = 0.02
DROPOUT = 0.10
GRAD_CLIP_NORM = 5.0

PMASK_SWEEP_EPOCHS = 50
PMASK_SWEEP_PATIENCE = 10
PMASK_VALUES = [0.30, 0.50, 0.60, 0.70, 0.90]

MULTISEED_VALUES = [41, 42, 43]
MULTISEED_POLICIES = ["Standard", "Random", "Clinical"]

DATASET_NAME = "PTBXL"



def require_exists(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


def write_json(path: Path, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def save_history_csv(history: List[Dict[str, Any]], path: Path) -> None:
    if not history:
        return
    pd.DataFrame(history).to_csv(path, index=False)


def set_warmup_cosine_lr(
    optimizer: torch.optim.Optimizer,
    epoch: int,
    total_epochs: int,
    base_lr: float,
    warmup_epochs: int,
    min_lr_ratio: float,
) -> float:
    if epoch < warmup_epochs:
        lr = base_lr * float(epoch + 1) / float(max(1, warmup_epochs))
    else:
        progress = (epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr = base_lr * (min_lr_ratio + (1.0 - min_lr_ratio) * cosine)

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr


def compute_class_weights(labels: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(labels.astype(int), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def softmax_np(logits: torch.Tensor) -> np.ndarray:
    return torch.softmax(logits.float(), dim=1).detach().cpu().numpy()


def metrics_from_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    p, r, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0,
    )

    for i, cls in enumerate(CLASS_NAMES):
        out[f"precision_{cls}"] = float(p[i])
        out[f"recall_{cls}"] = float(r[i])
        out[f"f1_{cls}"] = float(f1[i])
        out[f"support_{cls}"] = int(support[i])

    return out



class LeadMasker:
    def __init__(self, policy: str, p_mask: float):
        self.policy = policy
        self.p_mask = float(p_mask)

        if self.policy not in ["Standard", "Random", "Clinical"]:
            raise ValueError(f"Unknown policy: {policy}")

    def __call__(self, x: np.ndarray) -> Tuple[np.ndarray, Dict[str, Any]]:
        x = np.asarray(x, dtype=np.float32).copy()  # [T, 12]

        if self.policy == "Standard":
            return x, {
                "mask_applied": False,
                "policy": "Standard",
                "kept_leads": list(range(N_LEADS)),
                "kept_count": N_LEADS,
            }

        if random.random() > self.p_mask:
            return x, {
                "mask_applied": False,
                "policy": self.policy,
                "kept_leads": list(range(N_LEADS)),
                "kept_count": N_LEADS,
            }

        if self.policy == "Clinical":
            kept = sorted(random.choice(CLINICAL_SUBSETS))

        elif self.policy == "Random":
            k = int(random.choice(CARDINALITY_POOL))
            kept = sorted(random.sample(range(N_LEADS), k=k))

        else:
            raise ValueError(self.policy)

        zero_leads = [i for i in range(N_LEADS) if i not in kept]
        x[:, zero_leads] = 0.0

        return x, {
            "mask_applied": True,
            "policy": self.policy,
            "kept_leads": kept,
            "kept_count": len(kept),
        }


class ECGTrainDataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        split: str,
        policy: str,
        p_mask: float,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(data_dir / f"{split}_signals.npy", mmap_mode=mmap_mode)
        self.labels = np.load(data_dir / f"{split}_labels.npy").astype(np.int64)
        self.metadata_path = data_dir / f"{split}_metadata.csv"
        self.metadata = pd.read_csv(self.metadata_path) if self.metadata_path.exists() else pd.DataFrame()
        self.masker = LeadMasker(policy=policy, p_mask=p_mask)

        self._validate(split)

    def _validate(self, split: str) -> None:
        if self.signals.ndim != 3:
            raise ValueError(f"{split} signals must be 3D, got {self.signals.shape}")
        if self.signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
            raise ValueError(
                f"Expected {split} signals shape (N,{TARGET_LENGTH},{N_LEADS}), got {self.signals.shape}"
            )
        if len(self.signals) != len(self.labels):
            raise ValueError(f"{split}: signals/labels length mismatch")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32)
        y = int(self.labels[idx])

        x, meta = self.masker(x)

        x_t = torch.from_numpy(x.T.copy()).float()

        return {
            "signal": x_t,
            "label": torch.tensor(y, dtype=torch.long),
            "mask_meta": meta,
            "record_idx": int(idx),
        }


class ECGFixedLeadDataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        split: str,
        keep_leads: Optional[List[int]] = None,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(data_dir / f"{split}_signals.npy", mmap_mode=mmap_mode)
        self.labels = np.load(data_dir / f"{split}_labels.npy").astype(np.int64)
        self.keep_leads = list(range(N_LEADS)) if keep_leads is None else sorted(map(int, keep_leads))
        self.zero_leads = [i for i in range(N_LEADS) if i not in self.keep_leads]
        self.metadata_path = data_dir / f"{split}_metadata.csv"
        self.metadata = pd.read_csv(self.metadata_path) if self.metadata_path.exists() else pd.DataFrame()

        self._validate(split)

    def _validate(self, split: str) -> None:
        if self.signals.ndim != 3:
            raise ValueError(f"{split} signals must be 3D, got {self.signals.shape}")
        if self.signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
            raise ValueError(
                f"Expected {split} signals shape (N,{TARGET_LENGTH},{N_LEADS}), got {self.signals.shape}"
            )
        if len(self.signals) != len(self.labels):
            raise ValueError(f"{split}: signals/labels length mismatch")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32).copy()
        y = int(self.labels[idx])

        if self.zero_leads:
            x[:, self.zero_leads] = 0.0

        x_t = torch.from_numpy(x.T.copy()).float()

        return {
            "signal": x_t,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(idx),
        }


def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    out = {
        "signal": torch.stack([b["signal"] for b in batch], dim=0),
        "label": torch.stack([b["label"] for b in batch], dim=0),
        "record_idx": torch.tensor([b["record_idx"] for b in batch], dtype=torch.long),
    }
    if "mask_meta" in batch[0]:
        out["mask_meta"] = [b["mask_meta"] for b in batch]
    return out


def make_loader(
    ds: Dataset,
    batch_size: int,
    shuffle: bool,
    drop_last: bool = False,
) -> DataLoader:
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=drop_last,
        collate_fn=collate_fn,
    )



class BasicBlock1D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        kernel_size: int = 7,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        padding = kernel_size // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            bias=False,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.drop(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(identity)

        out = out + identity
        out = self.relu(out)
        return out


class ResNet1DEncoder(nn.Module):
    def __init__(self, input_channels: int = 12, base_filters: int = 64, embedding_dim: int = 512):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(input_channels, base_filters, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )

        self.backbone = nn.Sequential(
            self._make_stage(base_filters, base_filters, n_blocks=2, first_stride=1),
            self._make_stage(base_filters, base_filters * 2, n_blocks=2, first_stride=2),
            self._make_stage(base_filters * 2, base_filters * 4, n_blocks=2, first_stride=2),
            self._make_stage(base_filters * 4, base_filters * 8, n_blocks=2, first_stride=2),
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base_filters * 8, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=DROPOUT),
        )

        self.embedding_dim = embedding_dim
        self._init_weights()

    def _make_stage(
        self,
        in_channels: int,
        out_channels: int,
        n_blocks: int,
        first_stride: int,
    ) -> nn.Sequential:
        blocks = [
            BasicBlock1D(
                in_channels=in_channels,
                out_channels=out_channels,
                stride=first_stride,
                kernel_size=7,
                dropout=DROPOUT,
            )
        ]
        for _ in range(1, n_blocks):
            blocks.append(
                BasicBlock1D(
                    in_channels=out_channels,
                    out_channels=out_channels,
                    stride=1,
                    kernel_size=7,
                    dropout=DROPOUT,
                )
            )
        return nn.Sequential(*blocks)

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.backbone(x)
        x = self.global_pool(x)
        x = self.embedding_head(x)
        return x


class ECGClassifier(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.encoder = ResNet1DEncoder(input_channels=N_LEADS, base_filters=64, embedding_dim=512)
        self.classifier = nn.Linear(self.encoder.embedding_dim, num_classes)

        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        return self.classifier(z)



def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: GradScaler,
) -> Dict[str, Any]:
    model.train()

    total_loss = 0.0
    total_n = 0
    y_true_all = []
    y_pred_all = []
    masked_count = 0
    unmasked_count = 0

    for batch in tqdm(loader, desc="train", leave=False):
        x = batch["signal"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        if "mask_meta" in batch:
            for m in batch["mask_meta"]:
                if m["mask_applied"]:
                    masked_count += 1
                else:
                    unmasked_count += 1

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss: {loss.item()}")

        scaler.scale(loss).backward()

        if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        scaler.step(optimizer)
        scaler.update()

        pred = torch.argmax(logits.detach(), dim=1)

        bs = y.numel()
        total_loss += float(loss.item()) * bs
        total_n += bs

        y_true_all.extend(y.detach().cpu().numpy().tolist())
        y_pred_all.extend(pred.detach().cpu().numpy().tolist())

    y_true = np.asarray(y_true_all, dtype=np.int64)
    y_pred = np.asarray(y_pred_all, dtype=np.int64)

    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_n)
    metrics["masked_count"] = int(masked_count)
    metrics["unmasked_count"] = int(unmasked_count)
    metrics["observed_mask_rate"] = float(masked_count / max(1, masked_count + unmasked_count))

    return metrics


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    criterion: Optional[nn.Module] = None,
    save_probs: bool = False,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    model.eval()

    total_loss = 0.0
    total_n = 0

    y_true_all = []
    y_pred_all = []
    record_idx_all = []
    prob_rows = []

    for batch in tqdm(loader, desc="eval", leave=False):
        x = batch["signal"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        with autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y) if criterion is not None else None

        probs = torch.softmax(logits.float(), dim=1)
        pred = torch.argmax(probs, dim=1)

        bs = y.numel()
        if loss is not None:
            total_loss += float(loss.item()) * bs
        total_n += bs

        y_true_all.extend(y.detach().cpu().numpy().tolist())
        y_pred_all.extend(pred.detach().cpu().numpy().tolist())
        record_idx_all.extend(batch["record_idx"].cpu().numpy().tolist())

        if save_probs:
            prob_rows.append(probs.detach().cpu().numpy())

    y_true = np.asarray(y_true_all, dtype=np.int64)
    y_pred = np.asarray(y_pred_all, dtype=np.int64)

    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_n) if criterion is not None else np.nan

    pred_df = pd.DataFrame({
        "record_idx": record_idx_all,
        "y_true": y_true,
        "y_true_name": [CLASS_NAMES[i] for i in y_true],
        "y_pred": y_pred,
        "y_pred_name": [CLASS_NAMES[i] for i in y_pred],
    })

    if save_probs and prob_rows:
        probs_all = np.concatenate(prob_rows, axis=0)
        for i, cls in enumerate(CLASS_NAMES):
            pred_df[f"prob_{cls}"] = probs_all[:, i]

    return metrics, pred_df


def train_variant(
    data_dir: Path,
    output_dir: Path,
    policy: str,
    p_mask: float,
    seed: int,
    max_epochs: int,
    patience: int,
    eval_val_known_reduced_each_epoch: bool = False,
) -> Dict[str, Any]:
    seed_everything(seed)
    output_dir.mkdir(parents=True, exist_ok=True)

    train_labels = np.load(data_dir / "train_labels.npy").astype(np.int64)
    class_weights = compute_class_weights(train_labels, NUM_CLASSES).to(DEVICE)

    train_ds = ECGTrainDataset(data_dir=data_dir, split="train", policy=policy, p_mask=p_mask)
    val_full_ds = ECGFixedLeadDataset(data_dir=data_dir, split="val", keep_leads=list(range(N_LEADS)))

    train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_full_loader = make_loader(val_full_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(device="cuda", enabled=USE_AMP)

    history = []
    best_val_macro = -1.0
    best_epoch = -1
    bad_epochs = 0

    best_path = output_dir / "best_model.pt"
    latest_path = output_dir / "latest_model.pt"

    start_time = time.time()

    for epoch in range(max_epochs):
        lr = set_warmup_cosine_lr(
            optimizer=optimizer,
            epoch=epoch,
            total_epochs=max_epochs,
            base_lr=BASE_LR,
            warmup_epochs=WARMUP_EPOCHS,
            min_lr_ratio=MIN_LR_RATIO,
        )

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
        )

        val_metrics, _ = evaluate_model(
            model=model,
            loader=val_full_loader,
            criterion=criterion,
            save_probs=False,
        )

        row = {
            "epoch": epoch + 1,
            "policy": policy,
            "p_mask": p_mask,
            "seed": seed,
            "lr": lr,
            "train_loss": train_metrics["loss"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_observed_mask_rate": train_metrics["observed_mask_rate"],
            "val_full_loss": val_metrics["loss"],
            "val_full_macro_f1": val_metrics["macro_f1"],
            "val_full_weighted_f1": val_metrics["weighted_f1"],
            "is_best": False,
        }

        if eval_val_known_reduced_each_epoch:
            val_condition_scores = []
            for key in KNOWN_REDUCED_KEYS:
                val_ds = ECGFixedLeadDataset(data_dir=data_dir, split="val", keep_leads=LEAD_CONFIGS[key]["lead_indices"])
                val_loader = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
                vm, _ = evaluate_model(model, val_loader, criterion=None, save_probs=False)
                row[f"val_{key}_macro_f1"] = vm["macro_f1"]
                val_condition_scores.append(vm["macro_f1"])

            row["val_known_reduced_mean_macro_f1"] = float(np.mean(val_condition_scores))
        else:
            row["val_known_reduced_mean_macro_f1"] = np.nan

        improved = val_metrics["macro_f1"] > best_val_macro
        if improved:
            best_val_macro = float(val_metrics["macro_f1"])
            best_epoch = epoch + 1
            bad_epochs = 0
            row["is_best"] = True

            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch + 1,
                "best_val_full_macro_f1": best_val_macro,
                "policy": policy,
                "p_mask": p_mask,
                "seed": seed,
                "class_names": CLASS_NAMES,
            }, best_path)
        else:
            bad_epochs += 1

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch + 1,
            "best_val_full_macro_f1": best_val_macro,
            "policy": policy,
            "p_mask": p_mask,
            "seed": seed,
            "class_names": CLASS_NAMES,
        }, latest_path)

        history.append(row)
        save_history_csv(history, output_dir / "history.csv")

        print(
            f"[{policy} | p={p_mask:.2f} | seed={seed}] "
            f"Epoch {epoch+1:03d}/{max_epochs} | "
            f"train macro={train_metrics['macro_f1']:.4f} | "
            f"val full macro={val_metrics['macro_f1']:.4f} | "
            f"best={best_val_macro:.4f} | "
            f"bad={bad_epochs}/{patience}"
        )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if bad_epochs >= patience:
            break

    elapsed_min = (time.time() - start_time) / 60.0

    best_ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state_dict"], strict=True)
    model.eval()

    val_rows = []
    for key in ["12lead_full"] + KNOWN_REDUCED_KEYS:
        val_ds = ECGFixedLeadDataset(data_dir=data_dir, split="val", keep_leads=LEAD_CONFIGS[key]["lead_indices"])
        val_loader = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
        vm, _ = evaluate_model(model, val_loader, criterion=None, save_probs=False)

        val_rows.append({
            "dataset": DATASET_NAME,
            "policy": policy,
            "p_mask": p_mask,
            "seed": seed,
            "lead_condition": key,
            "lead_display_name": LEAD_CONFIGS[key]["display_name"],
            "val_macro_f1": vm["macro_f1"],
            "val_weighted_f1": vm["weighted_f1"],
            "val_accuracy": vm["accuracy"],
            "val_balanced_accuracy": vm["balanced_accuracy"],
        })

    val_eval_df = pd.DataFrame(val_rows)
    val_eval_df.to_csv(output_dir / "val_condition_metrics.csv", index=False)

    summary = {
        "dataset": DATASET_NAME,
        "policy": policy,
        "p_mask": float(p_mask),
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_val_full_macro_f1": float(best_val_macro),
        "val_known_reduced_mean_macro_f1": float(
            val_eval_df[val_eval_df["lead_condition"].isin(KNOWN_REDUCED_KEYS)]["val_macro_f1"].mean()
        ),
        "val_all_known_mean_macro_f1": float(val_eval_df["val_macro_f1"].mean()),
        "elapsed_minutes": float(elapsed_min),
        "best_checkpoint": str(best_path),
        "history_csv": str(output_dir / "history.csv"),
        "val_condition_metrics_csv": str(output_dir / "val_condition_metrics.csv"),
    }

    write_json(output_dir / "summary.json", summary)
    return summary


def load_model_from_checkpoint(ckpt_path: Path) -> nn.Module:
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()
    return model


def evaluate_checkpoint_all_test_conditions(
    ckpt_path: Path,
    data_dir: Path,
    output_dir: Path,
    policy: str,
    p_mask: float,
    seed: int,
) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)
    model = load_model_from_checkpoint(ckpt_path)

    rows = []

    for key in ALL_EVAL_KEYS:
        ds = ECGFixedLeadDataset(data_dir=data_dir, split="test", keep_leads=LEAD_CONFIGS[key]["lead_indices"])
        loader = make_loader(ds, batch_size=BATCH_SIZE, shuffle=False)

        metrics, pred_df = evaluate_model(model, loader, criterion=None, save_probs=True)

        pred_df["dataset"] = DATASET_NAME
        pred_df["policy"] = policy
        pred_df["p_mask"] = p_mask
        pred_df["seed"] = seed
        pred_df["lead_condition"] = key

        pred_path = output_dir / f"predictions_{DATASET_NAME}_{policy}_p{p_mask:.2f}_seed{seed}_{key}.csv"
        pred_df.to_csv(pred_path, index=False)

        row = {
            "dataset": DATASET_NAME,
            "policy": policy,
            "p_mask": p_mask,
            "seed": seed,
            "lead_condition": key,
            "lead_display_name": LEAD_CONFIGS[key]["display_name"],
            "known_unseen": LEAD_CONFIGS[key]["known_unseen"],
            "prediction_csv": str(pred_path),
        }
        row.update(metrics)
        rows.append(row)

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_dir / f"test_metrics_{DATASET_NAME}_{policy}_p{p_mask:.2f}_seed{seed}.csv", index=False)
    return out_df



def run_step24a_pmask_sweep() -> None:
    print("=" * 120)
    print("STEP 24A — p_mask SWEEP")
    print("=" * 120)

    require_exists(PTBXL_DATA_DIR / "train_signals.npy", "PTB-XL train signals")
    require_exists(PTBXL_DATA_DIR / "val_signals.npy", "PTB-XL val signals")

    summaries = []

    for policy in ["Random", "Clinical"]:
        for p in PMASK_VALUES:
            run_dir = PMASK_SWEEP_DIR / f"{DATASET_NAME}_{policy}_p{p:.2f}_seed42"
            if (run_dir / "summary.json").exists():
                print(f"[SKIP] Existing summary found: {run_dir / 'summary.json'}")
                with open(run_dir / "summary.json", "r", encoding="utf-8") as f:
                    summaries.append(json.load(f))
                continue

            summary = train_variant(
                data_dir=PTBXL_DATA_DIR,
                output_dir=run_dir,
                policy=policy,
                p_mask=p,
                seed=42,
                max_epochs=PMASK_SWEEP_EPOCHS,
                patience=PMASK_SWEEP_PATIENCE,
                eval_val_known_reduced_each_epoch=False,
            )
            summaries.append(summary)

    summary_df = pd.DataFrame(summaries)

    s1 = summary_df[
        [
            "dataset",
            "policy",
            "p_mask",
            "seed",
            "best_epoch",
            "best_val_full_macro_f1",
            "val_known_reduced_mean_macro_f1",
            "val_all_known_mean_macro_f1",
            "elapsed_minutes",
        ]
    ].copy()

    s1 = s1.sort_values(["policy", "p_mask"]).reset_index(drop=True)
    s1["rank_within_policy_by_known_reduced"] = (
        s1.groupby("policy")["val_known_reduced_mean_macro_f1"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    best_by_policy = (
        s1.sort_values(["policy", "val_known_reduced_mean_macro_f1"], ascending=[True, False])
        .groupby("policy")
        .head(1)
        .reset_index(drop=True)
    )

    s1_path = PMASK_SWEEP_DIR / "Supplementary_Table_S1_pmask_sweep_validation_macro_f1.csv"
    best_path = PMASK_SWEEP_DIR / "Supplementary_Table_S1_best_pmask_by_policy.csv"

    s1.to_csv(s1_path, index=False)
    best_by_policy.to_csv(best_path, index=False)

    latex_path = PMASK_SWEEP_DIR / "Supplementary_Table_S1_pmask_sweep_validation_macro_f1.tex"
    with open(latex_path, "w", encoding="utf-8") as f:
        f.write(
            s1.to_latex(
                index=False,
                float_format="%.4f",
                caption=(
                    "Validation Macro-F1 sweep for the lead-masking probability "
                    "$p_{\\mathrm{mask}}$ on PTB-XL. Selection was performed on validation data only "
                    "before final test-set evaluation."
                ),
                label="tab:supp_pmask_sweep",
            )
        )

    print("Saved:")
    print(s1_path)
    print(best_path)
    print(latex_path)



def run_step24b_multiseed() -> None:
    print("=" * 120)
    print("STEP 24B — MULTI-SEED RUNS")
    print("=" * 120)

    require_exists(PTBXL_DATA_DIR / "train_signals.npy", "PTB-XL train signals")
    require_exists(PTBXL_DATA_DIR / "test_signals.npy", "PTB-XL test signals")

    run_summaries = []
    all_test_metrics = []

    for policy in MULTISEED_POLICIES:
        for seed in MULTISEED_VALUES:
            p_mask = 0.0 if policy == "Standard" else 0.60

            run_dir = MULTISEED_DIR / f"{DATASET_NAME}_{policy}_p{p_mask:.2f}_seed{seed}"

            if (run_dir / "summary.json").exists() and (run_dir / f"test_metrics_{DATASET_NAME}_{policy}_p{p_mask:.2f}_seed{seed}.csv").exists():
                print(f"[SKIP] Existing multiseed run: {run_dir}")
                with open(run_dir / "summary.json", "r", encoding="utf-8") as f:
                    summary = json.load(f)
                test_df = pd.read_csv(run_dir / f"test_metrics_{DATASET_NAME}_{policy}_p{p_mask:.2f}_seed{seed}.csv")
            else:
                summary = train_variant(
                    data_dir=PTBXL_DATA_DIR,
                    output_dir=run_dir,
                    policy=policy,
                    p_mask=p_mask,
                    seed=seed,
                    max_epochs=MAX_EPOCHS,
                    patience=EARLY_STOP_PATIENCE,
                    eval_val_known_reduced_each_epoch=False,
                )

                test_df = evaluate_checkpoint_all_test_conditions(
                    ckpt_path=Path(summary["best_checkpoint"]),
                    data_dir=PTBXL_DATA_DIR,
                    output_dir=run_dir,
                    policy=policy,
                    p_mask=p_mask,
                    seed=seed,
                )

            run_summaries.append(summary)
            all_test_metrics.append(test_df)

    run_summary_df = pd.DataFrame(run_summaries)
    all_test_df = pd.concat(all_test_metrics, ignore_index=True)

    run_summary_path = MULTISEED_DIR / "multiseed_training_run_summaries.csv"
    all_test_path = MULTISEED_DIR / "multiseed_all_test_condition_metrics_long.csv"

    run_summary_df.to_csv(run_summary_path, index=False)
    all_test_df.to_csv(all_test_path, index=False)

    agg = (
        all_test_df.groupby(["dataset", "policy", "lead_condition", "lead_display_name", "known_unseen"])
        .agg(
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            weighted_f1_mean=("weighted_f1", "mean"),
            weighted_f1_std=("weighted_f1", "std"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_std=("balanced_accuracy", "std"),
            n_seeds=("seed", "nunique"),
        )
        .reset_index()
    )

    agg_path = MULTISEED_DIR / "Supplementary_Table_S2_multiseed_mean_std_by_policy_condition.csv"
    agg.to_csv(agg_path, index=False)

    def fmt_mean_std(mean, std):
        if pd.isna(std):
            return f"{mean:.4f}"
        return f"{mean:.4f} $\\pm$ {std:.4f}"

    table_rows = []
    for cond in ALL_EVAL_KEYS:
        row = {
            "lead_condition": cond,
            "lead_display_name": LEAD_CONFIGS[cond]["display_name"],
            "known_unseen": LEAD_CONFIGS[cond]["known_unseen"],
        }
        for policy in MULTISEED_POLICIES:
            sub = agg[(agg["lead_condition"] == cond) & (agg["policy"] == policy)]
            if len(sub) == 1:
                row[policy] = fmt_mean_std(float(sub.iloc[0]["macro_f1_mean"]), float(sub.iloc[0]["macro_f1_std"]))
            else:
                row[policy] = ""
        table_rows.append(row)

    table_df = pd.DataFrame(table_rows)
    table_path = MULTISEED_DIR / "Supplementary_Table_S2_multiseed_macro_f1_mean_std_pivot.csv"
    table_df.to_csv(table_path, index=False)

    latex_path = MULTISEED_DIR / "Supplementary_Table_S2_multiseed_macro_f1_mean_std.tex"
    with open(latex_path, "w", encoding="utf-8") as f:
        f.write(
            table_df.to_latex(
                index=False,
                escape=False,
                caption=(
                    "Multi-seed PTB-XL Macro-F1 sensitivity analysis. Values are mean $\\pm$ standard deviation "
                    "over three independent training seeds."
                ),
                label="tab:supp_multiseed",
            )
        )

    mean_pivot = agg.pivot_table(
        index="lead_condition",
        columns="policy",
        values="macro_f1_mean",
        aggfunc="first",
    )
    std_pivot = agg.pivot_table(
        index="lead_condition",
        columns="policy",
        values="macro_f1_std",
        aggfunc="first",
    )

    margin_rows = []
    for cond in KNOWN_REDUCED_KEYS:
        if cond in mean_pivot.index and {"Clinical", "Random"}.issubset(set(mean_pivot.columns)):
            margin_rows.append({
                "lead_condition": cond,
                "lead_display_name": LEAD_CONFIGS[cond]["display_name"],
                "clinical_mean_macro_f1": float(mean_pivot.loc[cond, "Clinical"]),
                "clinical_std_macro_f1": float(std_pivot.loc[cond, "Clinical"]),
                "random_mean_macro_f1": float(mean_pivot.loc[cond, "Random"]),
                "random_std_macro_f1": float(std_pivot.loc[cond, "Random"]),
                "clinical_minus_random_mean_macro_f1": float(mean_pivot.loc[cond, "Clinical"] - mean_pivot.loc[cond, "Random"]),
            })

    margin_df = pd.DataFrame(margin_rows)
    margin_path = MULTISEED_DIR / "Supplementary_Table_S3_multiseed_clinical_minus_random_known_reduced.csv"
    margin_df.to_csv(margin_path, index=False)

    print("Saved:")
    print(run_summary_path)
    print(all_test_path)
    print(agg_path)
    print(table_path)
    print(latex_path)
    print(margin_path)



def locate_prediction_files_for_hyp() -> List[Path]:
    
    candidate_roots = [
        STEP22_DIR,
        STEP24_OUT_DIR,
        PROJECT_ROOT / "lead_masking_final",
    ]

    files = []
    for root in candidate_roots:
        if root.exists():
            files.extend(list(root.rglob("*.csv")))

    likely = []
    for p in files:
        name = p.name.lower()
        if "prediction" in name or "predictions" in name:
            likely.append(p)

    return sorted(set(likely))


def infer_method_from_path_or_df(path: Path, df: pd.DataFrame) -> Optional[str]:
    for col in ["method", "policy", "model_key", "display_name"]:
        if col in df.columns:
            val = str(df[col].iloc[0])
            low = val.lower()
            if "standard" in low:
                return "Standard"
            if "random" in low:
                return "Random"
            if "clinical" in low:
                return "Clinical"

    lowpath = str(path).lower()
    if "standard" in lowpath:
        return "Standard"
    if "random" in lowpath:
        return "Random"
    if "clinical" in lowpath:
        return "Clinical"
    return None


def infer_condition_from_path_or_df(path: Path, df: pd.DataFrame) -> Optional[str]:
    for col in ["lead_condition", "lead_key", "condition"]:
        if col in df.columns:
            val = str(df[col].iloc[0])
            if val in LEAD_CONFIGS:
                return val

            aliases = {
                "12_lead_full": "12lead_full",
                "12lead": "12lead_full",
                "full": "12lead_full",
                "3_limb_I_II_III": "3_limb",
                "lead_II": "lead_II_only",
                "V5": "V5_only",
                "Lead I only": "lead_I_only_unseen",
                "V1 only": "V1_only_unseen",
                "I+II": "I_II_unseen",
                "V1+V5": "V1_V5_unseen",
            }
            if val in aliases:
                return aliases[val]

    low = path.name.lower()
    checks = [
        ("12lead_full", ["12lead_full", "12_lead_full", "12lead", "full"]),
        ("6_limb", ["6_limb", "6limb"]),
        ("6_precordial", ["6_precordial", "6precordial"]),
        ("3_limb", ["3_limb", "3_limb_i_ii_iii", "3limb"]),
        ("lead_II_only", ["lead_ii_only", "lead_ii"]),
        ("V5_only", ["v5_only", "v5"]),
        ("lead_I_only_unseen", ["lead_i_only_unseen", "lead_i_only"]),
        ("V1_only_unseen", ["v1_only_unseen", "v1_only"]),
        ("I_II_unseen", ["i_ii_unseen", "i+ii"]),
        ("V1_V5_unseen", ["v1_v5_unseen", "v1+v5"]),
    ]

    for key, pats in checks:
        for pat in pats:
            if pat in low:
                return key

    return None


def run_step24c_hyp_perclass_table() -> None:
    print("=" * 120)
    print("STEP 24C — PTB-XL HYP PER-CLASS F1 SUPPLEMENTARY TABLE")
    print("=" * 120)

    pred_files = locate_prediction_files_for_hyp()
    print(f"Candidate prediction CSVs found: {len(pred_files)}")

    rows = []

    for p in pred_files:
        try:
            df = pd.read_csv(p)
        except Exception:
            continue

        if "y_true" not in df.columns or "y_pred" not in df.columns:
            continue

        if "dataset" in df.columns:
            dataset_val = str(df["dataset"].iloc[0]).lower()
            if "ptb" not in dataset_val:
                continue

        method = infer_method_from_path_or_df(p, df)
        cond = infer_condition_from_path_or_df(p, df)

        if method is None or cond is None:
            continue

        y_true = df["y_true"].astype(int).to_numpy()
        y_pred = df["y_pred"].astype(int).to_numpy()

        if y_true.max() >= NUM_CLASSES or y_pred.max() >= NUM_CLASSES:
            continue

        pr, rc, f1, sup = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=np.arange(NUM_CLASSES),
            zero_division=0,
        )

        row = {
            "dataset": "PTB-XL",
            "method": method,
            "lead_condition": cond,
            "lead_display_name": LEAD_CONFIGS[cond]["display_name"],
            "known_unseen": LEAD_CONFIGS[cond]["known_unseen"],
            "prediction_file": str(p),
        }

        for i, cls in enumerate(CLASS_NAMES):
            row[f"precision_{cls}"] = float(pr[i])
            row[f"recall_{cls}"] = float(rc[i])
            row[f"f1_{cls}"] = float(f1[i])
            row[f"support_{cls}"] = int(sup[i])

        rows.append(row)

    if not rows:
        raise RuntimeError(
            "No usable PTB-XL prediction CSVs found. "
            "Make sure Step 22 prediction files exist and include y_true/y_pred columns."
        )

    raw_df = pd.DataFrame(rows)

    raw_df["priority"] = raw_df["prediction_file"].apply(lambda s: 0 if "step22" in s.lower() else 1)
    raw_df = (
        raw_df.sort_values(["method", "lead_condition", "priority"])
        .drop_duplicates(["method", "lead_condition"], keep="first")
        .drop(columns=["priority"])
        .reset_index(drop=True)
    )

    expected_methods = {"Standard", "Random", "Clinical"}
    expected_conditions = set(LEAD_CONFIGS.keys())

    missing = []
    for m in expected_methods:
        for c in expected_conditions:
            if not ((raw_df["method"] == m) & (raw_df["lead_condition"] == c)).any():
                missing.append((m, c))

    if missing:
        print("WARNING: Missing method-condition prediction files:")
        for item in missing[:30]:
            print("  ", item)
        if len(missing) > 30:
            print(f"  ... and {len(missing)-30} more")

    raw_path = HYP_TABLE_DIR / "Supplementary_Table_S4_PTBXL_all_perclass_metrics_from_predictions.csv"
    raw_df.to_csv(raw_path, index=False)

    hyp_df = raw_df[
        [
            "dataset",
            "method",
            "lead_condition",
            "lead_display_name",
            "known_unseen",
            "precision_HYP",
            "recall_HYP",
            "f1_HYP",
            "support_HYP",
        ]
    ].copy()

    order_map = {k: i for i, k in enumerate(ALL_EVAL_KEYS)}
    method_map = {"Standard": 0, "Random": 1, "Clinical": 2}
    hyp_df["condition_order"] = hyp_df["lead_condition"].map(order_map)
    hyp_df["method_order"] = hyp_df["method"].map(method_map)
    hyp_df = hyp_df.sort_values(["condition_order", "method_order"]).drop(columns=["condition_order", "method_order"])

    hyp_path = HYP_TABLE_DIR / "Supplementary_Table_S4_PTBXL_HYP_precision_recall_f1_all_conditions.csv"
    hyp_df.to_csv(hyp_path, index=False)

    pivot_rows = []
    for cond in ALL_EVAL_KEYS:
        row = {
            "lead_condition": cond,
            "lead_display_name": LEAD_CONFIGS[cond]["display_name"],
            "known_unseen": LEAD_CONFIGS[cond]["known_unseen"],
        }
        for m in ["Standard", "Random", "Clinical"]:
            sub = hyp_df[(hyp_df["method"] == m) & (hyp_df["lead_condition"] == cond)]
            if len(sub) == 1:
                row[f"{m}_HYP_F1"] = float(sub.iloc[0]["f1_HYP"])
                row[f"{m}_HYP_Precision"] = float(sub.iloc[0]["precision_HYP"])
                row[f"{m}_HYP_Recall"] = float(sub.iloc[0]["recall_HYP"])
            else:
                row[f"{m}_HYP_F1"] = np.nan
                row[f"{m}_HYP_Precision"] = np.nan
                row[f"{m}_HYP_Recall"] = np.nan
        pivot_rows.append(row)

    pivot_df = pd.DataFrame(pivot_rows)
    pivot_path = HYP_TABLE_DIR / "Supplementary_Table_S4_PTBXL_HYP_F1_pivot.csv"
    pivot_df.to_csv(pivot_path, index=False)

    latex_path = HYP_TABLE_DIR / "Supplementary_Table_S4_PTBXL_HYP_F1_pivot.tex"
    latex_cols = [
        "lead_display_name",
        "known_unseen",
        "Standard_HYP_F1",
        "Random_HYP_F1",
        "Clinical_HYP_F1",
    ]

    with open(latex_path, "w", encoding="utf-8") as f:
        f.write(
            pivot_df[latex_cols].to_latex(
                index=False,
                float_format="%.4f",
                caption=(
                    "PTB-XL HYP-class F1-score across all lead conditions and training policies. "
                    "HYP has 69 test examples; therefore, class-specific conclusions should be interpreted cautiously."
                ),
                label="tab:supp_hyp_f1",
            )
        )

    print("Saved:")
    print(raw_path)
    print(hyp_path)
    print(pivot_path)
    print(latex_path)



def find_bootstrap_csv() -> Path:
    candidates = [
        STEP22_TABLE_DIR / "step22_pairwise_bootstrap_macro_f1.csv",
        STEP22_TABLE_DIR / "pairwise_bootstrap_macro_f1.csv",
        STEP22_DIR / "step22_pairwise_bootstrap_macro_f1.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    found = list(STEP22_DIR.rglob("*bootstrap*macro*f1*.csv"))
    if found:
        return found[0]

    raise FileNotFoundError("Could not locate Step 22 bootstrap Macro-F1 CSV.")


def normalize_bootstrap_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "observed_gain" not in df.columns and "observed_diff" in df.columns:
        df["observed_gain"] = df["observed_diff"]

    if "comparison" in df.columns:
        df["comparison"] = df["comparison"].astype(str)
        df["comparison"] = df["comparison"].str.replace("_vs_", " - ", regex=False)
        df["comparison"] = df["comparison"].str.replace("Clinical_vs_Standard", "Clinical - Standard", regex=False)
        df["comparison"] = df["comparison"].str.replace("Clinical_vs_Random", "Clinical - Random", regex=False)
        df["comparison"] = df["comparison"].str.replace("Random_vs_Standard", "Random - Standard", regex=False)

    if "lead_condition" in df.columns:
        df["lead_condition"] = df["lead_condition"].astype(str)
        aliases = {
            "12_lead_full": "12lead_full",
            "12-lead full": "12lead_full",
            "12lead": "12lead_full",
            "full": "12lead_full",
        }
        df["lead_condition"] = df["lead_condition"].replace(aliases)

    return df


def run_step24d_bootstrap_12lead_ci() -> None:
    print("=" * 120)
    print("STEP 24D — 12-LEAD BOOTSTRAP CI EXTRACTION")
    print("=" * 120)

    boot_csv = find_bootstrap_csv()
    boot_df = pd.read_csv(boot_csv)
    boot_df = normalize_bootstrap_columns(boot_df)

    required = {"dataset", "lead_condition", "comparison", "ci_lower", "ci_upper"}
    missing = required - set(boot_df.columns)
    if missing:
        raise RuntimeError(f"Bootstrap CSV missing required columns: {missing}")

    if "observed_gain" not in boot_df.columns:
        raise RuntimeError("Bootstrap CSV must include observed_gain or observed_diff.")

    twelve = boot_df[boot_df["lead_condition"].astype(str).isin(["12lead_full", "12_lead_full"])].copy()

    if len(twelve) == 0:
        raise RuntimeError(
            "No 12-lead bootstrap rows found. Check lead_condition naming in bootstrap file."
        )

    keep_cols = [
        c for c in [
            "dataset",
            "lead_condition",
            "comparison",
            "observed_gain",
            "bootstrap_mean",
            "ci_lower",
            "ci_upper",
            "p_gain_leq_0",
            "n_bootstrap",
        ] if c in twelve.columns
    ]

    twelve = twelve[keep_cols].sort_values(["dataset", "comparison"]).reset_index(drop=True)

    def interpret(row):
        lo, hi, gain = float(row["ci_lower"]), float(row["ci_upper"]), float(row["observed_gain"])
        if lo > 0:
            return "Significant positive difference"
        if hi < 0:
            return "Significant negative difference"
        return "CI overlaps zero"

    twelve["interpretation"] = twelve.apply(interpret, axis=1)

    out_csv = BOOT12_DIR / "Supplementary_Table_S5_12lead_bootstrap_CIs.csv"
    twelve.to_csv(out_csv, index=False)

    latex_path = BOOT12_DIR / "Supplementary_Table_S5_12lead_bootstrap_CIs.tex"
    with open(latex_path, "w", encoding="utf-8") as f:
        f.write(
            twelve.to_latex(
                index=False,
                float_format="%.4f",
                caption=(
                    "Paired bootstrap confidence intervals for 12-lead full-condition Macro-F1 comparisons. "
                    "These results test whether masking changes full-lead performance."
                ),
                label="tab:supp_12lead_bootstrap",
            )
        )

    print("Loaded bootstrap file:")
    print(boot_csv)
    print("Saved:")
    print(out_csv)
    print(latex_path)
    print(twelve)



def write_step24_manifest() -> None:
    manifest = {
        "step": "STEP 24 — reviewer-requested supplementary experiments",
        "project_root": str(PROJECT_ROOT),
        "ptbxl_data_dir": str(PTBXL_DATA_DIR),
        "step22_dir": str(STEP22_DIR),
        "output_dir": str(STEP24_OUT_DIR),
        "modules": {
            "24A_pmask_sweep": {
                "enabled": RUN_24A_PMASK_SWEEP,
                "p_values": PMASK_VALUES,
                "output_dir": str(PMASK_SWEEP_DIR),
            },
            "24B_multiseed": {
                "enabled": RUN_24B_MULTI_SEED,
                "seeds": MULTISEED_VALUES,
                "policies": MULTISEED_POLICIES,
                "output_dir": str(MULTISEED_DIR),
            },
            "24C_hyp_table": {
                "enabled": RUN_24C_HYP_TABLE,
                "output_dir": str(HYP_TABLE_DIR),
            },
            "24D_bootstrap_12lead": {
                "enabled": RUN_24D_BOOTSTRAP_12LEAD,
                "output_dir": str(BOOT12_DIR),
            },
        },
    }
    write_json(STEP24_OUT_DIR / "step24_manifest.json", manifest)


def main() -> None:
    print("=" * 120)
    print("STEP 24 — EAAI REVIEWER-REQUESTED SUPPLEMENTARY EXPERIMENTS")
    print("=" * 120)
    print(f"Device        : {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"Project root  : {PROJECT_ROOT}")
    print(f"Output dir    : {STEP24_OUT_DIR}")
    print("=" * 120)

    write_step24_manifest()

    if RUN_24C_HYP_TABLE:
        run_step24c_hyp_perclass_table()

    if RUN_24D_BOOTSTRAP_12LEAD:
        run_step24d_bootstrap_12lead_ci()

    if RUN_24A_PMASK_SWEEP:
        run_step24a_pmask_sweep()

    if RUN_24B_MULTI_SEED:
        run_step24b_multiseed()

    print("=" * 120)
    print("DONE — STEP 24 completed.")
    print("=" * 120)
    print("Output folders:")
    print(f"  24A p_mask sweep      : {PMASK_SWEEP_DIR}")
    print(f"  24B multi-seed        : {MULTISEED_DIR}")
    print(f"  24C HYP table         : {HYP_TABLE_DIR}")
    print(f"  24D 12-lead bootstrap : {BOOT12_DIR}")


if __name__ == "__main__":
    main()

08_hyp_table_fix

In [ ]:

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import pandas as pd



PROJECT_ROOT = Path(
    r"PROJECT_PATH"
)

STEP22_TABLE_DIR = (
    PROJECT_ROOT
    / "lead_masking_final"
    / "step22_final_evaluation_bootstrap_mi"
    / "tables"
)

STEP24_ROOT = (
    PROJECT_ROOT
    / "lead_masking_final"
    / "step24_reviewer_requested_supplementary"
)

OUT_DIR = STEP24_ROOT / "step24c_hyp_perclass_table_FIXED_complete_from_step22"
OUT_DIR.mkdir(parents=True, exist_ok=True)

STEP22_LONG_CSV = STEP22_TABLE_DIR / "step22_all_methods_all_conditions_metrics_long.csv"

MULTISEED_LONG_CSV = (
    STEP24_ROOT
    / "step24b_multiseed_runs"
    / "multiseed_all_test_condition_metrics_long.csv"
)

HYP_COMPLETE_CSV = OUT_DIR / "Supplementary_Table_S4_PTBXL_HYP_precision_recall_f1_COMPLETE.csv"
HYP_F1_PIVOT_CSV = OUT_DIR / "Supplementary_Table_S4_PTBXL_HYP_F1_pivot_COMPLETE.csv"
HYP_F1_PIVOT_TEX = OUT_DIR / "Supplementary_Table_S4_PTBXL_HYP_F1_pivot_COMPLETE.tex"
HYP_DETAIL_TEX = OUT_DIR / "Supplementary_Table_S4_PTBXL_HYP_precision_recall_f1_COMPLETE.tex"

HYP_MULTISEED_CSV = OUT_DIR / "Supplementary_Table_S4B_PTBXL_HYP_F1_multiseed_mean_std.csv"
HYP_MULTISEED_PIVOT_CSV = OUT_DIR / "Supplementary_Table_S4B_PTBXL_HYP_F1_multiseed_pivot.csv"
HYP_MULTISEED_TEX = OUT_DIR / "Supplementary_Table_S4B_PTBXL_HYP_F1_multiseed_pivot.tex"

AUDIT_TXT = OUT_DIR / "step24c_fix_hyp_table_audit_report.txt"
AUDIT_JSON = OUT_DIR / "step24c_fix_hyp_table_audit_report.json"



DATASET_NAME = "PTBXL"

METHOD_ORDER = ["Standard", "Random", "Clinical"]

LEAD_CONDITION_ORDER = [
    "12lead_full",
    "6_limb",
    "6_precordial",
    "3_limb",
    "lead_II_only",
    "V5_only",
    "lead_I_only_unseen",
    "V1_only_unseen",
    "I_II_unseen",
    "V1_V5_unseen",
]

LEAD_LABELS = {
    "12lead_full": "12-lead full",
    "6_limb": "6 limb",
    "6_precordial": "6 precordial",
    "3_limb": "3 limb",
    "lead_II_only": "Lead II only",
    "V5_only": "V5 only",
    "lead_I_only_unseen": "Lead I only$^{\\dagger}$",
    "V1_only_unseen": "V1 only$^{\\dagger}$",
    "I_II_unseen": "I+II$^{\\dagger}$",
    "V1_V5_unseen": "V1+V5$^{\\dagger}$",
}

CONDITION_GROUP = {
    "12lead_full": "known",
    "6_limb": "known",
    "6_precordial": "known",
    "3_limb": "known",
    "lead_II_only": "known",
    "V5_only": "known",
    "lead_I_only_unseen": "unseen",
    "V1_only_unseen": "unseen",
    "I_II_unseen": "unseen",
    "V1_V5_unseen": "unseen",
}



def require_exists(path: Path, desc: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {desc}: {path}")


def fmt4(x) -> str:
    if pd.isna(x):
        return ""
    return f"{float(x):.4f}"


def fmt_mean_std(mean, std) -> str:
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{float(mean):.4f}"
    return f"{float(mean):.4f} $\\pm$ {float(std):.4f}"


def normalize_method_name(x: str) -> str:
    s = str(x).strip()
    low = s.lower()
    if "standard" in low:
        return "Standard"
    if "random" in low:
        return "Random"
    if "clinical" in low:
        return "Clinical"
    return s


def add_order_columns(df: pd.DataFrame, method_col: str = "method") -> pd.DataFrame:
    out = df.copy()
    lead_order = {k: i for i, k in enumerate(LEAD_CONDITION_ORDER)}
    method_order = {k: i for i, k in enumerate(METHOD_ORDER)}
    out["_lead_order"] = out["lead_condition"].map(lead_order)
    out["_method_order"] = out[method_col].map(method_order)
    return out.sort_values(["_lead_order", "_method_order"]).drop(columns=["_lead_order", "_method_order"])



def load_step22_hyp_table() -> pd.DataFrame:
    require_exists(STEP22_LONG_CSV, "Step 22 long metrics CSV")

    df = pd.read_csv(STEP22_LONG_CSV)

    required_cols = {
        "dataset",
        "method",
        "lead_condition",
        "condition_group",
        "precision_HYP",
        "recall_HYP",
        "f1_HYP",
        "support_HYP",
    }

    missing_cols = sorted(required_cols - set(df.columns))
    if missing_cols:
        raise RuntimeError(f"Step 22 long metrics table missing required columns: {missing_cols}")

    ptb = df[df["dataset"].astype(str).str.upper().isin(["PTBXL", "PTB-XL"])].copy()
    ptb["dataset"] = "PTB-XL"
    ptb["method"] = ptb["method"].apply(normalize_method_name)

    ptb = ptb[
        ptb["method"].isin(METHOD_ORDER)
        & ptb["lead_condition"].isin(LEAD_CONDITION_ORDER)
    ].copy()

    return ptb


def audit_hyp_table(ptb: pd.DataFrame) -> Dict[str, Any]:
    audit: Dict[str, Any] = {}

    expected_pairs = {
        (m, c)
        for m in METHOD_ORDER
        for c in LEAD_CONDITION_ORDER
    }

    observed_pairs = set(zip(ptb["method"], ptb["lead_condition"]))
    missing_pairs = sorted(expected_pairs - observed_pairs)
    extra_pairs = sorted(observed_pairs - expected_pairs)

    audit["expected_rows"] = len(expected_pairs)
    audit["observed_rows"] = int(len(ptb))
    audit["missing_pairs"] = missing_pairs
    audit["extra_pairs"] = extra_pairs

    duplicate_counts = (
        ptb.groupby(["method", "lead_condition"])
        .size()
        .reset_index(name="n")
    )
    duplicates = duplicate_counts[duplicate_counts["n"] > 1]
    audit["duplicate_pairs"] = duplicates.to_dict(orient="records")

    support_values = sorted(ptb["support_HYP"].dropna().unique().tolist())
    audit["support_HYP_unique_values"] = [float(x) for x in support_values]

    metric_cols = ["precision_HYP", "recall_HYP", "f1_HYP"]
    range_issues = []
    for col in metric_cols:
        bad = ptb[(ptb[col] < 0) | (ptb[col] > 1) | (ptb[col].isna())]
        if len(bad) > 0:
            range_issues.append({
                "column": col,
                "bad_rows": int(len(bad)),
            })
    audit["range_issues"] = range_issues

    audit["all_30_method_condition_rows_present"] = (
        len(missing_pairs) == 0
        and len(extra_pairs) == 0
        and len(duplicates) == 0
        and len(ptb) == 30
    )

    audit["support_HYP_invariant"] = len(support_values) == 1
    audit["metric_ranges_valid"] = len(range_issues) == 0

    audit["overall_pass"] = (
        audit["all_30_method_condition_rows_present"]
        and audit["support_HYP_invariant"]
        and audit["metric_ranges_valid"]
    )

    return audit



def build_complete_hyp_table(ptb: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for cond in LEAD_CONDITION_ORDER:
        for method in METHOD_ORDER:
            sub = ptb[
                (ptb["lead_condition"] == cond)
                & (ptb["method"] == method)
            ]

            if len(sub) != 1:
                raise RuntimeError(
                    f"Expected exactly one row for method={method}, condition={cond}, found {len(sub)}"
                )

            r = sub.iloc[0]

            rows.append({
                "dataset": "PTB-XL",
                "method": method,
                "lead_condition": cond,
                "lead_display_name": LEAD_LABELS[cond],
                "condition_group": CONDITION_GROUP[cond],
                "precision_HYP": float(r["precision_HYP"]),
                "recall_HYP": float(r["recall_HYP"]),
                "f1_HYP": float(r["f1_HYP"]),
                "support_HYP": int(r["support_HYP"]),
            })

    out = pd.DataFrame(rows)
    out = add_order_columns(out, method_col="method")
    return out


def build_hyp_f1_pivot(hyp: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for cond in LEAD_CONDITION_ORDER:
        row = {
            "lead_condition": cond,
            "lead_display_name": LEAD_LABELS[cond],
            "condition_group": CONDITION_GROUP[cond],
        }

        for method in METHOD_ORDER:
            sub = hyp[
                (hyp["lead_condition"] == cond)
                & (hyp["method"] == method)
            ]
            if len(sub) == 1:
                row[f"{method}_HYP_F1"] = float(sub.iloc[0]["f1_HYP"])
                row[f"{method}_HYP_precision"] = float(sub.iloc[0]["precision_HYP"])
                row[f"{method}_HYP_recall"] = float(sub.iloc[0]["recall_HYP"])
            else:
                row[f"{method}_HYP_F1"] = np.nan
                row[f"{method}_HYP_precision"] = np.nan
                row[f"{method}_HYP_recall"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)



def build_multiseed_hyp_table_if_available() -> Dict[str, Any]:
    result = {
        "multiseed_file_exists": MULTISEED_LONG_CSV.exists(),
        "multiseed_outputs_created": False,
    }

    if not MULTISEED_LONG_CSV.exists():
        return result

    ms = pd.read_csv(MULTISEED_LONG_CSV)

    required_cols = {
        "dataset",
        "policy",
        "seed",
        "lead_condition",
        "f1_HYP",
        "precision_HYP",
        "recall_HYP",
        "support_HYP",
    }
    missing_cols = sorted(required_cols - set(ms.columns))
    if missing_cols:
        raise RuntimeError(f"Multi-seed long metrics table missing required columns: {missing_cols}")

    ms = ms[ms["dataset"].astype(str).str.upper().isin(["PTBXL", "PTB-XL"])].copy()
    ms["policy"] = ms["policy"].apply(normalize_method_name)
    ms = ms[
        ms["policy"].isin(METHOD_ORDER)
        & ms["lead_condition"].isin(LEAD_CONDITION_ORDER)
    ].copy()

    expected_rows = len(METHOD_ORDER) * len(LEAD_CONDITION_ORDER) * 3
    result["observed_multiseed_rows"] = int(len(ms))
    result["expected_multiseed_rows_if_3_seeds"] = int(expected_rows)

    agg = (
        ms.groupby(["policy", "lead_condition"])
        .agg(
            precision_HYP_mean=("precision_HYP", "mean"),
            precision_HYP_std=("precision_HYP", "std"),
            recall_HYP_mean=("recall_HYP", "mean"),
            recall_HYP_std=("recall_HYP", "std"),
            f1_HYP_mean=("f1_HYP", "mean"),
            f1_HYP_std=("f1_HYP", "std"),
            support_HYP_min=("support_HYP", "min"),
            support_HYP_max=("support_HYP", "max"),
            n_seeds=("seed", "nunique"),
        )
        .reset_index()
        .rename(columns={"policy": "method"})
    )

    agg["lead_display_name"] = agg["lead_condition"].map(LEAD_LABELS)
    agg["condition_group"] = agg["lead_condition"].map(CONDITION_GROUP)
    agg = add_order_columns(agg, method_col="method")

    agg.to_csv(HYP_MULTISEED_CSV, index=False)

    pivot_rows = []
    for cond in LEAD_CONDITION_ORDER:
        row = {
            "lead_condition": cond,
            "lead_display_name": LEAD_LABELS[cond],
            "condition_group": CONDITION_GROUP[cond],
        }
        for method in METHOD_ORDER:
            sub = agg[
                (agg["method"] == method)
                & (agg["lead_condition"] == cond)
            ]
            if len(sub) == 1:
                row[method] = fmt_mean_std(
                    sub.iloc[0]["f1_HYP_mean"],
                    sub.iloc[0]["f1_HYP_std"],
                )
            else:
                row[method] = ""
        pivot_rows.append(row)

    pivot = pd.DataFrame(pivot_rows)
    pivot.to_csv(HYP_MULTISEED_PIVOT_CSV, index=False)

    latex_df = pivot[["lead_display_name", "condition_group", "Standard", "Random", "Clinical"]].copy()
    latex_df.columns = [
        "Lead Condition",
        "Group",
        "Standard HYP F1",
        "Random HYP F1",
        "Clinical HYP F1",
    ]

    with open(HYP_MULTISEED_TEX, "w", encoding="utf-8") as f:
        f.write(
            latex_df.to_latex(
                index=False,
                escape=False,
                caption=(
                    "Multi-seed PTB-XL HYP-class F1-score across lead conditions. "
                    "Values are mean $\\pm$ standard deviation over three independent seeds."
                ),
                label="tab:supp_hyp_multiseed_f1",
            )
        )

    result["multiseed_outputs_created"] = True
    result["multiseed_hyp_csv"] = str(HYP_MULTISEED_CSV)
    result["multiseed_hyp_pivot_csv"] = str(HYP_MULTISEED_PIVOT_CSV)
    result["multiseed_hyp_tex"] = str(HYP_MULTISEED_TEX)

    return result



def export_main_hyp_latex(hyp: pd.DataFrame, pivot: pd.DataFrame) -> None:
    detail = hyp.copy()
    detail = detail[
        [
            "lead_display_name",
            "condition_group",
            "method",
            "precision_HYP",
            "recall_HYP",
            "f1_HYP",
            "support_HYP",
        ]
    ].copy()

    detail.columns = [
        "Lead Condition",
        "Group",
        "Policy",
        "HYP Precision",
        "HYP Recall",
        "HYP F1",
        "HYP Support",
    ]

    with open(HYP_DETAIL_TEX, "w", encoding="utf-8") as f:
        f.write(
            detail.to_latex(
                index=False,
                escape=False,
                float_format="%.4f",
                caption=(
                    "Complete PTB-XL HYP-class precision, recall, and F1-score across all lead "
                    "conditions and training policies. The HYP class has 69 test samples in all "
                    "conditions because the same test records are evaluated under different lead masks."
                ),
                label="tab:supp_hyp_precision_recall_f1_complete",
            )
        )

    compact = pivot[
        [
            "lead_display_name",
            "condition_group",
            "Standard_HYP_F1",
            "Random_HYP_F1",
            "Clinical_HYP_F1",
        ]
    ].copy()

    compact.columns = [
        "Lead Condition",
        "Group",
        "Standard HYP F1",
        "Random HYP F1",
        "Clinical HYP F1",
    ]

    with open(HYP_F1_PIVOT_TEX, "w", encoding="utf-8") as f:
        f.write(
            compact.to_latex(
                index=False,
                escape=False,
                float_format="%.4f",
                caption=(
                    "Complete PTB-XL HYP-class F1-score across all lead conditions. "
                    "The symbol $^{\\dagger}$ denotes unseen lead configurations."
                ),
                label="tab:supp_hyp_f1_complete",
            )
        )



def save_audit_report(audit: Dict[str, Any], multiseed_result: Dict[str, Any]) -> None:
    report = {
        "step": "STEP 24C-FIX",
        "purpose": "Complete PTB-XL HYP supplementary table using Step 22 long metrics instead of prediction-file discovery.",
        "input_step22_long_csv": str(STEP22_LONG_CSV),
        "output_dir": str(OUT_DIR),
        "audit": audit,
        "multiseed": multiseed_result,
        "outputs": {
            "hyp_complete_csv": str(HYP_COMPLETE_CSV),
            "hyp_f1_pivot_csv": str(HYP_F1_PIVOT_CSV),
            "hyp_detail_tex": str(HYP_DETAIL_TEX),
            "hyp_f1_pivot_tex": str(HYP_F1_PIVOT_TEX),
            "audit_txt": str(AUDIT_TXT),
            "audit_json": str(AUDIT_JSON),
        },
    }

    with open(AUDIT_JSON, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    lines = []
    lines.append("=" * 120)
    lines.append("STEP 24C-FIX — COMPLETE PTB-XL HYP SUPPLEMENTARY TABLE AUDIT")
    lines.append("=" * 120)
    lines.append(f"Input Step 22 long CSV : {STEP22_LONG_CSV}")
    lines.append(f"Output directory       : {OUT_DIR}")
    lines.append("")
    lines.append("COMPLETENESS CHECK")
    lines.append(f"  Expected method-condition rows : {audit['expected_rows']}")
    lines.append(f"  Observed method-condition rows : {audit['observed_rows']}")
    lines.append(f"  Missing pairs                  : {audit['missing_pairs']}")
    lines.append(f"  Extra pairs                    : {audit['extra_pairs']}")
    lines.append(f"  Duplicate pairs                : {audit['duplicate_pairs']}")
    lines.append(f"  All 30 rows present            : {audit['all_30_method_condition_rows_present']}")
    lines.append("")
    lines.append("HYP SUPPORT CHECK")
    lines.append(f"  Unique HYP support values      : {audit['support_HYP_unique_values']}")
    lines.append(f"  HYP support invariant          : {audit['support_HYP_invariant']}")
    lines.append("")
    lines.append("METRIC RANGE CHECK")
    lines.append(f"  Range issues                   : {audit['range_issues']}")
    lines.append(f"  Metric ranges valid            : {audit['metric_ranges_valid']}")
    lines.append("")
    lines.append("OVERALL")
    lines.append(f"  Overall pass                   : {audit['overall_pass']}")
    lines.append("")
    lines.append("MULTI-SEED HYP OUTPUT")
    lines.append(f"  Multi-seed file exists         : {multiseed_result['multiseed_file_exists']}")
    lines.append(f"  Multi-seed outputs created     : {multiseed_result['multiseed_outputs_created']}")
    if multiseed_result.get("multiseed_outputs_created", False):
        lines.append(f"  Multi-seed HYP CSV             : {multiseed_result['multiseed_hyp_csv']}")
        lines.append(f"  Multi-seed HYP pivot CSV       : {multiseed_result['multiseed_hyp_pivot_csv']}")
        lines.append(f"  Multi-seed HYP LaTeX           : {multiseed_result['multiseed_hyp_tex']}")
    lines.append("")
    lines.append("FILES SAVED")
    lines.append(f"  - {HYP_COMPLETE_CSV}")
    lines.append(f"  - {HYP_F1_PIVOT_CSV}")
    lines.append(f"  - {HYP_DETAIL_TEX}")
    lines.append(f"  - {HYP_F1_PIVOT_TEX}")
    lines.append(f"  - {AUDIT_TXT}")
    lines.append(f"  - {AUDIT_JSON}")
    lines.append("=" * 120)

    with open(AUDIT_TXT, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("\n".join(lines))



def main() -> None:
    print("=" * 120)
    print("STEP 24C-FIX — COMPLETE PTB-XL HYP TABLE FROM STEP 22 LONG METRICS")
    print("=" * 120)
    print(f"Project root       : {PROJECT_ROOT}")
    print(f"Step 22 long CSV   : {STEP22_LONG_CSV}")
    print(f"Output directory   : {OUT_DIR}")
    print("=" * 120)

    ptb = load_step22_hyp_table()
    audit = audit_hyp_table(ptb)

    if not audit["overall_pass"]:
        print("WARNING: Audit did not fully pass. Details will be saved.")
        print(json.dumps(audit, indent=2))

    hyp = build_complete_hyp_table(ptb)
    pivot = build_hyp_f1_pivot(hyp)

    hyp.to_csv(HYP_COMPLETE_CSV, index=False)
    pivot.to_csv(HYP_F1_PIVOT_CSV, index=False)

    export_main_hyp_latex(hyp, pivot)

    multiseed_result = build_multiseed_hyp_table_if_available()

    save_audit_report(audit, multiseed_result)

    if audit["overall_pass"]:
        print("\nDONE ✅ Complete HYP supplementary table created successfully.")
        print("The previous V1+V5 unseen missing-row issue is resolved because this table is built from Step 22 long metrics.")
    else:
        print("\nDONE with warnings. Please inspect the audit report before using the table.")


if __name__ == "__main__":
    main()

09_reviewer_code_consolidation_step25

In [ ]:

from __future__ import annotations

import json
import math
import re
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



PROJECT_ROOT = Path(
    r"PROJECT_PATH"
)

LEAD_MASK_ROOT = PROJECT_ROOT / "lead_masking_final"

STEP22_DIR = LEAD_MASK_ROOT / "step22_final_evaluation_bootstrap_mi"
STEP22_TABLE_DIR = STEP22_DIR / "tables"

STEP24_DIR = LEAD_MASK_ROOT / "step24_reviewer_requested_supplementary"
STEP24A_DIR = STEP24_DIR / "step24a_pmask_sweep"
STEP24B_DIR = STEP24_DIR / "step24b_multiseed_runs"
STEP24C_FIX_DIR = STEP24_DIR / "step24c_hyp_perclass_table_FIXED_complete_from_step22"
STEP24D_DIR = STEP24_DIR / "step24d_12lead_bootstrap_ci"

CHAPMAN_PROCESSED_DIR = PROJECT_ROOT / "processed_chapman_4class_raw_ecgdata"

OUT_DIR = LEAD_MASK_ROOT / "step25_reviewer_code_requirement_consolidation"
TABLE_OUT = OUT_DIR / "tables"
FIG_OUT = OUT_DIR / "figures"
REPORT_OUT = OUT_DIR / "reports"

for d in [OUT_DIR, TABLE_OUT, FIG_OUT, REPORT_OUT]:
    d.mkdir(parents=True, exist_ok=True)



STEP22_LONG_CSV = STEP22_TABLE_DIR / "step22_all_methods_all_conditions_metrics_long.csv"
STEP22_MASTER_CSV = STEP22_TABLE_DIR / "step22_consolidated_master_macro_f1_table.csv"

BOOTSTRAP_CANDIDATES = [
    STEP22_TABLE_DIR / "step22_pairwise_bootstrap_macro_f1.csv",
    STEP22_TABLE_DIR / "pairwise_bootstrap_macro_f1.csv",
    STEP22_DIR / "step22_pairwise_bootstrap_macro_f1.csv",
]

PMASK_SWEEP_CSV = STEP24A_DIR / "Supplementary_Table_S1_pmask_sweep_validation_macro_f1.csv"
PMASK_BEST_CSV = STEP24A_DIR / "Supplementary_Table_S1_best_pmask_by_policy.csv"

MULTISEED_LONG_CSV = STEP24B_DIR / "multiseed_all_test_condition_metrics_long.csv"
MULTISEED_SUMMARY_CSV = STEP24B_DIR / "multiseed_training_run_summaries.csv"
MULTISEED_AGG_CSV = STEP24B_DIR / "Supplementary_Table_S2_multiseed_mean_std_by_policy_condition.csv"
MULTISEED_MARGIN_CSV = STEP24B_DIR / "Supplementary_Table_S3_multiseed_clinical_minus_random_known_reduced.csv"

HYP_COMPLETE_CSV = STEP24C_FIX_DIR / "Supplementary_Table_S4_PTBXL_HYP_precision_recall_f1_COMPLETE.csv"
HYP_PIVOT_CSV = STEP24C_FIX_DIR / "Supplementary_Table_S4_PTBXL_HYP_F1_pivot_COMPLETE.csv"
HYP_MULTI_CSV = STEP24C_FIX_DIR / "Supplementary_Table_S4B_PTBXL_HYP_F1_multiseed_mean_std.csv"



DATASETS = ["PTB-XL", "Chapman"]
METHODS = ["Standard", "Random", "Clinical"]

PTBXL_CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
CHAPMAN_CLASSES = ["SB", "AFIB", "GSVT", "SR"]

LEAD_ORDER = [
    "12lead_full",
    "6_limb",
    "6_precordial",
    "3_limb",
    "lead_II_only",
    "V5_only",
    "lead_I_only_unseen",
    "V1_only_unseen",
    "I_II_unseen",
    "V1_V5_unseen",
]

KNOWN_CONDITIONS = [
    "12lead_full",
    "6_limb",
    "6_precordial",
    "3_limb",
    "lead_II_only",
    "V5_only",
]

KNOWN_REDUCED_CONDITIONS = [
    "6_limb",
    "6_precordial",
    "3_limb",
    "lead_II_only",
    "V5_only",
]

UNSEEN_CONDITIONS = [
    "lead_I_only_unseen",
    "V1_only_unseen",
    "I_II_unseen",
    "V1_V5_unseen",
]

UNSEEN_SINGLE_LEAD_CONDITIONS = [
    "lead_I_only_unseen",
    "V1_only_unseen",
]

LEAD_DISPLAY = {
    "12lead_full": "12-lead full",
    "6_limb": "6 limb",
    "6_precordial": "6 precordial",
    "3_limb": "3 limb",
    "lead_II_only": "Lead II only",
    "V5_only": "V5 only",
    "lead_I_only_unseen": "Lead I only$^{\\dagger}$",
    "V1_only_unseen": "V1 only$^{\\dagger}$",
    "I_II_unseen": "I+II$^{\\dagger}$",
    "V1_V5_unseen": "V1+V5$^{\\dagger}$",
}

HEATMAP_LABELS = {
    "12lead_full": "12-Lead",
    "6_limb": "6 Limb",
    "6_precordial": "6 Precordial",
    "3_limb": "3 Limb",
    "lead_II_only": "Lead II",
    "V5_only": "V5",
    "lead_I_only_unseen": "Lead I†",
    "V1_only_unseen": "V1†",
    "I_II_unseen": "I+II†",
    "V1_V5_unseen": "V1+V5†",
}

COMPARISON_ORDER = [
    "Clinical - Standard",
    "Clinical - Random",
    "Random - Standard",
]



def require_exists(path: Path, desc: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {desc}: {path}")


def find_bootstrap_csv() -> Path:
    for p in BOOTSTRAP_CANDIDATES:
        if p.exists():
            return p

    found = list(STEP22_DIR.rglob("*bootstrap*macro*f1*.csv"))
    if len(found) == 0:
        raise FileNotFoundError(
            f"No bootstrap Macro-F1 CSV found under {STEP22_DIR}"
        )

    return found[0]


def normalize_dataset_name(x: Any) -> str:
    s = str(x).strip().lower()
    if "ptb" in s:
        return "PTB-XL"
    if "chap" in s:
        return "Chapman"
    return str(x)


def normalize_method_name(x: Any) -> str:
    s = str(x).strip().lower()
    if "standard" in s or s in ["std", "s"]:
        return "Standard"
    if "random" in s or s in ["rand", "r"]:
        return "Random"
    if "clinical" in s or s in ["clin", "c"]:
        return "Clinical"
    return str(x)


def normalize_lead_condition(x: Any) -> str:
    s = str(x).strip()
    low = s.lower()

    aliases = {
        "12_lead_full": "12lead_full",
        "12-lead full": "12lead_full",
        "12 lead full": "12lead_full",
        "12lead": "12lead_full",
        "full": "12lead_full",
        "6 limb leads": "6_limb",
        "6-limb": "6_limb",
        "6 limb": "6_limb",
        "6_limb": "6_limb",
        "6 precordial leads": "6_precordial",
        "6-precordial": "6_precordial",
        "6 precordial": "6_precordial",
        "6_precordial": "6_precordial",
        "3 limb leads": "3_limb",
        "3-limb": "3_limb",
        "3 limb": "3_limb",
        "3_limb_i_ii_iii": "3_limb",
        "3_limb": "3_limb",
        "lead ii only": "lead_II_only",
        "lead_ii_only": "lead_II_only",
        "lead ii": "lead_II_only",
        "v5 only": "V5_only",
        "v5_only": "V5_only",
        "v5": "V5_only",
        "lead i only†": "lead_I_only_unseen",
        "lead i only": "lead_I_only_unseen",
        "lead_i_only_unseen": "lead_I_only_unseen",
        "lead_i_only": "lead_I_only_unseen",
        "v1 only†": "V1_only_unseen",
        "v1 only": "V1_only_unseen",
        "v1_only_unseen": "V1_only_unseen",
        "v1_only": "V1_only_unseen",
        "i+ii†": "I_II_unseen",
        "i+ii": "I_II_unseen",
        "i_ii_unseen": "I_II_unseen",
        "v1+v5†": "V1_V5_unseen",
        "v1+v5": "V1_V5_unseen",
        "v1_v5_unseen": "V1_V5_unseen",
    }

    if low in aliases:
        return aliases[low]

    return s


def normalize_comparison(x: Any) -> str:
    s = str(x).strip()
    s = s.replace("_vs_", " - ")
    s = s.replace("_minus_", " - ")
    s = s.replace("Clinical_vs_Standard", "Clinical - Standard")
    s = s.replace("Clinical_vs_Random", "Clinical - Random")
    s = s.replace("Random_vs_Standard", "Random - Standard")
    s = s.replace("Clinical-Standard", "Clinical - Standard")
    s = s.replace("Clinical-Random", "Clinical - Random")
    s = s.replace("Random-Standard", "Random - Standard")
    s = re.sub(r"\s+", " ", s)
    return s


def round_half_up_float(x: float, ndigits: int = 3) -> str:
    q = Decimal("1." + "0" * ndigits)
    return str(Decimal(str(float(x))).quantize(q, rounding=ROUND_HALF_UP))


def signed_round_half_up(x: float, ndigits: int = 3) -> str:
    val = round_half_up_float(abs(float(x)), ndigits)
    sign = "+" if float(x) >= 0 else "-"
    return f"{sign}{val}"


def fmt4(x: Any) -> str:
    if pd.isna(x):
        return ""
    return f"{float(x):.4f}"


def fmt_ci(row: pd.Series) -> str:
    return f"[{float(row['ci_lower']):+.4f}, {float(row['ci_upper']):+.4f}]"


def fmt_mean_std(mean: Any, std: Any) -> str:
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{float(mean):.4f}"
    return f"{float(mean):.4f} $\\pm$ {float(std):.4f}"


def export_latex(df: pd.DataFrame, path: Path, caption: str, label: str, escape: bool = False) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(
            df.to_latex(
                index=False,
                escape=escape,
                float_format="%.4f",
                caption=caption,
                label=label,
            )
        )


def add_order_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["_dataset_order"] = out["dataset"].map({"PTB-XL": 0, "Chapman": 1})
    out["_method_order"] = out.get("method", pd.Series(index=out.index)).map({"Standard": 0, "Random": 1, "Clinical": 2})
    out["_lead_order"] = out["lead_condition"].map({k: i for i, k in enumerate(LEAD_ORDER)})
    return out



def load_step22_long() -> pd.DataFrame:
    require_exists(STEP22_LONG_CSV, "Step 22 long metrics")
    df = pd.read_csv(STEP22_LONG_CSV)

    required = {"dataset", "method", "lead_condition", "macro_f1"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise RuntimeError(f"Step22 long metrics missing columns: {missing}")

    df = df.copy()
    df["dataset"] = df["dataset"].apply(normalize_dataset_name)
    df["method"] = df["method"].apply(normalize_method_name)
    df["lead_condition"] = df["lead_condition"].apply(normalize_lead_condition)

    return df


def load_step22_master() -> pd.DataFrame:
    require_exists(STEP22_MASTER_CSV, "Step 22 master table")
    df = pd.read_csv(STEP22_MASTER_CSV)

    if "dataset" in df.columns:
        df["dataset"] = df["dataset"].apply(normalize_dataset_name)
    if "lead_condition" in df.columns:
        df["lead_condition"] = df["lead_condition"].apply(normalize_lead_condition)

    return df


def load_bootstrap() -> pd.DataFrame:
    boot_csv = find_bootstrap_csv()
    df = pd.read_csv(boot_csv)

    required_base = {"dataset", "lead_condition", "comparison", "ci_lower", "ci_upper"}
    missing = sorted(required_base - set(df.columns))
    if missing:
        raise RuntimeError(f"Bootstrap table missing columns: {missing}")

    df = df.copy()
    df["dataset"] = df["dataset"].apply(normalize_dataset_name)
    df["lead_condition"] = df["lead_condition"].apply(normalize_lead_condition)
    df["comparison"] = df["comparison"].apply(normalize_comparison)

    if "observed_gain" not in df.columns:
        if "observed_diff" in df.columns:
            df["observed_gain"] = df["observed_diff"]
        elif "delta" in df.columns:
            df["observed_gain"] = df["delta"]
        else:
            raise RuntimeError("Bootstrap table missing observed_gain/observed_diff/delta.")

    return df



def build_pmask_policy_summary() -> Dict[str, Any]:
    require_exists(PMASK_SWEEP_CSV, "Step 24A p_mask sweep table")
    pm = pd.read_csv(PMASK_SWEEP_CSV)

    required = {"policy", "p_mask", "best_val_full_macro_f1", "val_known_reduced_mean_macro_f1"}
    missing = sorted(required - set(pm.columns))
    if missing:
        raise RuntimeError(f"p_mask sweep table missing columns: {missing}")

    pm["policy"] = pm["policy"].apply(normalize_method_name)
    pm = pm[pm["policy"].isin(["Random", "Clinical"])].copy()

    pm = pm.sort_values(["policy", "p_mask"]).reset_index(drop=True)

    best = (
        pm.sort_values(["policy", "val_known_reduced_mean_macro_f1"], ascending=[True, False])
        .groupby("policy")
        .head(1)
        .reset_index(drop=True)
    )

    best["selected_p_mask_is_0_60"] = np.isclose(best["p_mask"].astype(float), 0.60)

    out_csv = TABLE_OUT / "Table_S1_pmask_sweep_per_policy_clean.csv"
    best_csv = TABLE_OUT / "Table_S1_pmask_best_per_policy_clean.csv"
    out_tex = TABLE_OUT / "Table_S1_pmask_sweep_per_policy_clean.tex"
    best_tex = TABLE_OUT / "Table_S1_pmask_best_per_policy_clean.tex"

    pm.to_csv(out_csv, index=False)
    best.to_csv(best_csv, index=False)

    latex_cols = [
        "policy",
        "p_mask",
        "best_epoch",
        "best_val_full_macro_f1",
        "val_known_reduced_mean_macro_f1",
        "val_all_known_mean_macro_f1",
    ]
    latex_cols = [c for c in latex_cols if c in pm.columns]

    export_latex(
        pm[latex_cols],
        out_tex,
        caption=(
            "Validation-only sweep of the lead-masking probability "
            "$p_{\\mathrm{mask}}$ by masking policy. The selection criterion is the "
            "mean validation Macro-F1 across known reduced-lead conditions."
        ),
        label="tab:supp_pmask_sweep_per_policy",
        escape=False,
    )

    best_latex_cols = [
        "policy",
        "p_mask",
        "val_known_reduced_mean_macro_f1",
        "selected_p_mask_is_0_60",
    ]
    export_latex(
        best[best_latex_cols],
        best_tex,
        caption=(
            "Best validation-selected masking probability for each masked policy."
        ),
        label="tab:supp_pmask_best_per_policy",
        escape=False,
    )

    return {
        "pmask_rows": int(len(pm)),
        "best_by_policy": best.to_dict(orient="records"),
        "files": [str(out_csv), str(best_csv), str(out_tex), str(best_tex)],
    }



def build_multiseed_main_tables() -> Dict[str, Any]:
    require_exists(MULTISEED_AGG_CSV, "Step 24B multiseed aggregate table")
    require_exists(MULTISEED_LONG_CSV, "Step 24B multiseed long table")
    require_exists(MULTISEED_SUMMARY_CSV, "Step 24B multiseed training summaries")

    agg = pd.read_csv(MULTISEED_AGG_CSV)
    long_df = pd.read_csv(MULTISEED_LONG_CSV)
    summ = pd.read_csv(MULTISEED_SUMMARY_CSV)

    for df in [agg, long_df, summ]:
        if "dataset" in df.columns:
            df["dataset"] = df["dataset"].apply(normalize_dataset_name)
        if "policy" in df.columns:
            df["policy"] = df["policy"].apply(normalize_method_name)
        if "method" in df.columns:
            df["method"] = df["method"].apply(normalize_method_name)
        if "lead_condition" in df.columns:
            df["lead_condition"] = df["lead_condition"].apply(normalize_lead_condition)

    expected_pairs = {(m, c) for m in METHODS for c in LEAD_ORDER}
    observed_pairs = set(zip(agg["policy"] if "policy" in agg.columns else agg["method"], agg["lead_condition"]))
    missing_pairs = sorted(expected_pairs - observed_pairs)

    policy_col = "policy" if "policy" in agg.columns else "method"

    pivot_rows = []
    for cond in LEAD_ORDER:
        row = {
            "dataset": "PTB-XL",
            "lead_condition": cond,
            "lead_display_name": LEAD_DISPLAY[cond],
        }
        for method in METHODS:
            sub = agg[(agg[policy_col] == method) & (agg["lead_condition"] == cond)]
            if len(sub) == 1:
                row[f"{method}_Macro_F1_mean_std"] = fmt_mean_std(
                    sub.iloc[0]["macro_f1_mean"],
                    sub.iloc[0]["macro_f1_std"],
                )
                row[f"{method}_Macro_F1_mean"] = float(sub.iloc[0]["macro_f1_mean"])
                row[f"{method}_Macro_F1_std"] = float(sub.iloc[0]["macro_f1_std"])
                row[f"{method}_n_seeds"] = int(sub.iloc[0]["n_seeds"])
            else:
                row[f"{method}_Macro_F1_mean_std"] = ""
                row[f"{method}_Macro_F1_mean"] = np.nan
                row[f"{method}_Macro_F1_std"] = np.nan
                row[f"{method}_n_seeds"] = np.nan

        if not pd.isna(row["Clinical_Macro_F1_mean"]) and not pd.isna(row["Random_Macro_F1_mean"]):
            row["Clinical_minus_Random_mean"] = row["Clinical_Macro_F1_mean"] - row["Random_Macro_F1_mean"]
        else:
            row["Clinical_minus_Random_mean"] = np.nan

        pivot_rows.append(row)

    pivot = pd.DataFrame(pivot_rows)

    out_csv = TABLE_OUT / "Main_Table_multiseed_PTBXL_MacroF1_mean_std_pivot.csv"
    out_tex = TABLE_OUT / "Main_Table_multiseed_PTBXL_MacroF1_mean_std_pivot.tex"
    margin_csv = TABLE_OUT / "Main_Table_multiseed_PTBXL_known_reduced_Clinical_minus_Random.csv"
    margin_tex = TABLE_OUT / "Main_Table_multiseed_PTBXL_known_reduced_Clinical_minus_Random.tex"

    pivot.to_csv(out_csv, index=False)

    latex_df = pivot[
        [
            "lead_display_name",
            "Standard_Macro_F1_mean_std",
            "Random_Macro_F1_mean_std",
            "Clinical_Macro_F1_mean_std",
            "Clinical_minus_Random_mean",
        ]
    ].copy()
    latex_df.columns = [
        "Lead Condition",
        "Standard",
        "Random",
        "Clinical",
        "Clinical--Random Mean",
    ]

    export_latex(
        latex_df,
        out_tex,
        caption=(
            "PTB-XL multi-seed Macro-F1 results. Values are mean $\\pm$ standard deviation "
            "over three independent training seeds."
        ),
        label="tab:main_multiseed_ptbxl_macro_f1",
        escape=False,
    )

    margins = pivot[pivot["lead_condition"].isin(KNOWN_REDUCED_CONDITIONS)].copy()
    margins = margins[
        [
            "lead_condition",
            "lead_display_name",
            "Random_Macro_F1_mean",
            "Random_Macro_F1_std",
            "Clinical_Macro_F1_mean",
            "Clinical_Macro_F1_std",
            "Clinical_minus_Random_mean",
        ]
    ]
    margins.to_csv(margin_csv, index=False)

    latex_margin = margins.copy()
    latex_margin.columns = [
        "Lead Condition",
        "Lead Display",
        "Random Mean",
        "Random Std",
        "Clinical Mean",
        "Clinical Std",
        "Clinical--Random Mean",
    ]
    export_latex(
        latex_margin,
        margin_tex,
        caption=(
            "Multi-seed PTB-XL Clinical--Random comparison across known reduced-lead conditions. "
            "Small single-lead margins should be interpreted cautiously."
        ),
        label="tab:main_multiseed_clinical_random_known_reduced",
        escape=False,
    )

    return {
        "aggregate_rows": int(len(agg)),
        "long_rows": int(len(long_df)),
        "summary_rows": int(len(summ)),
        "missing_policy_condition_pairs": missing_pairs,
        "files": [str(out_csv), str(out_tex), str(margin_csv), str(margin_tex)],
    }



def build_known_condition_bootstrap_tables() -> Dict[str, Any]:
    boot = load_bootstrap()

    known = boot[boot["lead_condition"].isin(KNOWN_CONDITIONS)].copy()
    known_reduced = boot[boot["lead_condition"].isin(KNOWN_REDUCED_CONDITIONS)].copy()

    if len(known) == 0:
        raise RuntimeError("No known-condition bootstrap rows found.")

    compact = known_reduced[
        known_reduced["comparison"].isin(["Clinical - Random", "Clinical - Standard"])
    ].copy()

    compact["lead_display_name"] = compact["lead_condition"].map(LEAD_DISPLAY)
    compact["ci_95"] = compact.apply(fmt_ci, axis=1)

    compact = add_order_cols(compact).sort_values(
        ["_dataset_order", "_lead_order", "comparison"]
    ).drop(columns=["_dataset_order", "_lead_order", "_method_order"], errors="ignore")

    compact_cols = [
        "dataset",
        "lead_display_name",
        "comparison",
        "observed_gain",
        "ci_lower",
        "ci_upper",
        "ci_95",
    ]
    for extra in ["p_gain_leq_0", "n_bootstrap"]:
        if extra in compact.columns:
            compact_cols.append(extra)

    compact = compact[compact_cols]

    full = known.copy()
    full["lead_display_name"] = full["lead_condition"].map(LEAD_DISPLAY)
    full["ci_95"] = full.apply(fmt_ci, axis=1)
    full = add_order_cols(full).sort_values(
        ["_dataset_order", "_lead_order", "comparison"]
    ).drop(columns=["_dataset_order", "_lead_order", "_method_order"], errors="ignore")

    compact_csv = TABLE_OUT / "Main_Table_known_reduced_bootstrap_CIs_compact.csv"
    compact_tex = TABLE_OUT / "Main_Table_known_reduced_bootstrap_CIs_compact.tex"
    full_csv = TABLE_OUT / "Supplementary_Table_all_known_condition_bootstrap_CIs_full.csv"
    full_tex = TABLE_OUT / "Supplementary_Table_all_known_condition_bootstrap_CIs_full.tex"

    compact.to_csv(compact_csv, index=False)
    full.to_csv(full_csv, index=False)

    latex_compact = compact.copy()
    latex_compact.columns = [
        c.replace("_", " ").title().replace("Ci", "CI").replace("95", "95%")
        for c in latex_compact.columns
    ]

    export_latex(
        latex_compact,
        compact_tex,
        caption=(
            "Paired bootstrap confidence intervals for known reduced-lead Macro-F1 comparisons. "
            "This table surfaces the uncertainty for the main reduced-lead policy claims."
        ),
        label="tab:known_reduced_bootstrap_ci",
        escape=False,
    )

    latex_full_cols = [
        "dataset",
        "lead_display_name",
        "comparison",
        "observed_gain",
        "ci_lower",
        "ci_upper",
        "ci_95",
    ]
    latex_full_cols = [c for c in latex_full_cols if c in full.columns]

    export_latex(
        full[latex_full_cols],
        full_tex,
        caption=(
            "Full paired bootstrap confidence intervals for all known lead conditions, including the 12-lead condition."
        ),
        label="tab:supp_all_known_bootstrap_ci",
        escape=False,
    )

    return {
        "known_rows": int(len(known)),
        "known_reduced_rows": int(len(known_reduced)),
        "compact_rows": int(len(compact)),
        "files": [str(compact_csv), str(compact_tex), str(full_csv), str(full_tex)],
    }



def build_all_class_unseen_singlelead_table(step22_long: pd.DataFrame) -> Dict[str, Any]:
    df = step22_long.copy()
    df = df[
        (df["dataset"] == "PTB-XL")
        & (df["lead_condition"].isin(UNSEEN_SINGLE_LEAD_CONDITIONS))
        & (df["method"].isin(METHODS))
    ].copy()

    required_cols = []
    for cls in PTBXL_CLASSES:
        required_cols.extend([f"precision_{cls}", f"recall_{cls}", f"f1_{cls}", f"support_{cls}"])

    missing = sorted(set(required_cols) - set(df.columns))
    if missing:
        raise RuntimeError(f"Step22 long table missing per-class columns: {missing}")

    rows = []
    for _, r in df.iterrows():
        for cls in PTBXL_CLASSES:
            rows.append({
                "dataset": "PTB-XL",
                "method": r["method"],
                "lead_condition": r["lead_condition"],
                "lead_display_name": LEAD_DISPLAY[r["lead_condition"]],
                "class": cls,
                "precision": float(r[f"precision_{cls}"]),
                "recall": float(r[f"recall_{cls}"]),
                "f1": float(r[f"f1_{cls}"]),
                "support": int(r[f"support_{cls}"]),
            })

    out = pd.DataFrame(rows)
    out["_lead_order"] = out["lead_condition"].map({k: i for i, k in enumerate(LEAD_ORDER)})
    out["_method_order"] = out["method"].map({m: i for i, m in enumerate(METHODS)})
    out["_class_order"] = out["class"].map({c: i for i, c in enumerate(PTBXL_CLASSES)})
    out = out.sort_values(["_lead_order", "_method_order", "_class_order"]).drop(
        columns=["_lead_order", "_method_order", "_class_order"]
    )

    out_csv = TABLE_OUT / "Supplementary_Table_all_PTBXL_classes_unseen_singlelead_precision_recall_f1.csv"
    out_tex = TABLE_OUT / "Supplementary_Table_all_PTBXL_classes_unseen_singlelead_precision_recall_f1.tex"

    out.to_csv(out_csv, index=False)

    export_latex(
        out,
        out_tex,
        caption=(
            "Per-class PTB-XL precision, recall, and F1-score under unseen single-lead conditions. "
            "This table checks whether the MI precision-collapse observation generalizes to other classes."
        ),
        label="tab:supp_all_classes_unseen_singlelead",
        escape=False,
    )

    summary = (
        out.sort_values(["lead_condition", "method", "precision"])
        .groupby(["lead_condition", "method"])
        .head(1)
        .reset_index(drop=True)
    )

    summary_csv = TABLE_OUT / "Supplementary_Table_unseen_singlelead_lowest_precision_class_summary.csv"
    summary.to_csv(summary_csv, index=False)

    return {
        "rows": int(len(out)),
        "expected_rows": int(len(UNSEEN_SINGLE_LEAD_CONDITIONS) * len(METHODS) * len(PTBXL_CLASSES)),
        "files": [str(out_csv), str(out_tex), str(summary_csv)],
    }



def build_hyp_main_extraction() -> Dict[str, Any]:
    require_exists(HYP_COMPLETE_CSV, "Step 24C-FIX complete HYP table")
    hyp = pd.read_csv(HYP_COMPLETE_CSV)

    required = {"method", "lead_condition", "precision_HYP", "recall_HYP", "f1_HYP", "support_HYP"}
    missing = sorted(required - set(hyp.columns))
    if missing:
        raise RuntimeError(f"HYP complete table missing columns: {missing}")

    hyp["method"] = hyp["method"].apply(normalize_method_name)
    hyp["lead_condition"] = hyp["lead_condition"].apply(normalize_lead_condition)

    standard_known = hyp[
        (hyp["method"] == "Standard")
        & (hyp["lead_condition"].isin(KNOWN_REDUCED_CONDITIONS))
    ].copy()

    if len(standard_known) == 0:
        raise RuntimeError("No Standard known-reduced HYP rows found.")

    worst_condition = standard_known.sort_values("f1_HYP").iloc[0]["lead_condition"]

    selected_conditions = ["12lead_full", worst_condition]

    out = hyp[hyp["lead_condition"].isin(selected_conditions)].copy()
    out["lead_display_name"] = out["lead_condition"].map(LEAD_DISPLAY)
    out["_lead_order"] = out["lead_condition"].map({k: i for i, k in enumerate(selected_conditions)})
    out["_method_order"] = out["method"].map({m: i for i, m in enumerate(METHODS)})
    out = out.sort_values(["_lead_order", "_method_order"]).drop(columns=["_lead_order", "_method_order"])

    out_csv = TABLE_OUT / "Main_Table_HYP_F1_full_and_worst_known_condition.csv"
    out_tex = TABLE_OUT / "Main_Table_HYP_F1_full_and_worst_known_condition.tex"

    out.to_csv(out_csv, index=False)

    latex_df = out[
        ["lead_display_name", "method", "precision_HYP", "recall_HYP", "f1_HYP", "support_HYP"]
    ].copy()
    latex_df.columns = [
        "Lead Condition",
        "Policy",
        "HYP Precision",
        "HYP Recall",
        "HYP F1",
        "HYP Support",
    ]

    export_latex(
        latex_df,
        out_tex,
        caption=(
            "PTB-XL HYP-class precision, recall, and F1-score for the full 12-lead condition "
            "and the most degraded known reduced-lead condition under Standard training."
        ),
        label="tab:main_hyp_full_worst_known",
        escape=False,
    )

    return {
        "worst_known_condition_by_standard_hyp_f1": str(worst_condition),
        "rows": int(len(out)),
        "files": [str(out_csv), str(out_tex)],
    }



def build_training_epoch_metadata_table() -> Dict[str, Any]:
    rows = []

    if MULTISEED_SUMMARY_CSV.exists():
        ms = pd.read_csv(MULTISEED_SUMMARY_CSV)
        for _, r in ms.iterrows():
            rows.append({
                "source": "Step24B_multiseed",
                "dataset": normalize_dataset_name(r.get("dataset", "PTB-XL")),
                "policy": normalize_method_name(r.get("policy", r.get("method", ""))),
                "seed": int(r["seed"]) if "seed" in r and not pd.isna(r["seed"]) else "",
                "p_mask": r.get("p_mask", ""),
                "best_epoch": r.get("best_epoch", ""),
                "best_val_macro_f1": r.get("best_val_full_macro_f1", r.get("best_val_macro_f1", "")),
                "elapsed_minutes": r.get("elapsed_minutes", r.get("total_training_minutes", "")),
                "checkpoint": r.get("best_checkpoint", ""),
            })

    for p in LEAD_MASK_ROOT.rglob("summary.json"):
        try:
            obj = json.loads(p.read_text())
        except Exception:
            continue

        policy = obj.get("policy", obj.get("method", obj.get("model_key", "")))
        if not policy and "training_result" in obj and isinstance(obj["training_result"], dict):
            tr = obj["training_result"]
            policy = tr.get("model_key", tr.get("display_name", ""))

        if not policy:
            continue

        dataset = "PTB-XL" if "ptb" in str(p).lower() else ("Chapman" if "chap" in str(p).lower() else "")

        rows.append({
            "source": "summary_json",
            "dataset": dataset,
            "policy": normalize_method_name(policy),
            "seed": obj.get("seed", ""),
            "p_mask": obj.get("p_mask", ""),
            "best_epoch": obj.get("best_epoch", obj.get("epoch", "")),
            "best_val_macro_f1": obj.get("best_val_full_macro_f1", obj.get("best_val_macro_f1", "")),
            "elapsed_minutes": obj.get("elapsed_minutes", obj.get("total_training_minutes", "")),
            "checkpoint": obj.get("best_checkpoint", ""),
            "summary_file": str(p),
        })

    meta = pd.DataFrame(rows)

    if len(meta) == 0:
        raise RuntimeError("No training metadata found from Step24B or summary.json files.")

    dedup_cols = [c for c in ["source", "dataset", "policy", "seed", "p_mask", "best_epoch", "checkpoint"] if c in meta.columns]
    meta = meta.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

    out_csv = TABLE_OUT / "Training_metadata_best_epoch_wallclock_audit.csv"
    out_tex = TABLE_OUT / "Training_metadata_best_epoch_wallclock_audit.tex"

    meta.to_csv(out_csv, index=False)

    compact = meta.copy()
    compact = compact[compact["best_epoch"].astype(str).str.len() > 0].copy()

    keep = [
        "source",
        "dataset",
        "policy",
        "seed",
        "p_mask",
        "best_epoch",
        "best_val_macro_f1",
        "elapsed_minutes",
    ]
    keep = [c for c in keep if c in compact.columns]

    export_latex(
        compact[keep].head(80),
        out_tex,
        caption=(
            "Training metadata audit showing available best-validation epochs and wall-clock time. "
            "Rows are extracted from saved run summaries and checkpoint metadata."
        ),
        label="tab:supp_training_epoch_metadata",
        escape=False,
    )

    return {
        "metadata_rows": int(len(meta)),
        "compact_rows_with_best_epoch": int(len(compact)),
        "files": [str(out_csv), str(out_tex)],
    }



def make_gain_heatmap(
    dataset: str,
    step22_long: pd.DataFrame,
    out_prefix: str,
) -> Dict[str, Any]:
    df = step22_long[step22_long["dataset"] == dataset].copy()

    required_pairs = {(m, c) for m in METHODS for c in LEAD_ORDER}
    observed_pairs = set(zip(df["method"], df["lead_condition"]))
    missing = sorted(required_pairs - observed_pairs)
    if missing:
        raise RuntimeError(f"Missing method-condition rows for {dataset}: {missing[:10]}")

    pivot = df.pivot_table(
        index="lead_condition",
        columns="method",
        values="macro_f1",
        aggfunc="first",
    ).reindex(LEAD_ORDER)

    gains = pd.DataFrame({
        "Clinical - Standard": pivot["Clinical"] - pivot["Standard"],
        "Clinical - Random": pivot["Clinical"] - pivot["Random"],
    }).T

    gain_csv = TABLE_OUT / f"{out_prefix}_clinical_gain_heatmap_values_full_precision.csv"
    gains.to_csv(gain_csv)

    plt.rcParams.update({
        "font.size": 12,
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
    })

    fig, ax = plt.subplots(figsize=(15, 4.8), facecolor="white")
    data = gains.values.astype(float)

    im = ax.imshow(data, cmap="RdBu_r", vmin=-0.40, vmax=0.40)

    ax.set_title(
        f"{dataset}: Clinical Gain Across Lead Conditions",
        fontsize=18,
        fontweight="bold",
        pad=14,
    )

    ax.set_xticks(np.arange(len(LEAD_ORDER)))
    ax.set_xticklabels([HEATMAP_LABELS[c] for c in LEAD_ORDER], rotation=0, ha="center", fontsize=10, fontweight="bold")

    ax.set_yticks(np.arange(2))
    ax.set_yticklabels(["Clinical - Standard", "Clinical - Random"], fontsize=12, fontweight="bold")

    ax.set_xticks(np.arange(-0.5, len(LEAD_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 2, 1), minor=True)
    ax.grid(which="minor", color="black", linestyle="-", linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            label = signed_round_half_up(val, ndigits=3)
            color = "white" if abs(val) >= 0.22 else "black"
            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                color=color,
                fontsize=11,
                fontweight="bold",
            )

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.ax.set_ylabel("Macro-F1 Gain", rotation=90, fontsize=12, fontweight="bold")

    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
        spine.set_edgecolor("black")

    fig.tight_layout()

    png = FIG_OUT / f"{out_prefix}_clinical_gain_heatmap_ROUND_HALF_UP.png"
    pdf = FIG_OUT / f"{out_prefix}_clinical_gain_heatmap_ROUND_HALF_UP.pdf"
    svg = FIG_OUT / f"{out_prefix}_clinical_gain_heatmap_ROUND_HALF_UP.svg"

    fig.savefig(png, dpi=400, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    fig.savefig(svg, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    return {
        "dataset": dataset,
        "value_csv": str(gain_csv),
        "figure_png": str(png),
        "figure_pdf": str(pdf),
        "figure_svg": str(svg),
    }


def regenerate_heatmaps(step22_long: pd.DataFrame) -> Dict[str, Any]:
    ptb = make_gain_heatmap("PTB-XL", step22_long, "Figure3_PTBXL")
    chap = make_gain_heatmap("Chapman", step22_long, "Figure4_Chapman")
    return {"PTB-XL": ptb, "Chapman": chap}



def audit_chapman_split_seed() -> Dict[str, Any]:
    candidates = []

    if CHAPMAN_PROCESSED_DIR.exists():
        candidates.extend(list(CHAPMAN_PROCESSED_DIR.rglob("*.json")))
        candidates.extend(list(CHAPMAN_PROCESSED_DIR.rglob("*summary*.txt")))
        candidates.extend(list(CHAPMAN_PROCESSED_DIR.rglob("*stats*.txt")))

    found = []

    seed_patterns = [
        r"random[_\s-]*state\s*[:=]\s*(\d+)",
        r"split[_\s-]*seed\s*[:=]\s*(\d+)",
        r"seed\s*[:=]\s*(\d+)",
        r"SEED\s*=\s*(\d+)",
        r"random_state\s*=\s*(\d+)",
    ]

    for p in candidates:
        try:
            text = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        for pat in seed_patterns:
            m = re.search(pat, text, flags=re.IGNORECASE)
            if m:
                found.append({
                    "file": str(p),
                    "pattern": pat,
                    "seed": int(m.group(1)),
                })

    out_json = REPORT_OUT / "chapman_split_seed_audit.json"
    out_txt = REPORT_OUT / "chapman_split_seed_audit.txt"

    result = {
        "chapman_processed_dir": str(CHAPMAN_PROCESSED_DIR),
        "candidate_files_scanned": int(len(candidates)),
        "seed_mentions_found": found,
        "unique_seeds_found": sorted(set([x["seed"] for x in found])),
        "status": "found" if found else "not_found",
    }

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    lines = []
    lines.append("=" * 100)
    lines.append("CHAPMAN SPLIT SEED AUDIT")
    lines.append("=" * 100)
    lines.append(f"Directory scanned: {CHAPMAN_PROCESSED_DIR}")
    lines.append(f"Candidate files scanned: {len(candidates)}")
    lines.append(f"Status: {result['status']}")
    lines.append(f"Unique seeds found: {result['unique_seeds_found']}")
    lines.append("")
    if found:
        for x in found[:50]:
            lines.append(f"Seed {x['seed']} found in {x['file']} using pattern {x['pattern']}")
    else:
        lines.append("No explicit Chapman split seed found in metadata/summary files.")
        lines.append("If the preprocessing code used random_state=42, report that from the code.")
        lines.append("Otherwise rerun/inspect the Chapman preprocessing notebook to confirm the seed.")
    lines.append("=" * 100)

    out_txt.write_text("\n".join(lines), encoding="utf-8")

    return {
        "files": [str(out_json), str(out_txt)],
        "status": result["status"],
        "unique_seeds_found": result["unique_seeds_found"],
    }



def write_final_report(results: Dict[str, Any]) -> None:
    report_json = REPORT_OUT / "step25_reviewer_code_requirement_fulfillment_manifest.json"
    report_txt = REPORT_OUT / "step25_reviewer_code_requirement_fulfillment_report.txt"

    with open(report_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    lines = []
    lines.append("=" * 130)
    lines.append("STEP 25 — REVIEWER CODE-REQUIREMENT CONSOLIDATION REPORT")
    lines.append("=" * 130)
    lines.append(f"Project root : {PROJECT_ROOT}")
    lines.append(f"Output dir   : {OUT_DIR}")
    lines.append("")

    checklist = [
        ("L-7", "p_mask sweep per policy", "pmask"),
        ("L-8", "multi-seed main-result tables", "multiseed"),
        ("L-9", "known reduced-lead bootstrap CIs", "known_bootstrap"),
        ("L-5", "all-class unseen single-lead precision/recall/F1", "all_class_unseen_singlelead"),
        ("S-2", "HYP main-paper extraction and complete HYP table", "hyp_main"),
        ("L-11/E-2", "best epoch and wall-clock metadata", "training_metadata"),
        ("D-1", "heatmap rounding fixed with ROUND_HALF_UP", "heatmaps"),
        ("S-6", "Chapman split seed audit", "chapman_seed"),
    ]

    for issue, desc, key in checklist:
        status = "DONE" if key in results and not results[key].get("error") else "CHECK"
        lines.append(f"{issue:<8s} | {status:<6s} | {desc}")

    lines.append("")
    lines.append("OUTPUT FILES BY TASK")
    lines.append("-" * 130)

    for key, obj in results.items():
        lines.append(f"\n[{key}]")
        if isinstance(obj, dict):
            if "files" in obj:
                for fpath in obj["files"]:
                    lines.append(f"  - {fpath}")
            else:
                lines.append(json.dumps(obj, indent=2)[:2000])
        else:
            lines.append(str(obj))

    lines.append("")
    lines.append("MANUSCRIPT ACTIONS NOW ENABLED")
    lines.append("-" * 130)
    lines.append("1. Replace single-seed PTB-XL reduced-lead result claims with multi-seed mean ± std wording.")
    lines.append("2. Add compact known-condition bootstrap CI table or cite the generated table.")
    lines.append("3. Add all-class unseen single-lead precision/recall/F1 table to supplementary material.")
    lines.append("4. Add HYP full-vs-worst-known condition row to the main text or main table.")
    lines.append("5. Add actual best-epoch/wall-clock metadata to Table 5 or supplementary training table.")
    lines.append("6. Replace Figure 3 and Figure 4 with ROUND_HALF_UP regenerated heatmaps.")
    lines.append("7. Add Chapman split seed if found; otherwise state the seed from preprocessing code.")
    lines.append("=" * 130)

    report_txt.write_text("\n".join(lines), encoding="utf-8")

    print("\n".join(lines))
    print(f"\nSaved manifest: {report_json}")
    print(f"Saved report  : {report_txt}")



def main() -> None:
    print("=" * 130)
    print("STEP 25 — REVIEWER CODE-REQUIREMENT CONSOLIDATION")
    print("=" * 130)
    print(f"Project root : {PROJECT_ROOT}")
    print(f"Output dir   : {OUT_DIR}")
    print("=" * 130)

    step22_long = load_step22_long()
    _ = load_step22_master()

    results: Dict[str, Any] = {}

    try:
        print("\n[1/8] Building p_mask per-policy summary...")
        results["pmask"] = build_pmask_policy_summary()
    except Exception as e:
        results["pmask"] = {"error": repr(e)}
        print("ERROR in p_mask summary:", repr(e))

    try:
        print("\n[2/8] Building multi-seed main-result tables...")
        results["multiseed"] = build_multiseed_main_tables()
    except Exception as e:
        results["multiseed"] = {"error": repr(e)}
        print("ERROR in multiseed tables:", repr(e))

    try:
        print("\n[3/8] Building known-condition bootstrap CI tables...")
        results["known_bootstrap"] = build_known_condition_bootstrap_tables()
    except Exception as e:
        results["known_bootstrap"] = {"error": repr(e)}
        print("ERROR in known bootstrap tables:", repr(e))

    try:
        print("\n[4/8] Building all-class unseen single-lead table...")
        results["all_class_unseen_singlelead"] = build_all_class_unseen_singlelead_table(step22_long)
    except Exception as e:
        results["all_class_unseen_singlelead"] = {"error": repr(e)}
        print("ERROR in all-class unseen single-lead table:", repr(e))

    try:
        print("\n[5/8] Building HYP main-paper extraction table...")
        results["hyp_main"] = build_hyp_main_extraction()
    except Exception as e:
        results["hyp_main"] = {"error": repr(e)}
        print("ERROR in HYP main extraction:", repr(e))

    try:
        print("\n[6/8] Building training metadata table...")
        results["training_metadata"] = build_training_epoch_metadata_table()
    except Exception as e:
        results["training_metadata"] = {"error": repr(e)}
        print("ERROR in training metadata:", repr(e))

    try:
        print("\n[7/8] Regenerating heatmaps with consistent rounding...")
        results["heatmaps"] = regenerate_heatmaps(step22_long)
    except Exception as e:
        results["heatmaps"] = {"error": repr(e)}
        print("ERROR in heatmap regeneration:", repr(e))

    try:
        print("\n[8/8] Auditing Chapman split seed...")
        results["chapman_seed"] = audit_chapman_split_seed()
    except Exception as e:
        results["chapman_seed"] = {"error": repr(e)}
        print("ERROR in Chapman seed audit:", repr(e))

    write_final_report(results)

    print("\nDONE ✅ Step 25 completed.")


if __name__ == "__main__":
    main()

10_chapman_multiseed_step26

In [ ]:

from __future__ import annotations

import gc
import csv
import json
import math
import time
import random
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm



PROJECT_ROOT = Path(
    r"PROJECT_PATH"
)

CHAPMAN_DATA_DIR = PROJECT_ROOT / "processed_chapman_4class_raw_ecgdata"

OUT_DIR = (
    PROJECT_ROOT
    / "lead_masking_final"
    / "step26_chapman_multiseed_sensitivity"
)

RUNS_DIR = OUT_DIR / "runs"
TABLE_DIR = OUT_DIR / "tables"
REPORT_DIR = OUT_DIR / "reports"

for d in [OUT_DIR, RUNS_DIR, TABLE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)



DATASET_NAME = "Chapman"

CLASS_NAMES = ["SB", "AFIB", "GSVT", "SR"]
NUM_CLASSES = len(CLASS_NAMES)

N_LEADS = 12
TARGET_LENGTH = 5000

LEAD_NAMES = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

LEAD_CONFIGS = {
    "12lead_full": {
        "display_name": "12-lead full",
        "lead_indices": list(range(12)),
        "known_unseen": "known",
    },
    "6_limb": {
        "display_name": "6 limb",
        "lead_indices": [0, 1, 2, 3, 4, 5],
        "known_unseen": "known",
    },
    "6_precordial": {
        "display_name": "6 precordial",
        "lead_indices": [6, 7, 8, 9, 10, 11],
        "known_unseen": "known",
    },
    "3_limb": {
        "display_name": "3 limb",
        "lead_indices": [0, 1, 2],
        "known_unseen": "known",
    },
    "lead_II_only": {
        "display_name": "Lead II only",
        "lead_indices": [1],
        "known_unseen": "known",
    },
    "V5_only": {
        "display_name": "V5 only",
        "lead_indices": [10],
        "known_unseen": "known",
    },
    "lead_I_only_unseen": {
        "display_name": "Lead I only†",
        "lead_indices": [0],
        "known_unseen": "unseen",
    },
    "V1_only_unseen": {
        "display_name": "V1 only†",
        "lead_indices": [6],
        "known_unseen": "unseen",
    },
    "I_II_unseen": {
        "display_name": "I+II†",
        "lead_indices": [0, 1],
        "known_unseen": "unseen",
    },
    "V1_V5_unseen": {
        "display_name": "V1+V5†",
        "lead_indices": [6, 10],
        "known_unseen": "unseen",
    },
}

ALL_EVAL_KEYS = list(LEAD_CONFIGS.keys())

KNOWN_REDUCED_KEYS = [
    "6_limb",
    "6_precordial",
    "3_limb",
    "lead_II_only",
    "V5_only",
]

POLICIES = ["Standard", "Random", "Clinical"]
SEEDS = [41, 42, 43]

CLINICAL_SUBSETS = [
    [0, 1, 2, 3, 4, 5],       # 6 limb
    [6, 7, 8, 9, 10, 11],     # 6 precordial
    [0, 1, 2],                # 3 limb
    [1],                      # Lead II only
    [10],                     # V5 only
]

CARDINALITY_POOL = [6, 6, 3, 1, 1]

P_MASK = 0.60

BATCH_SIZE = 128
MAX_EPOCHS = 70
EARLY_STOPPING_PATIENCE = 15
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
MIN_LR_RATIO = 0.02
GRAD_CLIP_NORM = 5.0
DROPOUT = 0.10

USE_CLASS_WEIGHTS = True

NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
PIN_MEMORY = torch.cuda.is_available()



def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


def require_exists(path: Path, desc: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {desc}: {path}")


def write_json(path: Path, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def save_history(history: List[Dict[str, Any]], path: Path) -> None:
    if history:
        pd.DataFrame(history).to_csv(path, index=False)


def compute_class_weights(labels: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(labels.astype(int), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)

    weights = counts.sum() / (num_classes * counts)
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32)


def set_warmup_cosine_lr(
    optimizer: torch.optim.Optimizer,
    epoch: int,
    total_epochs: int,
    base_lr: float,
    warmup_epochs: int,
    min_lr_ratio: float,
) -> float:
    if epoch < warmup_epochs:
        lr = base_lr * float(epoch + 1) / float(max(1, warmup_epochs))
    else:
        progress = (epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr = base_lr * (min_lr_ratio + (1.0 - min_lr_ratio) * cosine)

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr


def fmt_mean_std(mean: float, std: float) -> str:
    return f"{mean:.4f} $\\pm$ {std:.4f}"


def export_latex(
    df: pd.DataFrame,
    path: Path,
    caption: str,
    label: str,
    escape: bool = False,
) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(
            df.to_latex(
                index=False,
                escape=escape,
                float_format="%.4f",
                caption=caption,
                label=label,
            )
        )



class LeadMasker:
    def __init__(self, policy: str, p_mask: float = P_MASK):
        self.policy = policy
        self.p_mask = float(p_mask)

        if self.policy not in ["Standard", "Random", "Clinical"]:
            raise ValueError(f"Unknown policy: {policy}")

    def __call__(self, x: np.ndarray) -> Tuple[np.ndarray, Dict[str, Any]]:
        x = np.asarray(x, dtype=np.float32).copy()  # [T, 12]

        if self.policy == "Standard":
            return x, {
                "mask_applied": False,
                "policy": "Standard",
                "kept_leads": list(range(N_LEADS)),
                "kept_count": N_LEADS,
            }

        if random.random() > self.p_mask:
            return x, {
                "mask_applied": False,
                "policy": self.policy,
                "kept_leads": list(range(N_LEADS)),
                "kept_count": N_LEADS,
            }

        if self.policy == "Clinical":
            kept = sorted(random.choice(CLINICAL_SUBSETS))
        elif self.policy == "Random":
            k = int(random.choice(CARDINALITY_POOL))
            kept = sorted(random.sample(range(N_LEADS), k=k))
        else:
            raise ValueError(self.policy)

        zero_leads = [i for i in range(N_LEADS) if i not in kept]
        x[:, zero_leads] = 0.0

        return x, {
            "mask_applied": True,
            "policy": self.policy,
            "kept_leads": kept,
            "kept_count": len(kept),
        }



class ChapmanTrainDataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        split: str,
        policy: str,
        p_mask: float = P_MASK,
        mmap_mode: str = "r",
    ):
        self.signals = np.load(data_dir / f"{split}_signals.npy", mmap_mode=mmap_mode)
        self.labels = np.load(data_dir / f"{split}_labels.npy").astype(np.int64)
        self.metadata_path = data_dir / f"{split}_metadata.csv"
        self.metadata = pd.read_csv(self.metadata_path) if self.metadata_path.exists() else pd.DataFrame()
        self.masker = LeadMasker(policy=policy, p_mask=p_mask)
        self._validate(split)

    def _validate(self, split: str) -> None:
        if self.signals.ndim != 3:
            raise ValueError(f"{split} signals must be 3D, got {self.signals.shape}")
        if self.signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
            raise ValueError(
                f"Expected {split} signals shape (N,{TARGET_LENGTH},{N_LEADS}), got {self.signals.shape}"
            )
        if len(self.signals) != len(self.labels):
            raise ValueError(f"{split}: signals/labels mismatch")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32)
        y = int(self.labels[idx])

        x, mask_meta = self.masker(x)

        x = torch.from_numpy(x.T.copy()).float()

        return {
            "signal": x,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(idx),
            "mask_meta": mask_meta,
        }


class ChapmanFixedLeadDataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        split: str,
        keep_leads: List[int],
        mmap_mode: str = "r",
    ):
        self.signals = np.load(data_dir / f"{split}_signals.npy", mmap_mode=mmap_mode)
        self.labels = np.load(data_dir / f"{split}_labels.npy").astype(np.int64)
        self.keep_leads = sorted(map(int, keep_leads))
        self.zero_leads = [i for i in range(N_LEADS) if i not in self.keep_leads]
        self.metadata_path = data_dir / f"{split}_metadata.csv"
        self.metadata = pd.read_csv(self.metadata_path) if self.metadata_path.exists() else pd.DataFrame()
        self._validate(split)

    def _validate(self, split: str) -> None:
        if self.signals.ndim != 3:
            raise ValueError(f"{split} signals must be 3D, got {self.signals.shape}")
        if self.signals.shape[1:] != (TARGET_LENGTH, N_LEADS):
            raise ValueError(
                f"Expected {split} signals shape (N,{TARGET_LENGTH},{N_LEADS}), got {self.signals.shape}"
            )
        if len(self.signals) != len(self.labels):
            raise ValueError(f"{split}: signals/labels mismatch")
        if len(self.keep_leads) < 1:
            raise ValueError("At least one lead must be retained.")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        x = np.asarray(self.signals[idx], dtype=np.float32).copy()
        y = int(self.labels[idx])

        if self.zero_leads:
            x[:, self.zero_leads] = 0.0

        x = torch.from_numpy(x.T.copy()).float()

        return {
            "signal": x,
            "label": torch.tensor(y, dtype=torch.long),
            "record_idx": int(idx),
        }


def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    out = {
        "signal": torch.stack([b["signal"] for b in batch], dim=0),
        "label": torch.stack([b["label"] for b in batch], dim=0),
        "record_idx": torch.tensor([b["record_idx"] for b in batch], dtype=torch.long),
    }

    if "mask_meta" in batch[0]:
        out["mask_meta"] = [b["mask_meta"] for b in batch]

    return out


def make_loader(
    dataset: Dataset,
    batch_size: int = BATCH_SIZE,
    shuffle: bool = False,
    drop_last: bool = False,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=drop_last,
        collate_fn=collate_fn,
    )



class BasicBlock1D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        kernel_size: int = 7,
        dropout: float = DROPOUT,
    ):
        super().__init__()

        padding = kernel_size // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            bias=False,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.drop(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(identity)

        out = out + identity
        out = self.relu(out)

        return out


class ResNet1DEncoder(nn.Module):
    def __init__(
        self,
        input_channels: int = N_LEADS,
        base_filters: int = 64,
        embedding_dim: int = 512,
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                input_channels,
                base_filters,
                kernel_size=15,
                stride=2,
                padding=7,
                bias=False,
            ),
            nn.BatchNorm1d(base_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )

        self.backbone = nn.Sequential(
            self._make_stage(base_filters, base_filters, n_blocks=2, first_stride=1),
            self._make_stage(base_filters, base_filters * 2, n_blocks=2, first_stride=2),
            self._make_stage(base_filters * 2, base_filters * 4, n_blocks=2, first_stride=2),
            self._make_stage(base_filters * 4, base_filters * 8, n_blocks=2, first_stride=2),
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base_filters * 8, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=DROPOUT),
        )

        self.embedding_dim = embedding_dim
        self._init_weights()

    def _make_stage(
        self,
        in_channels: int,
        out_channels: int,
        n_blocks: int,
        first_stride: int,
    ) -> nn.Sequential:
        blocks = [
            BasicBlock1D(
                in_channels=in_channels,
                out_channels=out_channels,
                stride=first_stride,
                kernel_size=7,
                dropout=DROPOUT,
            )
        ]

        for _ in range(1, n_blocks):
            blocks.append(
                BasicBlock1D(
                    in_channels=out_channels,
                    out_channels=out_channels,
                    stride=1,
                    kernel_size=7,
                    dropout=DROPOUT,
                )
            )

        return nn.Sequential(*blocks)

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.backbone(x)
        x = self.global_pool(x)
        x = self.embedding_head(x)
        return x


class ECGClassifier(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.encoder = ResNet1DEncoder()
        self.classifier = nn.Linear(self.encoder.embedding_dim, num_classes)

        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        logits = self.classifier(z)
        return logits



def metrics_from_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0,
    )

    for i, cls in enumerate(CLASS_NAMES):
        out[f"precision_{cls}"] = float(precision[i])
        out[f"recall_{cls}"] = float(recall[i])
        out[f"f1_{cls}"] = float(f1[i])
        out[f"support_{cls}"] = int(support[i])

    return out



def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: GradScaler,
) -> Dict[str, Any]:
    model.train()

    total_loss = 0.0
    total_n = 0
    y_true_all = []
    y_pred_all = []

    masked_count = 0
    unmasked_count = 0

    for batch in tqdm(loader, desc="train", leave=False):
        x = batch["signal"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        if "mask_meta" in batch:
            for meta in batch["mask_meta"]:
                if meta["mask_applied"]:
                    masked_count += 1
                else:
                    unmasked_count += 1

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss: {loss.item()}")

        scaler.scale(loss).backward()

        if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        scaler.step(optimizer)
        scaler.update()

        pred = torch.argmax(logits.detach(), dim=1)

        bs = y.numel()
        total_loss += float(loss.item()) * bs
        total_n += bs

        y_true_all.extend(y.detach().cpu().numpy().tolist())
        y_pred_all.extend(pred.detach().cpu().numpy().tolist())

    y_true = np.asarray(y_true_all, dtype=np.int64)
    y_pred = np.asarray(y_pred_all, dtype=np.int64)

    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_n)
    metrics["masked_count"] = int(masked_count)
    metrics["unmasked_count"] = int(unmasked_count)
    metrics["observed_mask_rate"] = float(masked_count / max(1, masked_count + unmasked_count))

    return metrics


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    criterion: Optional[nn.Module] = None,
    save_probs: bool = False,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    model.eval()

    total_loss = 0.0
    total_n = 0

    y_true_all = []
    y_pred_all = []
    record_idx_all = []
    prob_rows = []

    for batch in tqdm(loader, desc="eval", leave=False):
        x = batch["signal"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        with autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y) if criterion is not None else None

        probs = torch.softmax(logits.float(), dim=1)
        pred = torch.argmax(probs, dim=1)

        bs = y.numel()

        if loss is not None:
            total_loss += float(loss.item()) * bs

        total_n += bs

        y_true_all.extend(y.detach().cpu().numpy().tolist())
        y_pred_all.extend(pred.detach().cpu().numpy().tolist())
        record_idx_all.extend(batch["record_idx"].cpu().numpy().tolist())

        if save_probs:
            prob_rows.append(probs.detach().cpu().numpy())

    y_true = np.asarray(y_true_all, dtype=np.int64)
    y_pred = np.asarray(y_pred_all, dtype=np.int64)

    metrics = metrics_from_predictions(y_true, y_pred)
    metrics["loss"] = total_loss / max(1, total_n) if criterion is not None else np.nan

    pred_df = pd.DataFrame({
        "record_idx": record_idx_all,
        "y_true": y_true,
        "y_true_name": [CLASS_NAMES[i] for i in y_true],
        "y_pred": y_pred,
        "y_pred_name": [CLASS_NAMES[i] for i in y_pred],
    })

    if save_probs and prob_rows:
        probs_all = np.concatenate(prob_rows, axis=0)
        for i, cls in enumerate(CLASS_NAMES):
            pred_df[f"prob_{cls}"] = probs_all[:, i]

    return metrics, pred_df


def load_best_model(checkpoint_path: Path) -> nn.Module:
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)

    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()

    return model



def train_variant(
    policy: str,
    seed: int,
) -> Dict[str, Any]:
    seed_everything(seed)

    run_dir = RUNS_DIR / f"{DATASET_NAME}_{policy}_p{P_MASK:.2f}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    summary_path = run_dir / "summary.json"
    test_metrics_path = run_dir / f"test_metrics_{DATASET_NAME}_{policy}_p{P_MASK:.2f}_seed{seed}.csv"

    if summary_path.exists() and test_metrics_path.exists():
        print(f"[SKIP] Existing completed run found: {run_dir}")
        with open(summary_path, "r", encoding="utf-8") as f:
            return json.load(f)

    print("=" * 140)
    print(f"Training Chapman multi-seed variant: policy={policy}, seed={seed}")
    print("=" * 140)

    train_labels = np.load(CHAPMAN_DATA_DIR / "train_labels.npy").astype(np.int64)

    if USE_CLASS_WEIGHTS:
        class_weights = compute_class_weights(train_labels, NUM_CLASSES).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        class_weights = None
        criterion = nn.CrossEntropyLoss()

    train_ds = ChapmanTrainDataset(
        data_dir=CHAPMAN_DATA_DIR,
        split="train",
        policy=policy,
        p_mask=P_MASK,
    )

    val_ds = ChapmanFixedLeadDataset(
        data_dir=CHAPMAN_DATA_DIR,
        split="val",
        keep_leads=list(range(N_LEADS)),
    )

    train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = ECGClassifier(num_classes=NUM_CLASSES).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY,
    )

    scaler = GradScaler(device="cuda", enabled=USE_AMP)

    best_val_macro_f1 = -1.0
    best_epoch = -1
    bad_epochs = 0

    history = []

    best_ckpt_path = run_dir / "best_model.pt"
    latest_ckpt_path = run_dir / "latest_model.pt"
    history_csv = run_dir / "history.csv"

    start_time = time.time()

    for epoch in range(MAX_EPOCHS):
        lr = set_warmup_cosine_lr(
            optimizer=optimizer,
            epoch=epoch,
            total_epochs=MAX_EPOCHS,
            base_lr=BASE_LR,
            warmup_epochs=WARMUP_EPOCHS,
            min_lr_ratio=MIN_LR_RATIO,
        )

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
        )

        val_metrics, _ = evaluate_model(
            model=model,
            loader=val_loader,
            criterion=criterion,
            save_probs=False,
        )

        is_best = val_metrics["macro_f1"] > best_val_macro_f1

        if is_best:
            best_val_macro_f1 = float(val_metrics["macro_f1"])
            best_epoch = epoch + 1
            bad_epochs = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch + 1,
                "best_val_macro_f1": best_val_macro_f1,
                "policy": policy,
                "seed": seed,
                "p_mask": P_MASK,
                "class_names": CLASS_NAMES,
                "use_class_weights": USE_CLASS_WEIGHTS,
                "class_weights": class_weights.detach().cpu().numpy().tolist() if class_weights is not None else None,
            }, best_ckpt_path)
        else:
            bad_epochs += 1

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch + 1,
            "best_val_macro_f1": best_val_macro_f1,
            "policy": policy,
            "seed": seed,
            "p_mask": P_MASK,
            "class_names": CLASS_NAMES,
            "use_class_weights": USE_CLASS_WEIGHTS,
        }, latest_ckpt_path)

        row = {
            "epoch": epoch + 1,
            "policy": policy,
            "seed": seed,
            "p_mask": P_MASK,
            "lr": lr,
            "train_loss": train_metrics["loss"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_observed_mask_rate": train_metrics["observed_mask_rate"],
            "val_loss": val_metrics["loss"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "is_best": bool(is_best),
            "best_val_macro_f1_so_far": best_val_macro_f1,
            "bad_epochs": bad_epochs,
        }

        history.append(row)
        save_history(history, history_csv)

        print(
            f"[{policy} | seed={seed}] "
            f"Epoch {epoch+1:03d}/{MAX_EPOCHS} | "
            f"train Macro-F1={train_metrics['macro_f1']:.4f} | "
            f"val Macro-F1={val_metrics['macro_f1']:.4f} | "
            f"best={best_val_macro_f1:.4f} | "
            f"bad={bad_epochs}/{EARLY_STOPPING_PATIENCE}"
        )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if bad_epochs >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}. Best epoch={best_epoch}.")
            break

    elapsed_minutes = (time.time() - start_time) / 60.0

    model = load_best_model(best_ckpt_path)

    test_rows = []

    for lead_key, cfg in LEAD_CONFIGS.items():
        test_ds = ChapmanFixedLeadDataset(
            data_dir=CHAPMAN_DATA_DIR,
            split="test",
            keep_leads=cfg["lead_indices"],
        )

        test_loader = make_loader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

        metrics, pred_df = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=None,
            save_probs=True,
        )

        pred_df["dataset"] = DATASET_NAME
        pred_df["policy"] = policy
        pred_df["seed"] = seed
        pred_df["p_mask"] = P_MASK
        pred_df["lead_condition"] = lead_key
        pred_df["lead_display_name"] = cfg["display_name"]

        pred_path = run_dir / f"predictions_{DATASET_NAME}_{policy}_p{P_MASK:.2f}_seed{seed}_{lead_key}.csv"
        pred_df.to_csv(pred_path, index=False)

        row = {
            "dataset": DATASET_NAME,
            "policy": policy,
            "seed": seed,
            "p_mask": P_MASK,
            "lead_condition": lead_key,
            "lead_display_name": cfg["display_name"],
            "known_unseen": cfg["known_unseen"],
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "elapsed_minutes": elapsed_minutes,
            "best_checkpoint": str(best_ckpt_path),
            "prediction_csv": str(pred_path),
        }

        row.update(metrics)
        test_rows.append(row)

        print(
            f"TEST [{policy} | seed={seed}] {lead_key:<22s} "
            f"Macro-F1={metrics['macro_f1']:.4f} | Weighted-F1={metrics['weighted_f1']:.4f}"
        )

    test_df = pd.DataFrame(test_rows)
    test_df.to_csv(test_metrics_path, index=False)

    summary = {
        "dataset": DATASET_NAME,
        "policy": policy,
        "seed": seed,
        "p_mask": P_MASK,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_macro_f1),
        "elapsed_minutes": float(elapsed_minutes),
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "batch_size": BATCH_SIZE,
        "base_lr": BASE_LR,
        "weight_decay": WEIGHT_DECAY,
        "use_class_weights": USE_CLASS_WEIGHTS,
        "best_checkpoint": str(best_ckpt_path),
        "latest_checkpoint": str(latest_ckpt_path),
        "history_csv": str(history_csv),
        "test_metrics_csv": str(test_metrics_path),
    }

    write_json(summary_path, summary)

    return summary



def aggregate_results(summaries: List[Dict[str, Any]]) -> None:
    print("=" * 140)
    print("Aggregating Chapman multi-seed results")
    print("=" * 140)

    summary_df = pd.DataFrame(summaries)
    summary_csv = TABLE_DIR / "chapman_multiseed_training_run_summaries.csv"
    summary_df.to_csv(summary_csv, index=False)

    all_metrics = []

    for _, r in summary_df.iterrows():
        p = Path(r["test_metrics_csv"])
        if not p.exists():
            raise FileNotFoundError(f"Missing test metrics file: {p}")

        all_metrics.append(pd.read_csv(p))

    long_df = pd.concat(all_metrics, ignore_index=True)

    long_csv = TABLE_DIR / "chapman_multiseed_all_test_condition_metrics_long.csv"
    long_df.to_csv(long_csv, index=False)

    agg = (
        long_df.groupby(["dataset", "policy", "lead_condition", "lead_display_name", "known_unseen"])
        .agg(
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            weighted_f1_mean=("weighted_f1", "mean"),
            weighted_f1_std=("weighted_f1", "std"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_std=("balanced_accuracy", "std"),
            n_seeds=("seed", "nunique"),
            best_epoch_mean=("best_epoch", "mean"),
            best_epoch_std=("best_epoch", "std"),
            elapsed_minutes_mean=("elapsed_minutes", "mean"),
            elapsed_minutes_std=("elapsed_minutes", "std"),
        )
        .reset_index()
    )

    agg_csv = TABLE_DIR / "Supplementary_Table_S8_Chapman_multiseed_mean_std_by_policy_condition.csv"
    agg.to_csv(agg_csv, index=False)

    policy_order = {"Standard": 0, "Random": 1, "Clinical": 2}
    lead_order = {k: i for i, k in enumerate(ALL_EVAL_KEYS)}

    pivot_rows = []
    for lead_key in ALL_EVAL_KEYS:
        row = {
            "lead_condition": lead_key,
            "lead_display_name": LEAD_CONFIGS[lead_key]["display_name"],
            "known_unseen": LEAD_CONFIGS[lead_key]["known_unseen"],
        }

        for policy in POLICIES:
            sub = agg[
                (agg["policy"] == policy)
                & (agg["lead_condition"] == lead_key)
            ]

            if len(sub) == 1:
                mean = float(sub.iloc[0]["macro_f1_mean"])
                std = float(sub.iloc[0]["macro_f1_std"])
                row[f"{policy}_Macro_F1_mean_std"] = fmt_mean_std(mean, std)
                row[f"{policy}_Macro_F1_mean"] = mean
                row[f"{policy}_Macro_F1_std"] = std
            else:
                row[f"{policy}_Macro_F1_mean_std"] = ""
                row[f"{policy}_Macro_F1_mean"] = np.nan
                row[f"{policy}_Macro_F1_std"] = np.nan

        if not pd.isna(row["Clinical_Macro_F1_mean"]) and not pd.isna(row["Random_Macro_F1_mean"]):
            row["Clinical_minus_Random_mean"] = row["Clinical_Macro_F1_mean"] - row["Random_Macro_F1_mean"]
        else:
            row["Clinical_minus_Random_mean"] = np.nan

        if not pd.isna(row["Clinical_Macro_F1_mean"]) and not pd.isna(row["Standard_Macro_F1_mean"]):
            row["Clinical_minus_Standard_mean"] = row["Clinical_Macro_F1_mean"] - row["Standard_Macro_F1_mean"]
        else:
            row["Clinical_minus_Standard_mean"] = np.nan

        if not pd.isna(row["Random_Macro_F1_mean"]) and not pd.isna(row["Standard_Macro_F1_mean"]):
            row["Random_minus_Standard_mean"] = row["Random_Macro_F1_mean"] - row["Standard_Macro_F1_mean"]
        else:
            row["Random_minus_Standard_mean"] = np.nan

        pivot_rows.append(row)

    pivot_df = pd.DataFrame(pivot_rows)

    pivot_csv = TABLE_DIR / "Supplementary_Table_S8_Chapman_multiseed_macro_f1_mean_std_pivot.csv"
    pivot_tex = TABLE_DIR / "Supplementary_Table_S8_Chapman_multiseed_macro_f1_mean_std_pivot.tex"

    pivot_df.to_csv(pivot_csv, index=False)

    latex_df = pivot_df[
        [
            "lead_display_name",
            "known_unseen",
            "Standard_Macro_F1_mean_std",
            "Random_Macro_F1_mean_std",
            "Clinical_Macro_F1_mean_std",
            "Clinical_minus_Random_mean",
        ]
    ].copy()

    latex_df.columns = [
        "Lead Condition",
        "Group",
        "Standard",
        "Random",
        "Clinical",
        "Clinical--Random Mean",
    ]

    export_latex(
        latex_df,
        pivot_tex,
        caption=(
            "Chapman multi-seed Macro-F1 sensitivity analysis. Values are mean $\\pm$ standard deviation "
            "over three independent training seeds."
        ),
        label="tab:supp_chapman_multiseed_macro_f1",
        escape=False,
    )

    margin_rows = []

    for lead_key in KNOWN_REDUCED_KEYS:
        sub = pivot_df[pivot_df["lead_condition"] == lead_key]

        if len(sub) != 1:
            continue

        r = sub.iloc[0]

        margin_rows.append({
            "lead_condition": lead_key,
            "lead_display_name": LEAD_CONFIGS[lead_key]["display_name"],
            "Random_Macro_F1_mean": r["Random_Macro_F1_mean"],
            "Random_Macro_F1_std": r["Random_Macro_F1_std"],
            "Clinical_Macro_F1_mean": r["Clinical_Macro_F1_mean"],
            "Clinical_Macro_F1_std": r["Clinical_Macro_F1_std"],
            "Clinical_minus_Random_mean": r["Clinical_minus_Random_mean"],
        })

    margin_df = pd.DataFrame(margin_rows)

    margin_csv = TABLE_DIR / "Supplementary_Table_S9_Chapman_multiseed_known_reduced_Clinical_minus_Random.csv"
    margin_tex = TABLE_DIR / "Supplementary_Table_S9_Chapman_multiseed_known_reduced_Clinical_minus_Random.tex"

    margin_df.to_csv(margin_csv, index=False)

    latex_margin = margin_df.copy()
    latex_margin.columns = [
        "Lead Condition Key",
        "Lead Condition",
        "Random Mean",
        "Random Std",
        "Clinical Mean",
        "Clinical Std",
        "Clinical--Random Mean",
    ]

    export_latex(
        latex_margin,
        margin_tex,
        caption=(
            "Chapman multi-seed Clinical--Random comparison across known reduced-lead conditions."
        ),
        label="tab:supp_chapman_multiseed_clinical_random_known_reduced",
        escape=False,
    )

    training_meta = summary_df[
        [
            "dataset",
            "policy",
            "seed",
            "p_mask",
            "best_epoch",
            "best_val_macro_f1",
            "elapsed_minutes",
            "use_class_weights",
            "best_checkpoint",
        ]
    ].copy()

    training_meta_csv = TABLE_DIR / "Supplementary_Table_S10_Chapman_multiseed_training_metadata.csv"
    training_meta_tex = TABLE_DIR / "Supplementary_Table_S10_Chapman_multiseed_training_metadata.tex"

    training_meta.to_csv(training_meta_csv, index=False)

    export_latex(
        training_meta.drop(columns=["best_checkpoint"]),
        training_meta_tex,
        caption=(
            "Chapman multi-seed training metadata, including best-validation epoch and wall-clock time."
        ),
        label="tab:supp_chapman_multiseed_training_metadata",
        escape=False,
    )

    expected_rows = len(POLICIES) * len(SEEDS) * len(ALL_EVAL_KEYS)
    expected_agg_rows = len(POLICIES) * len(ALL_EVAL_KEYS)

    audit = {
        "expected_long_rows": expected_rows,
        "observed_long_rows": int(len(long_df)),
        "expected_aggregate_rows": expected_agg_rows,
        "observed_aggregate_rows": int(len(agg)),
        "policies": POLICIES,
        "seeds": SEEDS,
        "conditions": ALL_EVAL_KEYS,
        "all_long_rows_present": int(len(long_df)) == expected_rows,
        "all_aggregate_rows_present": int(len(agg)) == expected_agg_rows,
        "files": {
            "summary_csv": str(summary_csv),
            "long_csv": str(long_csv),
            "agg_csv": str(agg_csv),
            "pivot_csv": str(pivot_csv),
            "pivot_tex": str(pivot_tex),
            "margin_csv": str(margin_csv),
            "margin_tex": str(margin_tex),
            "training_meta_csv": str(training_meta_csv),
            "training_meta_tex": str(training_meta_tex),
        },
    }

    audit_json = REPORT_DIR / "step26_chapman_multiseed_audit.json"
    audit_txt = REPORT_DIR / "step26_chapman_multiseed_audit.txt"

    write_json(audit_json, audit)

    lines = []
    lines.append("=" * 140)
    lines.append("STEP 26 — CHAPMAN MULTI-SEED SENSITIVITY SUMMARY")
    lines.append("=" * 140)
    lines.append(f"Expected long rows      : {expected_rows}")
    lines.append(f"Observed long rows      : {len(long_df)}")
    lines.append(f"Expected aggregate rows : {expected_agg_rows}")
    lines.append(f"Observed aggregate rows : {len(agg)}")
    lines.append(f"All long rows present   : {audit['all_long_rows_present']}")
    lines.append(f"All aggregate rows      : {audit['all_aggregate_rows_present']}")
    lines.append("")
    lines.append("OUTPUT FILES")
    for _, fp in audit["files"].items():
        lines.append(f"  - {fp}")
    lines.append("")
    lines.append("RECOMMENDED PAPER USE")
    lines.append("  - Add pivot table as Supplementary Table S8.")
    lines.append("  - Add known reduced Clinical--Random table as Supplementary Table S9.")
    lines.append("  - Add training metadata as Supplementary Table S10.")
    lines.append("  - In main text, mention Chapman multi-seed sensitivity was performed as supplementary verification.")
    lines.append("=" * 140)

    audit_txt.write_text("\n".join(lines), encoding="utf-8")

    print("\n".join(lines))



def preflight() -> None:
    required = [
        CHAPMAN_DATA_DIR / "train_signals.npy",
        CHAPMAN_DATA_DIR / "val_signals.npy",
        CHAPMAN_DATA_DIR / "test_signals.npy",
        CHAPMAN_DATA_DIR / "train_labels.npy",
        CHAPMAN_DATA_DIR / "val_labels.npy",
        CHAPMAN_DATA_DIR / "test_labels.npy",
    ]

    for p in required:
        require_exists(p, p.name)

    train_labels = np.load(CHAPMAN_DATA_DIR / "train_labels.npy")
    val_labels = np.load(CHAPMAN_DATA_DIR / "val_labels.npy")
    test_labels = np.load(CHAPMAN_DATA_DIR / "test_labels.npy")

    print("Chapman split sizes:")
    print(f"  train: {len(train_labels)}")
    print(f"  val  : {len(val_labels)}")
    print(f"  test : {len(test_labels)}")

    print("Class counts:")
    for split_name, labels in [
        ("train", train_labels),
        ("val", val_labels),
        ("test", test_labels),
    ]:
        counts = {
            CLASS_NAMES[i]: int((labels == i).sum())
            for i in range(NUM_CLASSES)
        }
        print(f"  {split_name}: {counts}")

    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")



def main() -> None:
    print("=" * 140)
    print("STEP 26 — CHAPMAN MULTI-SEED SENSITIVITY ANALYSIS")
    print("=" * 140)
    print(f"Project root : {PROJECT_ROOT}")
    print(f"Chapman data : {CHAPMAN_DATA_DIR}")
    print(f"Output dir   : {OUT_DIR}")
    print(f"Policies     : {POLICIES}")
    print(f"Seeds        : {SEEDS}")
    print(f"p_mask       : {P_MASK}")
    print(f"Class weights: {USE_CLASS_WEIGHTS}")
    print("=" * 140)

    preflight()

    summaries = []

    for policy in POLICIES:
        for seed in SEEDS:
            summary = train_variant(policy=policy, seed=seed)
            summaries.append(summary)

    aggregate_results(summaries)

    print("\nDONE ✅ STEP 26 Chapman multi-seed sensitivity analysis completed.")


if __name__ == "__main__":
    main()